In [184]:

# 1. 학습 테이블 생성
#    1-1. 라이브러리·경로 설정
#    1-2. CSV 데이터 로드
#    1-3. 핵심 데이터 구조 점검
#    1-4. 영업시간 기준 시간 격자 생성
#    1-5. 달력·점포·상품 피처 결합
#    1-6. 방문객·시간대 피처 생성
#    1-7. 재고 로트 집계
#    1-8. 통합 학습 테이블 생성

# 2. 판매 데이터 연결 및 시계열 피처
#    2-1. 판매 데이터 시간별 집계
#    2-2. 영업시간 무판매 행 sales_qty=0 처리
#    2-3. 할인 거래 및 정가 기준 처리
#    2-4. 판매량 lag·rolling 피처 생성
#    2-5. 시간순 Train·Validation·Test 분할

# 3. 기준선 모델 만들기
#    3-1. Seasonal Naive
#    3-2. Moving Average
#    3-3. Croston 계열

# 4. LightGBM Quantile
#    4-1. 모델 입력 피처 확정
#    4-2. P10 모델
#    4-3. P50 모델
#    4-4. P90 모델
#    4-5. 정가 조건 기본수요 추론

# 5. 평가
#    5-1. MAE·RMSE·WAPE
#    5-2. Pinball Loss
#    5-3. 예측구간 커버리지·폭
#    5-4. 분위수 역전 확인
#    5-5. 점포·상품군·시간대별 평가

## 희소수요에 적합한 모델 구조로 개선

# 6. 고도화 모델 비교
#    6-1. DeepAR
#    6-2. TFT
#    6-3. PatchTST
#    6-4. Chronos-Bolt

# 7. 최종 모델 선정
#    7-1. 상품군별 모델 성능 비교
#    7-2. 최종 모델 결정


In [185]:
# ============================================================
#  라이브러리 import

!pip install -q lightgbm pyarrow

# 파일 및 경로 관리
import os
import json
import time
import random
import warnings

from pathlib import Path
from typing import Optional, List, Dict, Tuple

# 데이터 처리
import numpy as np
import pandas as pd

# 시각화
import matplotlib.pyplot as plt

# 머신러닝 및 평가
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# 모델 저장
import joblib

# 불필요한 경고 숨김
warnings.filterwarnings("ignore")

print("라이브러리 import 완료")

라이브러리 import 완료


In [186]:
# ============================================================
# 재현성 및 pandas 출력 옵션 설정

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# 표가 중간에 너무 많이 생략되지 않도록 설정
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print(f"Random seed: {SEED}")
print("pandas 출력 옵션 설정 완료")

Random seed: 42
pandas 출력 옵션 설정 완료


In [187]:
# ============================================================
#  설치된 라이브러리 버전 확인
import sklearn
import pyarrow

version_info = {
    "Python": os.sys.version.split()[0],
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "Scikit-learn": sklearn.__version__,
    "LightGBM": lgb.__version__,
    "PyArrow": pyarrow.__version__,
}

version_df = pd.DataFrame(
    version_info.items(),
    columns=["library", "version"]
)

display(version_df)

,library,version
0,Python,3.12.13
1,NumPy,2.0.2
2,Pandas,2.2.2
3,Scikit-learn,1.6.1
4,LightGBM,4.6.0
5,PyArrow,18.1.0


In [188]:
# ============================================================
#  현재 실행 환경 확인

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print("Google Colab 환경 여부:", IS_COLAB)
print("현재 작업 폴더:", os.getcwd())

Google Colab 환경 여부: True
현재 작업 폴더: /content


In [189]:
# ============================================================
#  데이터 및 결과 저장 경로 설정(데이터폴더랑 결과폴더 설정)

from pathlib import Path

# 현재 CSV가 업로드된 위치
DATA_DIR = Path("/content")

# 중간 산출물과 최종 결과를 저장할 위치
OUTPUT_DIR = Path("/content/a1_outputs")
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"

# 폴더가 없으면 생성
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("결과 폴더:", OUTPUT_DIR)
print("모델 저장 폴더:", MODEL_DIR)
print("평가 결과 폴더:", REPORT_DIR)

데이터 폴더: /content
결과 폴더: /content/a1_outputs
모델 저장 폴더: /content/a1_outputs/models
평가 결과 폴더: /content/a1_outputs/reports


In [190]:
# ============================================================
# 파일명 정의

FILE_PATHS = {
    "store": DATA_DIR / "store.csv",
    "product": DATA_DIR / "product.csv",
    "calendar": DATA_DIR / "calendar.csv",
    "store_calendar": DATA_DIR / "store_calendar.csv",
    "store_visitor_profile": DATA_DIR / "store_visitor_profile.csv",
    "inventory": DATA_DIR / "inventory.csv",
    "visitor": DATA_DIR / "visitor.csv",
    "customer": DATA_DIR / "customer.csv",
}

# A-1에서 직접 사용하는 파일
A1_REQUIRED_FILES = [
    "store",
    "product",
    "calendar",
    "store_calendar",
    "store_visitor_profile",
    "inventory",
    "visitor",
]

# customer.csv는 B 모델용이므로 A-1 필수 파일에서는 제외
print("A-1 필수 파일 수:", len(A1_REQUIRED_FILES))

A-1 필수 파일 수: 7


In [191]:
# ============================================================
#  A-1 사용 데이터 로드(# A-1 모델에 넣을 학습테이블만드는중..)
# ============================================================

from pathlib import Path
import pandas as pd

DATA_DIR = Path("/content")

# 점포·상품 정보
store_df = pd.read_csv(DATA_DIR / "store.csv")
product_df = pd.read_csv(DATA_DIR / "product.csv")

# 달력·점포별 영업일 정보
calendar_df = pd.read_csv(
    DATA_DIR / "calendar.csv",
    parse_dates=["date"]
)

store_calendar_df = pd.read_csv(
    DATA_DIR / "store_calendar.csv",
    parse_dates=["date"]
)

# 시간대별 방문 비율 프로필
visitor_profile_df = pd.read_csv(
    DATA_DIR / "store_visitor_profile.csv"
)

# 방문 이벤트
visitor_df = pd.read_csv(
    DATA_DIR / "visitor.csv",
    parse_dates=["visit_date"]
)

# 로트 단위 재고
inventory_df = pd.read_csv(
    DATA_DIR / "inventory.csv",
    parse_dates=[
        "manufacture_date",
        "expiry_date",
        "current_date"
    ]
)
transaction_df = pd.read_csv(f"{DATA_DIR}/transaction.csv")
receipt_df = pd.read_csv(f"{DATA_DIR}/receipt.csv")

print("store_df:", store_df.shape)
print("product_df:", product_df.shape)
print("calendar_df:", calendar_df.shape)
print("store_calendar_df:", store_calendar_df.shape)
print("visitor_profile_df:", visitor_profile_df.shape)
print("visitor_df:", visitor_df.shape)
print("inventory_df:", inventory_df.shape)
print("transaction_df:", transaction_df.shape)
print("receipt_df:", receipt_df.shape)

store_df: (3, 11)
product_df: (38, 17)
calendar_df: (365, 15)
store_calendar_df: (1095, 7)
visitor_profile_df: (45, 8)
visitor_df: (629170, 8)
inventory_df: (75537, 24)
transaction_df: (107604, 15)
receipt_df: (118791, 18)


In [192]:
# ============================================================
# 모델링용 핵심 컬럼 dtype 정리
# ============================================================

# 점포 ID를 문자열로 통일
store_df["store_id"] = store_df["store_id"].astype(str)
store_calendar_df["store_id"] = store_calendar_df["store_id"].astype(str)
visitor_df["store_id"] = visitor_df["store_id"].astype(str)
inventory_df["store_id"] = inventory_df["store_id"].astype(str)

# 상품 ID를 문자열로 통일
product_df["product_id"] = product_df["product_id"].astype(str)
inventory_df["product_id"] = inventory_df["product_id"].astype(str)

# 날짜에서 시간 정보 제거
calendar_df["date"] = pd.to_datetime(
    calendar_df["date"]
).dt.normalize()

store_calendar_df["date"] = pd.to_datetime(
    store_calendar_df["date"]
).dt.normalize()

visitor_df["visit_date"] = pd.to_datetime(
    visitor_df["visit_date"]
).dt.normalize()

inventory_df["current_date"] = pd.to_datetime(
    inventory_df["current_date"]
).dt.normalize()

inventory_df["manufacture_date"] = pd.to_datetime(
    inventory_df["manufacture_date"]
).dt.normalize()

inventory_df["expiry_date"] = pd.to_datetime(
    inventory_df["expiry_date"]
).dt.normalize()

# 시간 컬럼 숫자형 정리
store_df["open_hour"] = pd.to_numeric(
    store_df["open_hour"],
    errors="coerce"
)

store_df["close_hour"] = pd.to_numeric(
    store_df["close_hour"],
    errors="coerce"
)

store_calendar_df["open_hour"] = pd.to_numeric(
    store_calendar_df["open_hour"],
    errors="coerce"
)

store_calendar_df["close_hour"] = pd.to_numeric(
    store_calendar_df["close_hour"],
    errors="coerce"
)

visitor_profile_df["start_hour"] = pd.to_numeric(
    visitor_profile_df["start_hour"],
    errors="coerce"
)

visitor_profile_df["end_hour"] = pd.to_numeric(
    visitor_profile_df["end_hour"],
    errors="coerce"
)

visitor_profile_df["close_hour"] = pd.to_numeric(
    visitor_profile_df["close_hour"],
    errors="coerce"
)

# inventory 할인율
# 최종 데이터에서는 discount_rate가 이미 0~0.4 비율로 저장되어 있음
inventory_df["discount_rate"] = pd.to_numeric(
    inventory_df["discount_rate"],
    errors="coerce"
)

# 이후 기존 코드와 호환하기 위한 파생 컬럼
inventory_df["current_discount_rate_ratio"] = (
    inventory_df["discount_rate"]
)

# product 최대 할인율도 이미 0~0.4 비율
product_df["max_discount_rate"] = pd.to_numeric(
    product_df["max_discount_rate"],
    errors="coerce"
)

print("핵심 컬럼 dtype 정리 완료")

핵심 컬럼 dtype 정리 완료


In [193]:
# ============================================================
# 주요 키와 파일 간 참조 관계 점검
# A-1 사용 테이블 기준
# ============================================================

valid_store_ids = set(store_df["store_id"].dropna())
valid_product_ids = set(product_df["product_id"].dropna())
valid_dates = set(calendar_df["date"].dropna())

structure_check = {
    # --------------------------------------------------------
    # 1. 기준 테이블 기본키 중복
    # --------------------------------------------------------
    "store_id 중복":
        store_df.duplicated(["store_id"]).sum(),

    "product_id 중복":
        product_df.duplicated(["product_id"]).sum(),

    "calendar date 중복":
        calendar_df.duplicated(["date"]).sum(),

    # --------------------------------------------------------
    # 2. 테이블별 행 식별자 중복
    # --------------------------------------------------------
    "store_calendar 점포-날짜 중복":
        store_calendar_df.duplicated(
            ["store_id", "date"]
        ).sum(),

    "visitor_id 중복":
        visitor_df.duplicated(["visitor_id"]).sum(),

    "inventory_id 중복":
        inventory_df.duplicated(["inventory_id"]).sum(),

    # 동일 로트의 동일 날짜 재고 스냅샷 중복 확인
    "inventory 점포-상품-로트-기준일 중복":
        inventory_df.duplicated(
            ["store_id", "product_id", "lot_id", "current_date"]
        ).sum(),

    # --------------------------------------------------------
    # 3. store 참조 관계
    # --------------------------------------------------------
    "store_calendar 미등록 store_id":
        (~store_calendar_df["store_id"].isin(valid_store_ids)).sum(),

    "visitor 미등록 store_id":
        (~visitor_df["store_id"].isin(valid_store_ids)).sum(),

    "inventory 미등록 store_id":
        (~inventory_df["store_id"].isin(valid_store_ids)).sum(),

    # --------------------------------------------------------
    # 4. product 참조 관계
    # --------------------------------------------------------
    "inventory 미등록 product_id":
        (~inventory_df["product_id"].isin(valid_product_ids)).sum(),

    # --------------------------------------------------------
    # 5. calendar 참조 관계
    # --------------------------------------------------------
    "store_calendar 미등록 date":
        (~store_calendar_df["date"].isin(valid_dates)).sum(),

    "visitor 미등록 visit_date":
        (~visitor_df["visit_date"].isin(valid_dates)).sum(),

    "inventory 미등록 current_date":
        (~inventory_df["current_date"].isin(valid_dates)).sum(),
}

structure_check_df = pd.DataFrame(
    structure_check.items(),
    columns=["check_item", "issue_count"]
)

display(structure_check_df)

,check_item,issue_count
0,store_id 중복,0
1,product_id 중복,0
2,calendar date 중복,0
3,store_calendar 점포-날짜 중복,0
4,visitor_id 중복,0
5,inventory_id 중복,0
6,inventory 점포-상품-로트-기준일 중복,0
7,store_calendar 미등록 store_id,0
8,visitor 미등록 store_id,0
9,inventory 미등록 store_id,0


In [194]:
# ============================================================
# 파일별 날짜 범위 확인
# ============================================================

date_range_df = pd.DataFrame([
    {
        "data": "calendar",
        "start_date": calendar_df["date"].min(),
        "end_date": calendar_df["date"].max(),
        "unique_dates": calendar_df["date"].nunique()
    },
    {
        "data": "store_calendar",
        "start_date": store_calendar_df["date"].min(),
        "end_date": store_calendar_df["date"].max(),
        "unique_dates": store_calendar_df["date"].nunique()
    },
    {
        "data": "visitor",
        "start_date": visitor_df["visit_date"].min(),
        "end_date": visitor_df["visit_date"].max(),
        "unique_dates": visitor_df["visit_date"].nunique()
    },
    {
        "data": "inventory",
        "start_date": inventory_df["current_date"].min(),
        "end_date": inventory_df["current_date"].max(),
        "unique_dates": inventory_df["current_date"].nunique()
    }
])
print('4-3결과')
display(date_range_df)

4-3결과


,data,start_date,end_date,unique_dates
0,calendar,2025-01-01,2025-12-31,365
1,store_calendar,2025-01-01,2025-12-31,365
2,visitor,2025-01-01,2025-12-31,365
3,inventory,2025-01-01,2025-12-31,365


In [195]:
# ============================================================
# 4-4. 영업시간 및 할인율 범위 점검
# ============================================================

# ------------------------------------------------------------
# 1. 점검에 필요한 컬럼 숫자형 정리
# ------------------------------------------------------------

store_df["open_hour"] = pd.to_numeric(
    store_df["open_hour"],
    errors="coerce"
)

store_df["close_hour"] = pd.to_numeric(
    store_df["close_hour"],
    errors="coerce"
)

store_calendar_df["is_open"] = pd.to_numeric(
    store_calendar_df["is_open"],
    errors="coerce"
)

store_calendar_df["open_hour"] = pd.to_numeric(
    store_calendar_df["open_hour"],
    errors="coerce"
)

store_calendar_df["close_hour"] = pd.to_numeric(
    store_calendar_df["close_hour"],
    errors="coerce"
)

# inventory 원본 할인율
# 값의 의미: 11은 11%, 40은 40%
inventory_df["discount_rate"] = pd.to_numeric(
    inventory_df["discount_rate"],
    errors="coerce"
)

# 모델링에 사용할 할인율 비율
# 예: 11 → 0.11, 25 → 0.25, 40 → 0.40
inventory_df["current_discount_rate_ratio"] = (
    inventory_df["discount_rate"] / 100
)

# product 최대 할인율은 이미 0~0.4 비율
product_df["max_discount_rate"] = pd.to_numeric(
    product_df["max_discount_rate"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. 실제 영업일만 추출
# ------------------------------------------------------------

open_store_calendar = store_calendar_df[
    store_calendar_df["is_open"] == 1
].copy()

# ------------------------------------------------------------
# 3. 영업시간 및 할인율 규칙 점검
# ------------------------------------------------------------

business_rule_check = {
    # 영업일 영업시간 결측
    "영업일인데 open_hour 결측":
        open_store_calendar["open_hour"].isna().sum(),

    "영업일인데 close_hour 결측":
        open_store_calendar["close_hour"].isna().sum(),

    # 영업 시작 시간이 종료 시간보다 늦거나 같은 경우
    "영업일인데 open_hour >= close_hour":
        (
            open_store_calendar["open_hour"].notna()
            & open_store_calendar["close_hour"].notna()
            & (
                open_store_calendar["open_hour"]
                >= open_store_calendar["close_hour"]
            )
        ).sum(),

    # store 기본 영업시간 범위
    "store open_hour 범위 오류":
        (
            store_df["open_hour"].notna()
            & ~store_df["open_hour"].between(0, 23)
        ).sum(),

    "store close_hour 범위 오류":
        (
            store_df["close_hour"].notna()
            & ~store_df["close_hour"].between(1, 24)
        ).sum(),

    # store_calendar 영업시간 범위
    "store_calendar open_hour 범위 오류":
        (
            open_store_calendar["open_hour"].notna()
            & ~open_store_calendar["open_hour"].between(0, 23)
        ).sum(),

    "store_calendar close_hour 범위 오류":
        (
            open_store_calendar["close_hour"].notna()
            & ~open_store_calendar["close_hour"].between(1, 24)
        ).sum(),

    # inventory 원본 할인율
    "inventory 할인율 결측":
        inventory_df["discount_rate"].isna().sum(),

    "inventory 원본 할인율 0~40 범위 이탈":
        (
            inventory_df["discount_rate"].notna()
            & ~inventory_df["discount_rate"].between(0, 40)
        ).sum(),

    # 모델링용 변환 할인율
    "inventory 변환 할인율 0~0.4 범위 이탈":
        (
            inventory_df[
                "current_discount_rate_ratio"
            ].notna()
            & ~inventory_df[
                "current_discount_rate_ratio"
            ].between(0, 0.4)
        ).sum(),

    # product 최대 할인율
    "product 최대 할인율 결측":
        product_df["max_discount_rate"].isna().sum(),

    "product 최대 할인율 0~0.4 범위 이탈":
        (
            product_df["max_discount_rate"].notna()
            & ~product_df[
                "max_discount_rate"
            ].between(0, 0.4)
        ).sum(),
}

# ------------------------------------------------------------
# 4. 결과 출력
# ------------------------------------------------------------

business_rule_check_df = pd.DataFrame(
    business_rule_check.items(),
    columns=["check_item", "issue_count"]
)

print("4-4 결과")
display(business_rule_check_df)

4-4 결과


,check_item,issue_count
0,영업일인데 open_hour 결측,0
1,영업일인데 close_hour 결측,0
2,영업일인데 open_hour >= close_hour,0
3,store open_hour 범위 오류,0
4,store close_hour 범위 오류,0
5,store_calendar open_hour 범위 오류,0
6,store_calendar close_hour 범위 오류,0
7,inventory 할인율 결측,0
8,inventory 원본 할인율 0~40 범위 이탈,0
9,inventory 변환 할인율 0~0.4 범위 이탈,0


In [196]:
# ============================================================
# 4-5. 재고 스냅샷 구조 확인
# ============================================================

inventory_snapshot_base = inventory_df.copy()

# 점포-상품 조합 식별용 키
inventory_snapshot_base["store_product_key"] = (
    inventory_snapshot_base["store_id"].astype(str)
    + "_"
    + inventory_snapshot_base["product_id"].astype(str)
)

# 점포-상품-로트 조합 식별용 키
inventory_snapshot_base["store_product_lot_key"] = (
    inventory_snapshot_base["store_id"].astype(str)
    + "_"
    + inventory_snapshot_base["product_id"].astype(str)
    + "_"
    + inventory_snapshot_base["lot_id"].astype(str)
)

inventory_snapshot_df = (
    inventory_snapshot_base
    .groupby("current_date")
    .agg(
        # 해당 날짜 재고 데이터 행 수
        inventory_rows=("inventory_id", "size"),

        # 해당 날짜 재고가 존재하는 점포 수
        store_count=("store_id", "nunique"),

        # 해당 날짜 재고가 존재하는 상품 수
        product_count=("product_id", "nunique"),

        # 해당 날짜의 점포-상품 조합 수
        store_product_count=("store_product_key", "nunique"),

        # 해당 날짜의 점포-상품-로트 조합 수
        lot_count=("store_product_lot_key", "nunique")
    )
    .reset_index()
    .sort_values("current_date")
)

print("4-5 결과")
display(inventory_snapshot_df.head(10))
display(inventory_snapshot_df.tail(10))

4-5 결과


,current_date,inventory_rows,store_count,product_count,store_product_count,lot_count
0,2025-01-01,270,3,38,114,270
1,2025-01-02,274,3,38,114,274
2,2025-01-03,242,3,38,114,242
3,2025-01-04,222,3,38,114,222
4,2025-01-05,217,3,38,114,217
5,2025-01-06,220,3,38,114,220
6,2025-01-07,210,3,38,114,210
7,2025-01-08,194,3,38,113,194
8,2025-01-09,205,3,38,114,205
9,2025-01-10,196,3,38,114,196


,current_date,inventory_rows,store_count,product_count,store_product_count,lot_count
355,2025-12-22,208,3,38,114,208
356,2025-12-23,208,3,38,114,208
357,2025-12-24,189,3,38,114,189
358,2025-12-25,208,3,38,114,208
359,2025-12-26,202,3,38,114,202
360,2025-12-27,203,3,38,114,203
361,2025-12-28,177,3,38,114,177
362,2025-12-29,211,3,38,114,211
363,2025-12-30,193,3,38,114,193
364,2025-12-31,207,3,38,114,207


In [197]:
# ============================================================
#  재고 날짜별 연속성 확인
# ============================================================

# ------------------------------------------------------------
# 1. 전체 inventory 날짜 연속성
# ------------------------------------------------------------

inventory_dates = pd.Series(
    sorted(
        inventory_df["current_date"]
        .dropna()
        .unique()
    ),
    name="current_date"
)

inventory_date_gap = inventory_dates.diff().dt.days

inventory_date_summary = pd.DataFrame({
    "current_date": inventory_dates,
    "gap_from_previous_day": inventory_date_gap
})

print("전체 재고 날짜 연속성")
display(inventory_date_summary.head(10))

print(
    "재고 스냅샷 날짜 수:",
    inventory_df["current_date"].nunique()
)

print(
    "전체 기준 1일보다 큰 날짜 간격 수:",
    (inventory_date_gap > 1).sum()
)

# 날짜 간격이 1일보다 큰 구간만 출력
inventory_global_gap_df = inventory_date_summary[
    inventory_date_summary["gap_from_previous_day"] > 1
].copy()

print("전체 기준 날짜 누락 구간")
display(inventory_global_gap_df)


# ------------------------------------------------------------
# 2. 점포별 inventory 날짜 연속성
# ------------------------------------------------------------

inventory_store_date_df = (
    inventory_df[
        ["store_id", "current_date"]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values(
        ["store_id", "current_date"]
    )
)

inventory_store_date_df[
    "gap_from_previous_day"
] = (
    inventory_store_date_df
    .groupby("store_id")["current_date"]
    .diff()
    .dt.days
)

# 점포별 1일보다 큰 날짜 간격
inventory_store_gap_df = (
    inventory_store_date_df[
        inventory_store_date_df[
            "gap_from_previous_day"
        ] > 1
    ]
    .copy()
)

# 점포별 날짜 현황 요약
inventory_store_date_summary = (
    inventory_store_date_df
    .groupby("store_id")
    .agg(
        start_date=("current_date", "min"),
        end_date=("current_date", "max"),
        unique_dates=("current_date", "nunique"),
        max_date_gap=(
            "gap_from_previous_day",
            "max"
        ),
        gap_count=(
            "gap_from_previous_day",
            lambda x: (x > 1).sum()
        )
    )
    .reset_index()
)

print("점포별 재고 날짜 연속성 요약")
display(inventory_store_date_summary)

print(
    "날짜 누락이 있는 점포 수:",
    inventory_store_date_summary.loc[
        inventory_store_date_summary["gap_count"] > 0,
        "store_id"
    ].nunique()
)

print(
    "점포별 1일보다 큰 날짜 간격 수:",
    len(inventory_store_gap_df)
)

print("점포별 날짜 누락 구간")
display(inventory_store_gap_df.head(20))

전체 재고 날짜 연속성


,current_date,gap_from_previous_day
0,2025-01-01,NaN
1,2025-01-02,1.0000
2,2025-01-03,1.0000
3,2025-01-04,1.0000
4,2025-01-05,1.0000
5,2025-01-06,1.0000
6,2025-01-07,1.0000
7,2025-01-08,1.0000
8,2025-01-09,1.0000
9,2025-01-10,1.0000


재고 스냅샷 날짜 수: 365
전체 기준 1일보다 큰 날짜 간격 수: 0
전체 기준 날짜 누락 구간


,current_date,gap_from_previous_day


점포별 재고 날짜 연속성 요약


,store_id,start_date,end_date,unique_dates,max_date_gap,gap_count
0,S01,2025-01-01,2025-12-31,365,1.0000,0
1,S02,2025-01-01,2025-12-31,365,1.0000,0
2,S03,2025-01-01,2025-12-31,365,1.0000,0


날짜 누락이 있는 점포 수: 0
점포별 1일보다 큰 날짜 간격 수: 0
점포별 날짜 누락 구간


,store_id,current_date,gap_from_previous_day


In [198]:
# ============================================================
# 점포별 영업일 × 시간 격자 생성 및 결과 확인
# ============================================================

# 실제 영업일만 사용
open_calendar_df = (
    store_calendar_df.loc[
        store_calendar_df["is_open"] == 1,
        [
            "date",
            "store_id",
            "open_hour",
            "close_hour"
        ]
    ]
    .copy()
)

# 숫자형 변환
open_calendar_df["open_hour"] = pd.to_numeric(
    open_calendar_df["open_hour"],
    errors="coerce"
)

open_calendar_df["close_hour"] = pd.to_numeric(
    open_calendar_df["close_hour"],
    errors="coerce"
)

# 영업시간 결측 행 제외
open_calendar_df = open_calendar_df.dropna(
    subset=["open_hour", "close_hour"]
).copy()

# 정수형 변환
open_calendar_df["open_hour"] = (
    open_calendar_df["open_hour"].astype(int)
)

open_calendar_df["close_hour"] = (
    open_calendar_df["close_hour"].astype(int)
)

# 각 점포·날짜별 영업시간 리스트 생성
# 예: 10시 개점, 22시 폐점 → 10~21시
open_calendar_df["hour"] = open_calendar_df.apply(
    lambda row: list(
        range(
            row["open_hour"],
            row["close_hour"]
        )
    ),
    axis=1
)

# 한 시간당 한 행으로 펼치기
store_hour_grid_df = (
    open_calendar_df
    .explode("hour")
    .reset_index(drop=True)
)

store_hour_grid_df["hour"] = pd.to_numeric(
    store_hour_grid_df["hour"],
    errors="coerce"
).astype(int)

# 예측 기준 시각
store_hour_grid_df["forecast_datetime"] = (
    store_hour_grid_df["date"]
    + pd.to_timedelta(
        store_hour_grid_df["hour"],
        unit="h"
    )
)

# 폐점까지 남은 시간
store_hour_grid_df["hours_until_close"] = (
    store_hour_grid_df["close_hour"]
    - store_hour_grid_df["hour"]
)

# ============================================================
# 생성 결과 확인
# ============================================================

grid_check_df = (
    store_hour_grid_df
    .groupby(
        [
            "store_id",
            "date",
            "open_hour",
            "close_hour"
        ],
        as_index=False
    )
    .agg(
        generated_hour_count=("hour", "count"),
        first_hour=("hour", "min"),
        last_hour=("hour", "max"),
        min_hours_until_close=("hours_until_close", "min"),
        max_hours_until_close=("hours_until_close", "max")
    )
)

# 예상 시간대 개수
grid_check_df["expected_hour_count"] = (
    grid_check_df["close_hour"]
    - grid_check_df["open_hour"]
)

# 실제 생성 개수와 예상 개수가 다른 경우
grid_error_df = grid_check_df.loc[
    grid_check_df["generated_hour_count"]
    != grid_check_df["expected_hour_count"]
].copy()

print(
    "점포·날짜·시간 격자 크기:",
    store_hour_grid_df.shape
)

print(
    "점포·날짜·시간 중복 수:",
    store_hour_grid_df.duplicated(
        ["store_id", "date", "hour"]
    ).sum()
)

print(
    "시간 격자 생성 오류 수:",
    len(grid_error_df)
)

print("\n영업시간별 생성 결과")

display(
    grid_check_df[
        [
            "open_hour",
            "close_hour",
            "generated_hour_count",
            "first_hour",
            "last_hour",
            "min_hours_until_close",
            "max_hours_until_close"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["open_hour", "close_hour"]
    )
)

print("\n시간 격자 예시")
display(store_hour_grid_df.head(15))

print("\n오류 행")
display(grid_error_df.head(20))

점포·날짜·시간 격자 크기: (12958, 7)
점포·날짜·시간 중복 수: 0
시간 격자 생성 오류 수: 0

영업시간별 생성 결과


,open_hour,close_hour,generated_hour_count,first_hour,last_hour,min_hours_until_close,max_hours_until_close
341,10,22,12,10,21,1,12
0,10,23,13,10,22,1,13



시간 격자 예시


,date,store_id,open_hour,close_hour,hour,forecast_datetime,hours_until_close
0,2025-01-01,S01,10,23,10,2025-01-01 10:00:00,13
1,2025-01-01,S01,10,23,11,2025-01-01 11:00:00,12
2,2025-01-01,S01,10,23,12,2025-01-01 12:00:00,11
3,2025-01-01,S01,10,23,13,2025-01-01 13:00:00,10
4,2025-01-01,S01,10,23,14,2025-01-01 14:00:00,9
5,2025-01-01,S01,10,23,15,2025-01-01 15:00:00,8
6,2025-01-01,S01,10,23,16,2025-01-01 16:00:00,7
7,2025-01-01,S01,10,23,17,2025-01-01 17:00:00,6
8,2025-01-01,S01,10,23,18,2025-01-01 18:00:00,5
9,2025-01-01,S01,10,23,19,2025-01-01 19:00:00,4



오류 행


,store_id,date,open_hour,close_hour,generated_hour_count,first_hour,last_hour,min_hours_until_close,max_hours_until_close,expected_hour_count


In [199]:
# ============================================================
# 영업시간 격자에 해당 날짜의 점포별 재고 상품 결합
# ============================================================

# ------------------------------------------------------------
# 1. 날짜·점포·상품별 재고 존재 조합 생성
# 동일 상품에 여러 lot가 있어도 하나의 상품으로 중복 제거
# ------------------------------------------------------------

inventory_product_grid_df = (
    inventory_df[
        [
            "current_date",
            "store_id",
            "product_id"
        ]
    ]
    .dropna(
        subset=[
            "current_date",
            "store_id",
            "product_id"
        ]
    )
    .drop_duplicates()
    .rename(
        columns={
            "current_date": "date"
        }
    )
    .copy()
)

# ID 자료형 통일
inventory_product_grid_df["store_id"] = (
    inventory_product_grid_df["store_id"].astype(str)
)

inventory_product_grid_df["product_id"] = (
    inventory_product_grid_df["product_id"].astype(str)
)

# ------------------------------------------------------------
# 2. 점포·날짜·시간 격자에 해당 날짜의 재고 상품 결합
# ------------------------------------------------------------

time_grid_df = (
    store_hour_grid_df
    .merge(
        inventory_product_grid_df,
        on=[
            "date",
            "store_id"
        ],
        how="inner",
        validate="many_to_many"
    )
)

# ------------------------------------------------------------
# 3. A-1 최종 예측 단위에 맞게 정렬
# ------------------------------------------------------------

time_grid_df = (
    time_grid_df
    .sort_values(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. 결과 점검
# ------------------------------------------------------------

time_grid_duplicate_count = (
    time_grid_df
    .duplicated(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    )
    .sum()
)

print(
    "날짜·점포·상품 조합 수:",
    len(inventory_product_grid_df)
)

print(
    "A-1 시간 격자 크기:",
    time_grid_df.shape
)

print(
    "날짜·점포·상품·시간 중복 수:",
    time_grid_duplicate_count
)

print(
    "포함 점포 수:",
    time_grid_df["store_id"].nunique()
)

print(
    "포함 상품 수:",
    time_grid_df["product_id"].nunique()
)

print(
    "시작일:",
    time_grid_df["date"].min()
)

print(
    "종료일:",
    time_grid_df["date"].max()
)

display(time_grid_df.head(20))

날짜·점포·상품 조합 수: 41593
A-1 시간 격자 크기: (492404, 8)
날짜·점포·상품·시간 중복 수: 0
포함 점포 수: 3
포함 상품 수: 38
시작일: 2025-01-01 00:00:00
종료일: 2025-12-31 00:00:00


,date,store_id,open_hour,close_hour,hour,forecast_datetime,hours_until_close,product_id
0,2025-01-01,S01,10,23,10,2025-01-01 10:00:00,13,P001
1,2025-01-01,S01,10,23,11,2025-01-01 11:00:00,12,P001
2,2025-01-01,S01,10,23,12,2025-01-01 12:00:00,11,P001
3,2025-01-01,S01,10,23,13,2025-01-01 13:00:00,10,P001
4,2025-01-01,S01,10,23,14,2025-01-01 14:00:00,9,P001
5,2025-01-01,S01,10,23,15,2025-01-01 15:00:00,8,P001
6,2025-01-01,S01,10,23,16,2025-01-01 16:00:00,7,P001
7,2025-01-01,S01,10,23,17,2025-01-01 17:00:00,6,P001
8,2025-01-01,S01,10,23,18,2025-01-01 18:00:00,5,P001
9,2025-01-01,S01,10,23,19,2025-01-01 19:00:00,4,P001


In [200]:
# ============================================================
#  생성된 예측 단위 요약
# ============================================================

print(
    "날짜 수:",
    time_grid_df["date"].nunique()
)

print(
    "점포 수:",
    time_grid_df["store_id"].nunique()
)

print(
    "상품 수:",
    time_grid_df["product_id"].nunique()
)

print(
    "시간 범위:",
    time_grid_df["hour"].min(),
    "~",
    time_grid_df["hour"].max()
)

print(
    "전체 예측 행 수:",
    len(time_grid_df)
)

날짜 수: 365
점포 수: 3
상품 수: 38
시간 범위: 10 ~ 22
전체 예측 행 수: 492404


In [201]:
# ============================================================
# 1-5. 달력·점포·상품 기본 피처 결합
# ============================================================

# ------------------------------------------------------------
# 1. 달력 피처
# ------------------------------------------------------------

calendar_features = [
    "date",
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_month",
    "day_type",
    "is_weekend",
    "is_holiday",
    "holiday_name",
    "season",
    "season_index",
    "is_event_day",
    "event_type",
    "event_index"
]

# ------------------------------------------------------------
# 2. 점포 피처
# open_hour, close_hour는 time_grid_df에 이미 있으므로 제외
# ------------------------------------------------------------

store_features = [
    "store_id",
    "store_name",
    "area_type",
    "resident_pop",
    "floating_idx",
    "order_error_sigma"
]

# ------------------------------------------------------------
# 3. 상품 피처
# ------------------------------------------------------------

product_features = [
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "sales_type",
    "unit",
    "standard_weight_kg",
    "shelf_life_days",
    "base_cost",
    "base_price",
    "margin_rate",
    "freshness_decay_type",
    "baseline_waste_rate"
]

# ------------------------------------------------------------
# 4. 시간 격자에 달력 정보 결합
# ------------------------------------------------------------

a1_base_df = time_grid_df.merge(
    calendar_df[calendar_features],
    on="date",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 5. 점포 정보 결합
# ------------------------------------------------------------

a1_base_df = a1_base_df.merge(
    store_df[store_features],
    on="store_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 6. 상품 정보 결합
# ------------------------------------------------------------

a1_base_df = a1_base_df.merge(
    product_df[product_features],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 7. 시계열 단위에 맞게 정렬
# 동일 점포·상품의 시간이 순서대로 이어지도록 정렬
# ------------------------------------------------------------

a1_base_df = (
    a1_base_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 8. 결합 결과 점검
# ------------------------------------------------------------

print(
    "기본 피처 결합 후 크기:",
    a1_base_df.shape
)

print(
    "날짜·점포·상품·시간 중복 수:",
    a1_base_df.duplicated(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    ).sum()
)

print(
    "달력 정보 미결합 행 수:",
    a1_base_df["year"].isna().sum()
)

print(
    "점포 정보 미결합 행 수:",
    a1_base_df["store_name"].isna().sum()
)

print(
    "상품 정보 미결합 행 수:",
    a1_base_df["product_name"].isna().sum()
)

display(a1_base_df.head())

기본 피처 결합 후 크기: (492404, 39)
날짜·점포·상품·시간 중복 수: 0
달력 정보 미결합 행 수: 0
점포 정보 미결합 행 수: 0
상품 정보 미결합 행 수: 0


,date,store_id,open_hour,close_hour,hour,forecast_datetime,hours_until_close,product_id,year,month,day,day_of_week,week_of_month,day_type,is_weekend,is_holiday,holiday_name,season,season_index,is_event_day,event_type,event_index,store_name,area_type,resident_pop,floating_idx,order_error_sigma,product_name,category,subcategory,sales_type,unit,standard_weight_kg,shelf_life_days,base_cost,base_price,margin_rate,freshness_decay_type,baseline_waste_rate
0,2025-01-01,S01,10,23,10,2025-01-01 10:00:00,13,P001,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560
1,2025-01-01,S01,10,23,11,2025-01-01 11:00:00,12,P001,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560
2,2025-01-01,S01,10,23,12,2025-01-01 12:00:00,11,P001,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560
3,2025-01-01,S01,10,23,13,2025-01-01 13:00:00,10,P001,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560
4,2025-01-01,S01,10,23,14,2025-01-01 14:00:00,9,P001,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560


In [202]:
# ============================================================
# 1-5-1. 날짜·시간 주기 피처 생성 - 수정본
# ============================================================

# 계산에 사용할 숫자형 컬럼 생성
a1_base_df["hour_num"] = pd.to_numeric(
    a1_base_df["hour"],
    errors="coerce"
)

a1_base_df["month_num"] = pd.to_numeric(
    a1_base_df["month"],
    errors="coerce"
)

# 기존 day_of_week 형식과 관계없이 date에서 직접 생성
# 월요일=0, 화요일=1, ..., 일요일=6
a1_base_df["day_of_week_num"] = (
    pd.to_datetime(a1_base_df["date"]).dt.dayofweek
)

# 시간 주기 피처
a1_base_df["hour_sin"] = np.sin(
    2 * np.pi * a1_base_df["hour_num"] / 24
)

a1_base_df["hour_cos"] = np.cos(
    2 * np.pi * a1_base_df["hour_num"] / 24
)

# 요일 주기 피처
a1_base_df["day_of_week_sin"] = np.sin(
    2 * np.pi * a1_base_df["day_of_week_num"] / 7
)

a1_base_df["day_of_week_cos"] = np.cos(
    2 * np.pi * a1_base_df["day_of_week_num"] / 7
)

# 월 주기 피처
a1_base_df["month_sin"] = np.sin(
    2 * np.pi * (a1_base_df["month_num"] - 1) / 12
)

a1_base_df["month_cos"] = np.cos(
    2 * np.pi * (a1_base_df["month_num"] - 1) / 12
)

display(
    a1_base_df[
        [
            "forecast_datetime",
            "hour",
            "hour_sin",
            "hour_cos",
            "day_of_week",
            "day_of_week_num",
            "day_of_week_sin",
            "day_of_week_cos",
            "month",
            "month_sin",
            "month_cos"
        ]
    ].head(10)
)

,forecast_datetime,hour,hour_sin,hour_cos,day_of_week,day_of_week_num,day_of_week_sin,day_of_week_cos,month,month_sin,month_cos
0,2025-01-01 10:00:00,10,0.5000,-0.8660,WED,2,0.9749,-0.2225,1,0.0000,1.0000
1,2025-01-01 11:00:00,11,0.2588,-0.9659,WED,2,0.9749,-0.2225,1,0.0000,1.0000
2,2025-01-01 12:00:00,12,0.0000,-1.0000,WED,2,0.9749,-0.2225,1,0.0000,1.0000
3,2025-01-01 13:00:00,13,-0.2588,-0.9659,WED,2,0.9749,-0.2225,1,0.0000,1.0000
4,2025-01-01 14:00:00,14,-0.5000,-0.8660,WED,2,0.9749,-0.2225,1,0.0000,1.0000
5,2025-01-01 15:00:00,15,-0.7071,-0.7071,WED,2,0.9749,-0.2225,1,0.0000,1.0000
6,2025-01-01 16:00:00,16,-0.8660,-0.5000,WED,2,0.9749,-0.2225,1,0.0000,1.0000
7,2025-01-01 17:00:00,17,-0.9659,-0.2588,WED,2,0.9749,-0.2225,1,0.0000,1.0000
8,2025-01-01 18:00:00,18,-1.0000,-0.0000,WED,2,0.9749,-0.2225,1,0.0000,1.0000
9,2025-01-01 19:00:00,19,-0.9659,0.2588,WED,2,0.9749,-0.2225,1,0.0000,1.0000


In [203]:
# ============================================================
# 1-6-1. store_visitor_profile 시간 단위 변환
# ============================================================

visitor_profile_hour_df = visitor_profile_df.copy()

# ------------------------------------------------------------
# 1. 시간 컬럼 숫자형 변환
# ------------------------------------------------------------

time_columns = [
    "start_hour",
    "end_hour",
    "close_hour"
]

for col in time_columns:
    visitor_profile_hour_df[col] = pd.to_numeric(
        visitor_profile_hour_df[col],
        errors="coerce"
    )

# 방문객 비율 숫자형 변환
visitor_profile_hour_df["visitor_ratio"] = pd.to_numeric(
    visitor_profile_hour_df["visitor_ratio"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. 필수값 결측 행 확인 및 제거
# ------------------------------------------------------------

required_columns = [
    "area_type",
    "day_type",
    "start_hour",
    "end_hour",
    "close_hour",
    "time_slot",
    "visitor_ratio"
]

visitor_profile_missing_df = visitor_profile_hour_df[
    visitor_profile_hour_df[required_columns]
    .isna()
    .any(axis=1)
].copy()

print(
    "필수값 결측 행 수:",
    len(visitor_profile_missing_df)
)

visitor_profile_hour_df = (
    visitor_profile_hour_df
    .dropna(subset=required_columns)
    .copy()
)

# 시간 컬럼 정수형 변환
for col in time_columns:
    visitor_profile_hour_df[col] = (
        visitor_profile_hour_df[col].astype(int)
    )

# ------------------------------------------------------------
# 3. 시간 범위가 정상인 행만 사용
# ------------------------------------------------------------

invalid_time_rule = (
    ~visitor_profile_hour_df["start_hour"].between(0, 23)
    | ~visitor_profile_hour_df["end_hour"].between(1, 24)
    | ~visitor_profile_hour_df["close_hour"].between(1, 24)
    | (
        visitor_profile_hour_df["start_hour"]
        >= visitor_profile_hour_df["end_hour"]
    )
    | (
        visitor_profile_hour_df["end_hour"]
        > visitor_profile_hour_df["close_hour"]
    )
)

print(
    "시간 범위 오류 행 수:",
    invalid_time_rule.sum()
)

visitor_profile_hour_df = visitor_profile_hour_df[
    ~invalid_time_rule
].copy()

# ------------------------------------------------------------
# 4. 시간 구간을 1시간 단위로 확장
# start_hour 포함, end_hour 제외
# 예: 10~13 → 10, 11, 12
# ------------------------------------------------------------

visitor_profile_hour_df["hour"] = (
    visitor_profile_hour_df.apply(
        lambda row: list(
            range(
                row["start_hour"],
                row["end_hour"]
            )
        ),
        axis=1
    )
)

visitor_profile_hour_df = (
    visitor_profile_hour_df
    .explode("hour")
    .reset_index(drop=True)
)

visitor_profile_hour_df["hour"] = (
    visitor_profile_hour_df["hour"].astype(int)
)

# ------------------------------------------------------------
# 5. 결합에 필요한 컬럼만 선택
# ------------------------------------------------------------

visitor_profile_hour_df = visitor_profile_hour_df[
    [
        "area_type",
        "day_type",
        "close_hour",
        "hour",
        "time_slot",
        "visitor_ratio"
    ]
].copy()

# ------------------------------------------------------------
# 6. 결과 점검
# ------------------------------------------------------------

profile_duplicate_count = (
    visitor_profile_hour_df
    .duplicated(
        [
            "area_type",
            "day_type",
            "close_hour",
            "hour"
        ]
    )
    .sum()
)

print(
    "변환 후 행 수:",
    len(visitor_profile_hour_df)
)

print(
    "프로필 조건별 시간 중복 수:",
    profile_duplicate_count
)

print(
    "시간 범위:",
    visitor_profile_hour_df["hour"].min(),
    "~",
    visitor_profile_hour_df["hour"].max()
)

print(
    "visitor_ratio 범위:",
    visitor_profile_hour_df["visitor_ratio"].min(),
    "~",
    visitor_profile_hour_df["visitor_ratio"].max()
)

display(visitor_profile_hour_df.head(15))

필수값 결측 행 수: 0
시간 범위 오류 행 수: 0
변환 후 행 수: 114
프로필 조건별 시간 중복 수: 0
시간 범위: 10 ~ 22
visitor_ratio 범위: 0.1 ~ 0.35


,area_type,day_type,close_hour,hour,time_slot,visitor_ratio
0,residential,weekday,23,10,morning,0.1500
1,residential,weekday,23,11,morning,0.1500
2,residential,weekday,23,12,lunch,0.1500
3,residential,weekday,23,13,lunch,0.1500
4,residential,weekday,23,14,lunch,0.1500
5,residential,weekday,23,15,afternoon,0.2000
6,residential,weekday,23,16,afternoon,0.2000
7,residential,weekday,23,17,afternoon,0.2000
8,residential,weekday,23,18,evening,0.3500
9,residential,weekday,23,19,evening,0.3500


In [204]:
# ============================================================
# 1-6-2. visitor.csv 시간별 방문객 수 집계
# ============================================================

visitor_hourly_base_df = visitor_df.copy()

# ------------------------------------------------------------
# 1. ID 및 날짜 형식 정리
# ------------------------------------------------------------

visitor_hourly_base_df["store_id"] = (
    visitor_hourly_base_df["store_id"].astype(str)
)

visitor_hourly_base_df["visit_date"] = pd.to_datetime(
    visitor_hourly_base_df["visit_date"],
    errors="coerce"
).dt.normalize()

# ------------------------------------------------------------
# 2. visit_time에서 시간 추출
# ------------------------------------------------------------

parsed_visit_time = pd.to_datetime(
    visitor_hourly_base_df["visit_time"].astype(str),
    format="mixed",
    errors="coerce"
)

visitor_hourly_base_df["visit_hour"] = (
    parsed_visit_time.dt.hour
)

# ------------------------------------------------------------
# 3. 파싱 결과 점검
# ------------------------------------------------------------

print(
    "visit_date 변환 실패 행 수:",
    visitor_hourly_base_df["visit_date"].isna().sum()
)

print(
    "visit_time 변환 실패 행 수:",
    visitor_hourly_base_df["visit_hour"].isna().sum()
)

print(
    "visitor_id 결측 행 수:",
    visitor_hourly_base_df["visitor_id"].isna().sum()
)

# 필수값이 정상인 행만 집계에 사용
visitor_hourly_valid_df = (
    visitor_hourly_base_df
    .dropna(
        subset=[
            "store_id",
            "visit_date",
            "visit_hour",
            "visitor_id"
        ]
    )
    .copy()
)

visitor_hourly_valid_df["visit_hour"] = (
    visitor_hourly_valid_df["visit_hour"].astype(int)
)

# ------------------------------------------------------------
# 4. 점포·날짜·시간별 순방문객 수 집계
# 같은 visitor_id가 같은 시간에 여러 번 있어도 1명으로 계산
# ------------------------------------------------------------

visitor_hourly_count_df = (
    visitor_hourly_valid_df
    .groupby(
        [
            "store_id",
            "visit_date",
            "visit_hour"
        ],
        as_index=False
    )
    .agg(
        visitor_count=("visitor_id", "nunique")
    )
    .rename(
        columns={
            "visit_date": "date",
            "visit_hour": "hour"
        }
    )
    .sort_values(
        [
            "date",
            "store_id",
            "hour"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. 결과 점검
# ------------------------------------------------------------

print(
    "시간별 방문객 집계 행 수:",
    len(visitor_hourly_count_df)
)

print(
    "점포·날짜·시간 중복 수:",
    visitor_hourly_count_df.duplicated(
        ["store_id", "date", "hour"]
    ).sum()
)

print(
    "방문 시간 범위:",
    visitor_hourly_count_df["hour"].min(),
    "~",
    visitor_hourly_count_df["hour"].max()
)

print(
    "시간별 방문객 수 범위:",
    visitor_hourly_count_df["visitor_count"].min(),
    "~",
    visitor_hourly_count_df["visitor_count"].max()
)

display(visitor_hourly_count_df.head(10))

visit_date 변환 실패 행 수: 0
visit_time 변환 실패 행 수: 0
visitor_id 결측 행 수: 0
시간별 방문객 집계 행 수: 12958
점포·날짜·시간 중복 수: 0
방문 시간 범위: 10 ~ 22
시간별 방문객 수 범위: 9 ~ 99


,store_id,date,hour,visitor_count
0,S01,2025-01-01,10,51
1,S01,2025-01-01,11,63
2,S01,2025-01-01,12,55
3,S01,2025-01-01,13,50
4,S01,2025-01-01,14,58
5,S01,2025-01-01,15,46
6,S01,2025-01-01,16,53
7,S01,2025-01-01,17,42
8,S01,2025-01-01,18,46
9,S01,2025-01-01,19,58


In [205]:
# ============================================================
# 1-6-3. 점포·날짜·시간 단위 방문객 피처 생성
# ============================================================

# ------------------------------------------------------------
# 1. 상품 중복을 제거한 점포·날짜·시간 단위 기본 테이블
# ------------------------------------------------------------

visitor_hour_feature_df = (
    a1_base_df[
        [
            "date",
            "store_id",
            "area_type",
            "day_type",
            "open_hour",
            "close_hour",
            "hour",
            "forecast_datetime"
        ]
    ]
    .drop_duplicates(
        subset=[
            "date",
            "store_id",
            "hour"
        ]
    )
    .copy()
)

# 결합 키 자료형 정리
visitor_hour_feature_df["store_id"] = (
    visitor_hour_feature_df["store_id"].astype(str)
)

visitor_hour_feature_df["date"] = pd.to_datetime(
    visitor_hour_feature_df["date"],
    errors="coerce"
).dt.normalize()

visitor_hour_feature_df["hour"] = pd.to_numeric(
    visitor_hour_feature_df["hour"],
    errors="coerce"
).astype(int)

visitor_hour_feature_df["close_hour"] = pd.to_numeric(
    visitor_hour_feature_df["close_hour"],
    errors="coerce"
).astype(int)


# ------------------------------------------------------------
# 2. 프로필 기반 time_slot, visitor_ratio 연결
# ------------------------------------------------------------

visitor_hour_feature_df = visitor_hour_feature_df.merge(
    visitor_profile_hour_df,
    on=[
        "area_type",
        "day_type",
        "close_hour",
        "hour"
    ],
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 3. 실제 시간별 방문객 수 연결
# ------------------------------------------------------------

visitor_hourly_count_df["store_id"] = (
    visitor_hourly_count_df["store_id"].astype(str)
)

visitor_hourly_count_df["date"] = pd.to_datetime(
    visitor_hourly_count_df["date"],
    errors="coerce"
).dt.normalize()

visitor_hourly_count_df["hour"] = pd.to_numeric(
    visitor_hourly_count_df["hour"],
    errors="coerce"
).astype(int)

visitor_hour_feature_df = visitor_hour_feature_df.merge(
    visitor_hourly_count_df,
    on=[
        "store_id",
        "date",
        "hour"
    ],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 4. 방문 데이터 보유 여부 생성
# 날짜 전체가 아니라 점포·날짜 단위로 판단
# ------------------------------------------------------------

visitor_available_store_date_df = (
    visitor_hourly_valid_df[
        [
            "store_id",
            "visit_date"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "visit_date": "date"
        }
    )
    .copy()
)

visitor_available_store_date_df["store_id"] = (
    visitor_available_store_date_df["store_id"].astype(str)
)

visitor_available_store_date_df["date"] = pd.to_datetime(
    visitor_available_store_date_df["date"],
    errors="coerce"
).dt.normalize()

visitor_available_store_date_df[
    "visitor_data_available"
] = 1

visitor_hour_feature_df = visitor_hour_feature_df.merge(
    visitor_available_store_date_df,
    on=[
        "store_id",
        "date"
    ],
    how="left",
    validate="many_to_one"
)

visitor_hour_feature_df["visitor_data_available"] = (
    visitor_hour_feature_df["visitor_data_available"]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 5. 데이터가 있는 점포·날짜에서 방문 기록이 없으면 0명
# ------------------------------------------------------------

known_store_date_mask = (
    visitor_hour_feature_df["visitor_data_available"] == 1
)

visitor_hour_feature_df.loc[
    known_store_date_mask,
    "visitor_count"
] = (
    visitor_hour_feature_df.loc[
        known_store_date_mask,
        "visitor_count"
    ]
    .fillna(0)
)


# ------------------------------------------------------------
# 6. 시간순 정렬
# ------------------------------------------------------------

visitor_hour_feature_df = (
    visitor_hour_feature_df
    .sort_values(
        [
            "store_id",
            "date",
            "hour"
        ]
    )
    .reset_index(drop=True)
)

visitor_group = visitor_hour_feature_df.groupby(
    [
        "store_id",
        "date"
    ],
    sort=False
)


# ------------------------------------------------------------
# 7. 직전 영업시간 방문객 수
# 예: 11시 행에는 10시 방문객 수
# ------------------------------------------------------------

visitor_hour_feature_df["lag_1h_visitor_count"] = (
    visitor_group["visitor_count"].shift(1)
)


# ------------------------------------------------------------
# 8. 현재 시간 제외, 직전 최대 3시간 평균
# ------------------------------------------------------------

visitor_hour_feature_df["rolling_3h_visitor_mean"] = (
    visitor_group["visitor_count"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )
)


# ------------------------------------------------------------
# 9. 현재 시간 직전까지의 당일 누적 방문객 수
# 첫 영업시간에는 이전 방문객이 없으므로 0
# ------------------------------------------------------------

visitor_hour_feature_df["cumulative_daily_visitors"] = (
    visitor_group["visitor_count"]
    .transform(
        lambda x: (
            x.shift(1)
            .fillna(0)
            .cumsum()
        )
    )
)


# ------------------------------------------------------------
# 10. 방문 데이터가 없는 점포·날짜는 파생값도 결측 유지
# ------------------------------------------------------------

unavailable_mask = (
    visitor_hour_feature_df["visitor_data_available"] == 0
)

visitor_hour_feature_df.loc[
    unavailable_mask,
    [
        "visitor_count",
        "lag_1h_visitor_count",
        "rolling_3h_visitor_mean",
        "cumulative_daily_visitors"
    ]
] = np.nan


# ------------------------------------------------------------
# 11. 결과 점검
# ------------------------------------------------------------

print(
    "점포·날짜·시간 중복 수:",
    visitor_hour_feature_df.duplicated(
        [
            "store_id",
            "date",
            "hour"
        ]
    ).sum()
)

print(
    "프로필 미결합 행 수:",
    visitor_hour_feature_df["visitor_ratio"].isna().sum()
)

print(
    "방문 데이터 보유 점포·날짜 수:",
    visitor_hour_feature_df.loc[
        visitor_hour_feature_df["visitor_data_available"] == 1,
        ["store_id", "date"]
    ].drop_duplicates().shape[0]
)

print(
    "방문 데이터 미보유 점포·날짜 수:",
    visitor_hour_feature_df.loc[
        visitor_hour_feature_df["visitor_data_available"] == 0,
        ["store_id", "date"]
    ].drop_duplicates().shape[0]
)

print(
    "visitor_count 결측 행 수:",
    visitor_hour_feature_df["visitor_count"].isna().sum()
)

display(
    visitor_hour_feature_df[
        [
            "date",
            "store_id",
            "hour",
            "time_slot",
            "visitor_ratio",
            "visitor_count",
            "lag_1h_visitor_count",
            "rolling_3h_visitor_mean",
            "cumulative_daily_visitors",
            "visitor_data_available"
        ]
    ].tail(20)
)

점포·날짜·시간 중복 수: 0
프로필 미결합 행 수: 0
방문 데이터 보유 점포·날짜 수: 1023
방문 데이터 미보유 점포·날짜 수: 0
visitor_count 결측 행 수: 0


,date,store_id,hour,time_slot,visitor_ratio,visitor_count,lag_1h_visitor_count,rolling_3h_visitor_mean,cumulative_daily_visitors,visitor_data_available
12938,2025-12-30,S03,16,afternoon,0.2300,45.0000,52.0000,47.0000,279.0000,1
12939,2025-12-30,S03,17,afternoon,0.2300,60.0000,45.0000,48.0000,324.0000,1
12940,2025-12-30,S03,18,evening,0.3000,62.0000,60.0000,52.3333,384.0000,1
12941,2025-12-30,S03,19,evening,0.3000,57.0000,62.0000,55.6667,446.0000,1
12942,2025-12-30,S03,20,evening,0.3000,75.0000,57.0000,59.6667,503.0000,1
12943,2025-12-30,S03,21,closing,0.1500,39.0000,75.0000,64.6667,578.0000,1
12944,2025-12-30,S03,22,closing,0.1500,55.0000,39.0000,57.0000,617.0000,1
12945,2025-12-31,S03,10,morning,0.1200,51.0000,NaN,NaN,0.0000,1
12946,2025-12-31,S03,11,morning,0.1200,38.0000,51.0000,51.0000,51.0000,1
12947,2025-12-31,S03,12,lunch,0.2000,41.0000,38.0000,44.5000,89.0000,1


In [206]:
print(
    "전체 점포·날짜 조합 수:",
    store_calendar_df[
        ["store_id", "date"]
    ].drop_duplicates().shape[0]
)

print(
    "영업 점포·날짜 조합 수:",
    store_calendar_df.loc[
        store_calendar_df["is_open"] == 1,
        ["store_id", "date"]
    ].drop_duplicates().shape[0]
)

print(
    "휴점 점포·날짜 조합 수:",
    store_calendar_df.loc[
        store_calendar_df["is_open"] == 0,
        ["store_id", "date"]
    ].drop_duplicates().shape[0]
)

전체 점포·날짜 조합 수: 1095
영업 점포·날짜 조합 수: 1023
휴점 점포·날짜 조합 수: 72


In [207]:
# ============================================================
# 1-6-4. 방문객 피처를 상품별 A-1 테이블에 결합
# ============================================================

visitor_feature_columns = [
    "date",
    "store_id",
    "hour",
    "time_slot",
    "visitor_ratio",
    "visitor_count",
    "lag_1h_visitor_count",
    "rolling_3h_visitor_mean",
    "cumulative_daily_visitors",
    "visitor_data_available"
]

# ------------------------------------------------------------
# 1. 재실행 시 기존 방문객 피처 중복 방지
# ------------------------------------------------------------

visitor_value_columns = [
    col for col in visitor_feature_columns
    if col not in ["date", "store_id", "hour"]
]

existing_visitor_columns = [
    col for col in visitor_value_columns
    if col in a1_base_df.columns
]

if existing_visitor_columns:
    a1_base_df = a1_base_df.drop(
        columns=existing_visitor_columns
    )

# ------------------------------------------------------------
# 2. 결합 전 행 수 저장
# ------------------------------------------------------------

rows_before_merge = len(a1_base_df)

# ------------------------------------------------------------
# 3. 방문객 피처 결합
# 상품별 여러 행 : 점포·날짜·시간별 방문객 피처 한 행
# ------------------------------------------------------------

a1_base_df = a1_base_df.merge(
    visitor_hour_feature_df[
        visitor_feature_columns
    ],
    on=[
        "date",
        "store_id",
        "hour"
    ],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 4. 시계열 순서로 재정렬
# ------------------------------------------------------------

a1_base_df = (
    a1_base_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. 결합 결과 점검
# ------------------------------------------------------------

rows_after_merge = len(a1_base_df)

print(
    "방문객 피처 결합 전 행 수:",
    rows_before_merge
)

print(
    "방문객 피처 결합 후 행 수:",
    rows_after_merge
)

print(
    "결합으로 증가·감소한 행 수:",
    rows_after_merge - rows_before_merge
)

print(
    "날짜·점포·상품·시간 중복 수:",
    a1_base_df.duplicated(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    ).sum()
)

print(
    "visitor_ratio 미결합 행 수:",
    a1_base_df["visitor_ratio"].isna().sum()
)

print(
    "visitor_data_available 미결합 행 수:",
    a1_base_df["visitor_data_available"].isna().sum()
)

print(
    "visitor_count 결측 행 수:",
    a1_base_df["visitor_count"].isna().sum()
)

display(
    a1_base_df[
        [
            "forecast_datetime",
            "store_id",
            "product_id",
            "time_slot",
            "visitor_ratio",
            "visitor_count",
            "lag_1h_visitor_count",
            "rolling_3h_visitor_mean",
            "cumulative_daily_visitors",
            "visitor_data_available"
        ]
    ].tail(20)
)

방문객 피처 결합 전 행 수: 492404
방문객 피처 결합 후 행 수: 492404
결합으로 증가·감소한 행 수: 0
날짜·점포·상품·시간 중복 수: 0
visitor_ratio 미결합 행 수: 0
visitor_data_available 미결합 행 수: 0
visitor_count 결측 행 수: 0


,forecast_datetime,store_id,product_id,time_slot,visitor_ratio,visitor_count,lag_1h_visitor_count,rolling_3h_visitor_mean,cumulative_daily_visitors,visitor_data_available
492384,2025-12-30 16:00:00,S03,P038,afternoon,0.2300,45.0000,52.0000,47.0000,279.0000,1
492385,2025-12-30 17:00:00,S03,P038,afternoon,0.2300,60.0000,45.0000,48.0000,324.0000,1
492386,2025-12-30 18:00:00,S03,P038,evening,0.3000,62.0000,60.0000,52.3333,384.0000,1
492387,2025-12-30 19:00:00,S03,P038,evening,0.3000,57.0000,62.0000,55.6667,446.0000,1
492388,2025-12-30 20:00:00,S03,P038,evening,0.3000,75.0000,57.0000,59.6667,503.0000,1
492389,2025-12-30 21:00:00,S03,P038,closing,0.1500,39.0000,75.0000,64.6667,578.0000,1
492390,2025-12-30 22:00:00,S03,P038,closing,0.1500,55.0000,39.0000,57.0000,617.0000,1
492391,2025-12-31 10:00:00,S03,P038,morning,0.1200,51.0000,NaN,NaN,0.0000,1
492392,2025-12-31 11:00:00,S03,P038,morning,0.1200,38.0000,51.0000,51.0000,51.0000,1
492393,2025-12-31 12:00:00,S03,P038,lunch,0.2000,41.0000,38.0000,44.5000,89.0000,1


In [208]:
# # ============================================================
# # 1-7-0. 재고·유통기한 데이터 사전 점검
# # ============================================================

# inventory_expiry_check_df = inventory_df.copy()

# # 날짜형 정리
# for col in [
#     "current_date",
#     "manufacture_date",
#     "expiry_date"
# ]:
#     inventory_expiry_check_df[col] = pd.to_datetime(
#         inventory_expiry_check_df[col],
#         errors="coerce"
#     ).dt.normalize()

# # 숫자형 정리
# numeric_check_columns = [
#     "days_to_expiry",
#     "current_stock_qty",
#     "reserved_qty",
#     "available_qty",
#     "freshness_score",
#     "discount_rate"
# ]

# for col in numeric_check_columns:
#     inventory_expiry_check_df[col] = pd.to_numeric(
#         inventory_expiry_check_df[col],
#         errors="coerce"
#     )

# # 상품 카테고리와 전체 소비기한 연결
# product_expiry_reference_df = (
#     product_df[
#         [
#             "product_id",
#             "category",
#             "shelf_life_days"
#         ]
#     ]
#     .drop_duplicates("product_id")
#     .copy()
# )

# product_expiry_reference_df["shelf_life_days"] = pd.to_numeric(
#     product_expiry_reference_df["shelf_life_days"],
#     errors="coerce"
# )

# inventory_expiry_check_df = inventory_expiry_check_df.merge(
#     product_expiry_reference_df,
#     on="product_id",
#     how="left",
#     validate="many_to_one"
# )

# # 날짜로 잔여 소비기한 재계산
# inventory_expiry_check_df[
#     "calculated_days_to_expiry"
# ] = (
#     inventory_expiry_check_df["expiry_date"]
#     - inventory_expiry_check_df["current_date"]
# ).dt.days

# # 원본 days_to_expiry와 날짜 계산 결과 비교
# inventory_expiry_check_df[
#     "days_to_expiry_mismatch"
# ] = (
#     inventory_expiry_check_df["days_to_expiry"]
#     != inventory_expiry_check_df[
#         "calculated_days_to_expiry"
#     ]
# )

# # 재고 관계 확인
# inventory_expiry_check_df[
#     "calculated_available_qty"
# ] = (
#     inventory_expiry_check_df["current_stock_qty"]
#     - inventory_expiry_check_df["reserved_qty"]
# )

# inventory_expiry_check_df[
#     "available_qty_mismatch"
# ] = (
#     inventory_expiry_check_df["available_qty"]
#     != inventory_expiry_check_df[
#         "calculated_available_qty"
#     ]
# )

# print(
#     "days_to_expiry 날짜 계산 불일치 수:",
#     inventory_expiry_check_df[
#         "days_to_expiry_mismatch"
#     ].sum()
# )

# print(
#     "available_qty 계산 불일치 수:",
#     inventory_expiry_check_df[
#         "available_qty_mismatch"
#     ].sum()
# )

# print(
#     "days_to_expiry 결측 수:",
#     inventory_expiry_check_df[
#         "days_to_expiry"
#     ].isna().sum()
# )

# print(
#     "유통기한 경과 행 수:",
#     (
#         inventory_expiry_check_df["days_to_expiry"] < 0
#     ).sum()
# )

# print(
#     "판매가능수량 음수 행 수:",
#     (
#         inventory_expiry_check_df["available_qty"] < 0
#     ).sum()
# )

# print(
#     "freshness_score 0~100 범위 이탈:",
#     (
#         inventory_expiry_check_df["freshness_score"].notna()
#         & ~inventory_expiry_check_df[
#             "freshness_score"
#         ].between(0, 100)
#     ).sum()
# )

# print(
#     "할인율 0~40 범위 이탈:",
#     (
#         inventory_expiry_check_df["discount_rate"].notna()
#         & ~inventory_expiry_check_df[
#             "discount_rate"
#         ].between(0, 40)
#     ).sum()
# )

# # 임박재고 구간별 수량
# available_qty_safe = (
#     inventory_expiry_check_df["available_qty"]
#     .fillna(0)
#     .clip(lower=0)
# )

# inventory_expiry_check_df[
#     "expiry_today_qty"
# ] = np.where(
#     inventory_expiry_check_df["days_to_expiry"] == 0,
#     available_qty_safe,
#     0
# )

# inventory_expiry_check_df[
#     "near_expiry_1d_qty"
# ] = np.where(
#     inventory_expiry_check_df[
#         "days_to_expiry"
#     ].between(0, 1),
#     available_qty_safe,
#     0
# )

# inventory_expiry_check_df[
#     "markdown_window_2d_qty"
# ] = np.where(
#     inventory_expiry_check_df[
#         "days_to_expiry"
#     ].between(0, 2),
#     available_qty_safe,
#     0
# )

# expiry_category_summary_df = (
#     inventory_expiry_check_df
#     .groupby("category", as_index=False)
#     .agg(
#         row_count=("inventory_id", "size"),
#         min_days_to_expiry=("days_to_expiry", "min"),
#         median_days_to_expiry=("days_to_expiry", "median"),
#         max_days_to_expiry=("days_to_expiry", "max"),
#         total_available_qty=("available_qty", "sum"),
#         expiry_today_qty=("expiry_today_qty", "sum"),
#         near_expiry_1d_qty=("near_expiry_1d_qty", "sum"),
#         markdown_window_2d_qty=(
#             "markdown_window_2d_qty",
#             "sum"
#         )
#     )
# )

# expiry_category_summary_df[
#     "near_expiry_1d_ratio"
# ] = np.where(
#     expiry_category_summary_df["total_available_qty"] > 0,
#     expiry_category_summary_df["near_expiry_1d_qty"]
#     / expiry_category_summary_df["total_available_qty"],
#     np.nan
# )

# expiry_category_summary_df[
#     "markdown_window_2d_ratio"
# ] = np.where(
#     expiry_category_summary_df["total_available_qty"] > 0,
#     expiry_category_summary_df[
#         "markdown_window_2d_qty"
#     ]
#     / expiry_category_summary_df["total_available_qty"],
#     np.nan
# )

# print("\n카테고리별 유통기한·임박재고 현황")
# display(expiry_category_summary_df)

In [209]:
# # ============================================================
# # 유통기한 경과 재고 상세 확인
# # ============================================================

# expired_inventory_check_df = (
#     inventory_expiry_check_df.loc[
#         inventory_expiry_check_df["days_to_expiry"] < 0
#     ]
#     .copy()
# )

# print(
#     "유통기한 경과 행 수:",
#     len(expired_inventory_check_df)
# )

# print(
#     "유통기한 경과 재고 중 available_qty > 0 행 수:",
#     (
#         expired_inventory_check_df["available_qty"] > 0
#     ).sum()
# )

# print(
#     "유통기한 경과 판매가능수량 합계:",
#     expired_inventory_check_df["available_qty"]
#     .clip(lower=0)
#     .sum()
# )

# print(
#     "inventory_status 고유값:",
#     expired_inventory_check_df["inventory_status"]
#     .value_counts(dropna=False)
# )

# display(
#     expired_inventory_check_df[
#         [
#             "current_date",
#             "store_id",
#             "product_id",
#             "lot_id",
#             "expiry_date",
#             "days_to_expiry",
#             "current_stock_qty",
#             "reserved_qty",
#             "available_qty",
#             "inventory_status"
#         ]
#     ].head(20)
# )

In [210]:
# ============================================================
# inventory 수정본 최종 확인
# ============================================================

# 1. reserved_qty가 전부 0인지
print(
    "reserved_qty 고유값:",
    sorted(
        inventory_df["reserved_qty"]
        .dropna()
        .unique()
        .tolist()
    )
)


# 2. disposal_candidate가 판매 가능 만료 임박 상품에도 표시되는지
disposal_check_df = (
    inventory_df[
        inventory_df["disposal_candidate"] == 1
    ]
    .groupby(
        ["inventory_status"],
        dropna=False
    )
    .agg(
        row_count=("inventory_status", "size"),
        min_days_to_expiry=("days_to_expiry", "min"),
        max_days_to_expiry=("days_to_expiry", "max")
    )
    .reset_index()
)

print(
    "\ndisposal_candidate = 1인 행 수:",
    (inventory_df["disposal_candidate"] == 1).sum()
)

display(disposal_check_df)

reserved_qty 고유값: [0]

disposal_candidate = 1인 행 수: 11426


,inventory_status,row_count,min_days_to_expiry,max_days_to_expiry
0,DISCOUNT,5563,0,2
1,EXPIRED,5863,-2,-1


In [211]:
# ============================================================
# 1-7-1. 재고 로트 집계를 위한 계산 컬럼 생성
# ============================================================

inventory_work_df = inventory_df.copy()

# ------------------------------------------------------------
# 1. 숫자형 변환
# discount_rate는 0~40 퍼센트 값
# ------------------------------------------------------------

numeric_inventory_columns = [
    "days_to_expiry",
    "freshness_score",
    "current_stock_qty",
    "available_qty",
    "reserved_qty",
    "discount_rate"
]

for col in numeric_inventory_columns:
    inventory_work_df[col] = pd.to_numeric(
        inventory_work_df[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 2. 할인율 변환
# 예: 11 → 0.11, 40 → 0.40
# ------------------------------------------------------------

inventory_work_df["current_discount_rate_ratio"] = (
    inventory_work_df["discount_rate"] / 100
)

# ------------------------------------------------------------
# 3. 실제 기록된 판매가능수량
# ------------------------------------------------------------

inventory_work_df["physical_available_qty"] = (
    inventory_work_df["available_qty"]
    .fillna(0)
    .clip(lower=0)
)

# ------------------------------------------------------------
# 4. A-1에서 사용할 판매 가능 재고
# 유통기한 당일은 포함하고, 이미 지난 재고는 제외
# inventory_status가 EXPIRED인 경우도 제외
# ------------------------------------------------------------

saleable_inventory_mask = (
    inventory_work_df["days_to_expiry"].ge(0)
    & inventory_work_df["inventory_status"].ne("EXPIRED")
)

inventory_work_df["inventory_weight"] = np.where(
    saleable_inventory_mask,
    inventory_work_df["physical_available_qty"],
    0
)

# ------------------------------------------------------------
# 5. 가중평균 계산용 분자
# ------------------------------------------------------------

inventory_work_df["weighted_days_numerator"] = (
    inventory_work_df["days_to_expiry"]
    * inventory_work_df["inventory_weight"]
)

inventory_work_df["weighted_freshness_numerator"] = (
    inventory_work_df["freshness_score"]
    * inventory_work_df["inventory_weight"]
)

inventory_work_df["weighted_discount_numerator"] = (
    inventory_work_df["current_discount_rate_ratio"]
    * inventory_work_df["inventory_weight"]
)

# ------------------------------------------------------------
# 6. 유통기한 구간별 재고
# ------------------------------------------------------------

# D-Day 재고
inventory_work_df["expiry_today_qty_row"] = np.where(
    inventory_work_df["days_to_expiry"] == 0,
    inventory_work_df["inventory_weight"],
    0
)

# 근접 유통기한 재고: D-Day와 D-1
inventory_work_df["near_expiry_qty_row"] = np.where(
    inventory_work_df["days_to_expiry"].between(0, 1),
    inventory_work_df["inventory_weight"],
    0
)

# 할인 개입 검토 재고: D-Day부터 D-2
inventory_work_df["markdown_window_2d_qty_row"] = np.where(
    inventory_work_df["days_to_expiry"].between(0, 2),
    inventory_work_df["inventory_weight"],
    0
)

# 이미 유통기한이 지난 재고
# DISCOUNT 상태로 남은 333행의 수량도 여기에 포함
inventory_work_df["expired_qty_row"] = np.where(
    inventory_work_df["days_to_expiry"] < 0,
    inventory_work_df["physical_available_qty"],
    0
)

# ------------------------------------------------------------
# 7. 결과 확인
# ------------------------------------------------------------

print(
    "A-1 판매 가능 재고 합계:",
    inventory_work_df["inventory_weight"].sum()
)

print(
    "D-Day 재고 합계:",
    inventory_work_df["expiry_today_qty_row"].sum()
)

print(
    "D-1 이내 근접 재고 합계:",
    inventory_work_df["near_expiry_qty_row"].sum()
)

print(
    "D-2 이내 할인 검토 재고 합계:",
    inventory_work_df["markdown_window_2d_qty_row"].sum()
)

print(
    "유통기한 경과 재고 합계:",
    inventory_work_df["expired_qty_row"].sum()
)

print(
    "유통기한 경과인데 DISCOUNT 상태인 행 수:",
    (
        (inventory_work_df["days_to_expiry"] < 0)
        & (inventory_work_df["inventory_status"] == "DISCOUNT")
    ).sum()
)

display(
    inventory_work_df[
        [
            "current_date",
            "store_id",
            "product_id",
            "lot_id",
            "days_to_expiry",
            "available_qty",
            "inventory_status",
            "inventory_weight",
            "expiry_today_qty_row",
            "near_expiry_qty_row",
            "markdown_window_2d_qty_row",
            "expired_qty_row",
            "discount_rate",
            "current_discount_rate_ratio"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 품절 플래그 형식 정리
# ------------------------------------------------------------

inventory_df["sold_out_flag"] = (
    pd.to_numeric(
        inventory_df["sold_out_flag"],
        errors="coerce"
    )
    .fillna(0)
    .astype("int8")
)

# inventory_status와 sold_out_flag 일치 여부 확인
sold_out_mismatch_count = (
    (
        inventory_df["inventory_status"].eq("SOLD_OUT")
        !=
        inventory_df["sold_out_flag"].eq(1)
    )
    .sum()
)

print(
    "SOLD_OUT 상태와 sold_out_flag 불일치:",
    sold_out_mismatch_count
)

A-1 판매 가능 재고 합계: 363911
D-Day 재고 합계: 21829
D-1 이내 근접 재고 합계: 78088
D-2 이내 할인 검토 재고 합계: 125613
유통기한 경과 재고 합계: 0
유통기한 경과인데 DISCOUNT 상태인 행 수: 0


,current_date,store_id,product_id,lot_id,days_to_expiry,available_qty,inventory_status,inventory_weight,expiry_today_qty_row,near_expiry_qty_row,markdown_window_2d_qty_row,expired_qty_row,discount_rate,current_discount_rate_ratio
0,2025-01-01,S01,P001,LOT0000002,0,5,DISCOUNT,5,5,5,5,0,36,0.3600
1,2025-01-01,S01,P001,LOT0000003,6,10,NORMAL,10,0,0,0,0,0,0.0000
2,2025-01-01,S01,P001,LOT0000001,7,18,NORMAL,18,0,0,0,0,0,0.0000
3,2025-01-01,S01,P002,LOT0000004,3,1,NORMAL,1,0,0,0,0,0,0.0000
4,2025-01-01,S01,P002,LOT0000005,3,23,NORMAL,23,0,0,0,0,0,0.0000
5,2025-01-01,S01,P003,LOT0000006,0,0,SOLD_OUT,0,0,0,0,0,0,0.0000
6,2025-01-01,S01,P003,LOT0000007,2,15,NORMAL,15,0,0,15,0,0,0.0000
7,2025-01-01,S01,P003,LOT0000008,2,15,NORMAL,15,0,0,15,0,0,0.0000
8,2025-01-01,S01,P003,LOT0000009,2,9,NORMAL,9,0,0,9,0,0,0.0000
9,2025-01-01,S01,P004,LOT0000010,0,12,DISCOUNT,12,12,12,12,0,35,0.3500


SOLD_OUT 상태와 sold_out_flag 불일치: 0


In [212]:
# ============================================================
# 1-7-2. 점포·상품·날짜 단위 재고 집계
# 품절 플래그 반영
# ============================================================

inventory_agg_work_df = inventory_work_df.copy()


# ------------------------------------------------------------
# 0. sold_out_flag 형식 및 상태 일치 검증
# ------------------------------------------------------------

required_soldout_columns = [
    "sold_out_flag",
    "inventory_status"
]

missing_soldout_columns = [
    column
    for column in required_soldout_columns
    if column not in inventory_agg_work_df.columns
]

if missing_soldout_columns:
    raise ValueError(
        "inventory.csv에 품절 관련 필수 컬럼이 없습니다: "
        f"{missing_soldout_columns}"
    )


# sold_out_flag를 0·1 정수형으로 통일
inventory_agg_work_df["sold_out_flag"] = (
    pd.to_numeric(
        inventory_agg_work_df["sold_out_flag"],
        errors="coerce"
    )
    .fillna(0)
    .astype("int8")
)


# 0과 1 이외의 값이 있는지 확인
invalid_sold_out_flag_count = (
    ~inventory_agg_work_df[
        "sold_out_flag"
    ].isin([0, 1])
).sum()

if invalid_sold_out_flag_count > 0:
    raise ValueError(
        "sold_out_flag에 0 또는 1 이외의 값이 있습니다."
    )


# inventory_status와 sold_out_flag 일치 여부
sold_out_status_mismatch_count = (
    (
        inventory_agg_work_df[
            "inventory_status"
        ].eq("SOLD_OUT")
    )
    !=
    (
        inventory_agg_work_df[
            "sold_out_flag"
        ].eq(1)
    )
).sum()

print(
    "SOLD_OUT 상태와 sold_out_flag 불일치:",
    sold_out_status_mismatch_count
)

if sold_out_status_mismatch_count > 0:
    raise ValueError(
        "inventory_status와 sold_out_flag가 일치하지 않습니다."
    )


# 품절된 고유 로트 수 계산용
inventory_agg_work_df["sold_out_lot_id"] = (
    inventory_agg_work_df["lot_id"]
    .where(
        inventory_agg_work_df[
            "sold_out_flag"
        ].eq(1)
    )
)


# ------------------------------------------------------------
# 1. 판매 가능한 재고가 존재하는 로트만 계산에 사용
# ------------------------------------------------------------

saleable_lot_mask = (
    inventory_agg_work_df["inventory_weight"] > 0
)


# 판매 가능한 로트의 잔여 유통기한
inventory_agg_work_df["saleable_days_to_expiry"] = (
    inventory_agg_work_df["days_to_expiry"]
    .where(saleable_lot_mask)
)


# 판매 가능한 로트의 신선도
inventory_agg_work_df["saleable_freshness_score"] = (
    inventory_agg_work_df["freshness_score"]
    .where(saleable_lot_mask)
)


# 판매 가능한 로트의 할인율
inventory_agg_work_df["saleable_discount_rate"] = (
    inventory_agg_work_df[
        "current_discount_rate_ratio"
    ].where(saleable_lot_mask)
)


# 판매 가능한 로트 수 계산용
inventory_agg_work_df["saleable_lot_id"] = (
    inventory_agg_work_df["lot_id"]
    .where(saleable_lot_mask)
)


# 판매 불가능으로 제외된 수량
# 경과 재고 및 기타 판매 불가능 상태 포함
inventory_agg_work_df[
    "excluded_unsaleable_qty_row"
] = (
    inventory_agg_work_df["physical_available_qty"]
    - inventory_agg_work_df["inventory_weight"]
).clip(lower=0)


# ------------------------------------------------------------
# 2. 점포·상품·날짜 단위 집계
# ------------------------------------------------------------

inventory_agg_df = (
    inventory_agg_work_df
    .groupby(
        [
            "current_date",
            "store_id",
            "product_id"
        ],
        as_index=False
    )
    .agg(
        # 실제 시스템에 기록된 현재 재고
        total_current_stock_qty=(
            "current_stock_qty",
            "sum"
        ),

        # 원본 available_qty 합계
        # 유통기한 경과 재고도 포함될 수 있음
        total_physical_available_qty=(
            "physical_available_qty",
            "sum"
        ),

        # A-1에서 사용할 실제 판매 가능 재고
        # 유통기한 경과 재고 제외
        total_available_qty=(
            "inventory_weight",
            "sum"
        ),

        total_reserved_qty=(
            "reserved_qty",
            "sum"
        ),

        # 전체 로트 수
        lot_count=(
            "lot_id",
            "nunique"
        ),

        # 판매 가능한 재고가 있는 로트 수
        saleable_lot_count=(
            "saleable_lot_id",
            "nunique"
        ),

        # 품절된 로트 수
        sold_out_lot_count=(
            "sold_out_lot_id",
            "nunique"
        ),

        # 판매 가능 로트만 대상으로 한 잔여 유통기한
        min_days_to_expiry=(
            "saleable_days_to_expiry",
            "min"
        ),

        mean_days_to_expiry=(
            "saleable_days_to_expiry",
            "mean"
        ),

        # 판매 가능 로트만 대상으로 한 신선도
        mean_freshness_score=(
            "saleable_freshness_score",
            "mean"
        ),

        # 판매 가능 로트만 대상으로 한 평균 할인율
        mean_current_discount_rate=(
            "saleable_discount_rate",
            "mean"
        ),

        # D-Day 재고
        expiry_today_qty=(
            "expiry_today_qty_row",
            "sum"
        ),

        # D-Day~D-1 근접 유통기한 재고
        near_expiry_qty=(
            "near_expiry_qty_row",
            "sum"
        ),

        # D-Day~D-2 할인 개입 검토 재고
        markdown_window_2d_qty=(
            "markdown_window_2d_qty_row",
            "sum"
        ),

        # 유통기한 경과 재고
        expired_qty=(
            "expired_qty_row",
            "sum"
        ),

        # A-1 판매 가능 재고에서 제외된 수량
        excluded_unsaleable_qty=(
            "excluded_unsaleable_qty_row",
            "sum"
        ),

        # 가중평균 계산용 분자
        weighted_days_numerator=(
            "weighted_days_numerator",
            "sum"
        ),

        weighted_freshness_numerator=(
            "weighted_freshness_numerator",
            "sum"
        ),

        weighted_discount_numerator=(
            "weighted_discount_numerator",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 3. 품절 관련 집계 플래그 생성
# ------------------------------------------------------------

# 해당 날짜에 하나 이상의 로트가 SOLD_OUT이었는지
inventory_agg_df[
    "any_sold_out_lot_flag"
] = (
    inventory_agg_df["sold_out_lot_count"] > 0
).astype("int8")


# 모든 로트가 SOLD_OUT이었는지
inventory_agg_df[
    "all_lots_sold_out_flag"
] = (
    (inventory_agg_df["lot_count"] > 0)
    &
    (
        inventory_agg_df["sold_out_lot_count"]
        == inventory_agg_df["lot_count"]
    )
).astype("int8")


# 하나 이상의 품절 로트가 존재하고,
# 다른 판매 가능 재고도 남아 있지 않은 경우
#
# 이 플래그는 상품 전체 품절에 가까운 상태를 식별하며
# 현재 시점의 일반 예측 피처가 아니라
# 검열된 수요 학습 행을 식별하는 용도로 사용
inventory_agg_df[
    "full_product_sold_out_flag"
] = (
    inventory_agg_df[
        "any_sold_out_lot_flag"
    ].eq(1)
    &
    inventory_agg_df[
        "total_available_qty"
    ].le(0)
).astype("int8")


# 일부 로트만 품절됐고
# 다른 판매 가능 로트가 남아 있는 경우
inventory_agg_df[
    "partial_lot_sold_out_flag"
] = (
    inventory_agg_df[
        "any_sold_out_lot_flag"
    ].eq(1)
    &
    inventory_agg_df[
        "full_product_sold_out_flag"
    ].eq(0)
).astype("int8")


# ------------------------------------------------------------
# 4. 재고량 가중 잔여 유통기한
# 분자와 분모 모두 판매 가능 재고 기준
# ------------------------------------------------------------

inventory_agg_df["weighted_days_to_expiry"] = np.where(
    inventory_agg_df["total_available_qty"] > 0,
    (
        inventory_agg_df["weighted_days_numerator"]
        / inventory_agg_df["total_available_qty"]
    ),
    inventory_agg_df["mean_days_to_expiry"]
)


# ------------------------------------------------------------
# 5. 재고량 가중 신선도
# ------------------------------------------------------------

inventory_agg_df["weighted_freshness_score"] = np.where(
    inventory_agg_df["total_available_qty"] > 0,
    (
        inventory_agg_df["weighted_freshness_numerator"]
        / inventory_agg_df["total_available_qty"]
    ),
    inventory_agg_df["mean_freshness_score"]
)


# ------------------------------------------------------------
# 6. 근접 유통기한 재고 비율
# ------------------------------------------------------------

inventory_agg_df["near_expiry_ratio"] = np.where(
    inventory_agg_df["total_available_qty"] > 0,
    (
        inventory_agg_df["near_expiry_qty"]
        / inventory_agg_df["total_available_qty"]
    ),
    0
)


# ------------------------------------------------------------
# 7. D-2 이내 할인 개입 검토 재고 비율
# ------------------------------------------------------------

inventory_agg_df["markdown_window_2d_ratio"] = np.where(
    inventory_agg_df["total_available_qty"] > 0,
    (
        inventory_agg_df["markdown_window_2d_qty"]
        / inventory_agg_df["total_available_qty"]
    ),
    0
)


# ------------------------------------------------------------
# 8. 유통기한 경과 재고 비율
# 원본 available_qty 중 경과 재고 비중
# ------------------------------------------------------------

inventory_agg_df["expired_qty_ratio"] = np.where(
    inventory_agg_df[
        "total_physical_available_qty"
    ] > 0,
    (
        inventory_agg_df["expired_qty"]
        / inventory_agg_df[
            "total_physical_available_qty"
        ]
    ),
    0
)


# ------------------------------------------------------------
# 9. 판매 가능 재고량 가중 할인율
# 결과는 0~0.4 비율
# ------------------------------------------------------------

inventory_agg_df["current_discount_rate"] = np.where(
    inventory_agg_df["total_available_qty"] > 0,
    (
        inventory_agg_df["weighted_discount_numerator"]
        / inventory_agg_df["total_available_qty"]
    ),
    inventory_agg_df["mean_current_discount_rate"]
)


# ------------------------------------------------------------
# 10. 계산용 중간 컬럼 제거
# ------------------------------------------------------------

inventory_agg_df = inventory_agg_df.drop(
    columns=[
        "mean_current_discount_rate",
        "weighted_days_numerator",
        "weighted_freshness_numerator",
        "weighted_discount_numerator"
    ]
)


# ------------------------------------------------------------
# 11. 학습 테이블 조인 키와 이름 통일
# ------------------------------------------------------------

inventory_agg_df = inventory_agg_df.rename(
    columns={
        "current_date": "date"
    }
)

inventory_agg_df = (
    inventory_agg_df
    .sort_values(
        [
            "date",
            "store_id",
            "product_id"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. 기존 재고 집계 결과 점검
# ------------------------------------------------------------

print(
    "재고 집계 테이블 크기:",
    inventory_agg_df.shape
)

print(
    "날짜·점포·상품 중복 수:",
    inventory_agg_df.duplicated(
        [
            "date",
            "store_id",
            "product_id"
        ]
    ).sum()
)

print(
    "판매 가능 재고보다 근접 재고가 큰 행 수:",
    (
        inventory_agg_df["near_expiry_qty"]
        > inventory_agg_df["total_available_qty"]
    ).sum()
)

print(
    "판매 가능 재고보다 D-2 재고가 큰 행 수:",
    (
        inventory_agg_df["markdown_window_2d_qty"]
        > inventory_agg_df["total_available_qty"]
    ).sum()
)

print(
    "near_expiry_ratio 0~1 범위 이탈:",
    (
        ~inventory_agg_df[
            "near_expiry_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "markdown_window_2d_ratio 0~1 범위 이탈:",
    (
        ~inventory_agg_df[
            "markdown_window_2d_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "가중 할인율 0~0.4 범위 이탈:",
    (
        inventory_agg_df[
            "current_discount_rate"
        ].notna()
        &
        ~inventory_agg_df[
            "current_discount_rate"
        ].between(0, 0.4)
    ).sum()
)

print(
    "원본 판매가능수량 합계:",
    inventory_agg_df[
        "total_physical_available_qty"
    ].sum()
)

print(
    "A-1 판매 가능 재고 합계:",
    inventory_agg_df[
        "total_available_qty"
    ].sum()
)

print(
    "제외된 판매 불가능 재고 합계:",
    inventory_agg_df[
        "excluded_unsaleable_qty"
    ].sum()
)

print(
    "유통기한 경과 재고 합계:",
    inventory_agg_df["expired_qty"].sum()
)


# ------------------------------------------------------------
# 13. 품절 관련 결과 점검
# ------------------------------------------------------------

print()
print("===== 품절 플래그 집계 결과 =====")

print(
    "원본 SOLD_OUT 로트 행 수:",
    int(
        inventory_agg_work_df[
            "sold_out_flag"
        ].sum()
    )
)

print(
    "집계된 품절 로트 수 합계:",
    int(
        inventory_agg_df[
            "sold_out_lot_count"
        ].sum()
    )
)

print(
    "하나 이상의 품절 로트가 있는 점포·상품·날짜 수:",
    int(
        inventory_agg_df[
            "any_sold_out_lot_flag"
        ].sum()
    )
)

print(
    "모든 로트가 품절인 점포·상품·날짜 수:",
    int(
        inventory_agg_df[
            "all_lots_sold_out_flag"
        ].sum()
    )
)

print(
    "판매 가능 재고가 없는 상품 전체 품절 수:",
    int(
        inventory_agg_df[
            "full_product_sold_out_flag"
        ].sum()
    )
)

print(
    "일부 로트만 품절이고 다른 재고가 남은 수:",
    int(
        inventory_agg_df[
            "partial_lot_sold_out_flag"
        ].sum()
    )
)

print(
    "전체 품절인데 판매 가능 재고가 남은 행 수:",
    (
        inventory_agg_df[
            "full_product_sold_out_flag"
        ].eq(1)
        &
        inventory_agg_df[
            "total_available_qty"
        ].gt(0)
    ).sum()
)

print(
    "품절 로트 수가 전체 로트 수보다 큰 행 수:",
    (
        inventory_agg_df[
            "sold_out_lot_count"
        ]
        >
        inventory_agg_df[
            "lot_count"
        ]
    ).sum()
)


display(
    inventory_agg_df.head(10)
)

SOLD_OUT 상태와 sold_out_flag 불일치: 0
재고 집계 테이블 크기: (41593, 28)
날짜·점포·상품 중복 수: 0
판매 가능 재고보다 근접 재고가 큰 행 수: 0
판매 가능 재고보다 D-2 재고가 큰 행 수: 0
near_expiry_ratio 0~1 범위 이탈: 0
markdown_window_2d_ratio 0~1 범위 이탈: 0
가중 할인율 0~0.4 범위 이탈: 0
원본 판매가능수량 합계: 363911
A-1 판매 가능 재고 합계: 363911
제외된 판매 불가능 재고 합계: 0
유통기한 경과 재고 합계: 0

===== 품절 플래그 집계 결과 =====
원본 SOLD_OUT 로트 행 수: 13870
집계된 품절 로트 수 합계: 13870
하나 이상의 품절 로트가 있는 점포·상품·날짜 수: 13083
모든 로트가 품절인 점포·상품·날짜 수: 34
판매 가능 재고가 없는 상품 전체 품절 수: 34
일부 로트만 품절이고 다른 재고가 남은 수: 13049
전체 품절인데 판매 가능 재고가 남은 행 수: 0
품절 로트 수가 전체 로트 수보다 큰 행 수: 0


,date,store_id,product_id,total_current_stock_qty,total_physical_available_qty,total_available_qty,total_reserved_qty,lot_count,saleable_lot_count,sold_out_lot_count,min_days_to_expiry,mean_days_to_expiry,mean_freshness_score,expiry_today_qty,near_expiry_qty,markdown_window_2d_qty,expired_qty,excluded_unsaleable_qty,any_sold_out_lot_flag,all_lots_sold_out_flag,full_product_sold_out_flag,partial_lot_sold_out_flag,weighted_days_to_expiry,weighted_freshness_score,near_expiry_ratio,markdown_window_2d_ratio,expired_qty_ratio,current_discount_rate
0,2025-01-01,S01,P001,33,33,33,0,3,3,0,0.0000,4.3333,0.4849,5,5,5,0,0,0,0,0,0,5.6364,0.6033,0.1515,0.1515,0.0000,0.0545
1,2025-01-01,S01,P002,24,24,24,0,2,2,0,3.0000,3.0000,0.6667,0,0,0,0,0,0,0,0,0,3.0000,0.6667,0.0000,0.0000,0.0000,0.0000
2,2025-01-01,S01,P003,39,39,39,0,4,3,1,2.0000,2.0000,0.6495,0,0,39,0,0,1,0,0,1,2.0000,0.6495,0.0000,1.0000,0.0000,0.0000
3,2025-01-01,S01,P004,27,27,27,0,2,2,0,0.0000,3.5000,0.5625,12,12,12,0,0,0,0,0,0,3.8889,0.6111,0.4444,0.4444,0.0000,0.1556
4,2025-01-01,S01,P005,20,20,20,0,2,2,0,2.0000,4.5000,0.6875,0,0,1,0,0,0,0,0,0,6.7500,0.9688,0.0000,0.0500,0.0000,0.0075
5,2025-01-01,S01,P006,37,37,37,0,3,3,0,0.0000,0.6667,0.2774,6,37,37,0,0,0,0,0,0,0.8378,0.3165,1.0000,1.0000,0.0000,0.1908
6,2025-01-01,S01,P007,30,30,30,0,2,2,0,1.0000,1.0000,0.3536,0,30,30,0,0,0,0,0,0,1.0000,0.3536,1.0000,1.0000,0.0000,0.1600
7,2025-01-01,S01,P008,45,45,45,0,4,4,0,1.0000,4.5000,0.6875,0,7,7,0,0,0,0,0,0,4.2000,0.6500,0.1556,0.1556,0.0000,0.1027
8,2025-01-01,S01,P009,19,19,19,0,2,2,0,11.0000,12.0000,0.9042,0,0,0,0,0,0,0,0,0,12.2632,0.9170,0.0000,0.0000,0.0000,0.0000
9,2025-01-01,S01,P010,20,20,20,0,2,2,0,10.0000,10.5000,0.8301,0,0,0,0,0,0,0,0,0,10.9000,0.8503,0.0000,0.0000,0.0000,0.0000


In [213]:
# ============================================================
# 1-7-3. 일별 재고 피처를 시간별 A-1 테이블에 결합
# 품절 진단 플래그 포함
# ============================================================


# ------------------------------------------------------------
# 1. 결합할 재고 피처
# ------------------------------------------------------------

inventory_feature_columns = [
    "date",
    "store_id",
    "product_id",

    # 전체 및 판매 가능 재고
    "total_current_stock_qty",
    "total_physical_available_qty",
    "total_available_qty",
    "total_reserved_qty",
    "excluded_unsaleable_qty",

    # 로트 수
    "lot_count",
    "saleable_lot_count",

    # 품절 관련 진단 컬럼
    # 일반 예측 피처가 아니라 검열 수요 식별용
    "sold_out_lot_count",
    "any_sold_out_lot_flag",
    "all_lots_sold_out_flag",
    "full_product_sold_out_flag",
    "partial_lot_sold_out_flag",

    # 잔여 유통기한
    "min_days_to_expiry",
    "mean_days_to_expiry",
    "weighted_days_to_expiry",

    # 신선도
    "mean_freshness_score",
    "weighted_freshness_score",

    # 유통기한 임박 재고
    "expiry_today_qty",
    "near_expiry_qty",
    "near_expiry_ratio",

    # D-2 이내 할인 개입 검토 재고
    "markdown_window_2d_qty",
    "markdown_window_2d_ratio",

    # 유통기한 경과 재고
    "expired_qty",
    "expired_qty_ratio",

    # 재고량 가중 평균 할인율
    "current_discount_rate"
]


# ------------------------------------------------------------
# 2. 결합 대상 컬럼 존재 여부 확인
# ------------------------------------------------------------

missing_inventory_feature_columns = [
    column
    for column in inventory_feature_columns
    if column not in inventory_agg_df.columns
]

if missing_inventory_feature_columns:
    raise ValueError(
        "inventory_agg_df에 결합할 컬럼이 없습니다: "
        f"{missing_inventory_feature_columns}"
    )


# ------------------------------------------------------------
# 3. 결합 키 자료형 정리
# ------------------------------------------------------------

a1_base_df["date"] = pd.to_datetime(
    a1_base_df["date"],
    errors="coerce"
).dt.normalize()

inventory_agg_df["date"] = pd.to_datetime(
    inventory_agg_df["date"],
    errors="coerce"
).dt.normalize()


a1_base_df["store_id"] = (
    a1_base_df["store_id"]
    .astype(str)
    .str.strip()
)

inventory_agg_df["store_id"] = (
    inventory_agg_df["store_id"]
    .astype(str)
    .str.strip()
)


a1_base_df["product_id"] = (
    a1_base_df["product_id"]
    .astype(str)
    .str.strip()
)

inventory_agg_df["product_id"] = (
    inventory_agg_df["product_id"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 4. 재실행 시 기존 재고 피처 제거
# _x, _y 컬럼 생성 방지
# ------------------------------------------------------------

inventory_value_columns = [
    column
    for column in inventory_feature_columns
    if column not in [
        "date",
        "store_id",
        "product_id"
    ]
]

existing_inventory_columns = [
    column
    for column in inventory_value_columns
    if column in a1_base_df.columns
]

if existing_inventory_columns:
    a1_base_df = a1_base_df.drop(
        columns=existing_inventory_columns
    )

    print(
        "재실행을 위해 제거한 기존 재고 컬럼 수:",
        len(existing_inventory_columns)
    )


# ------------------------------------------------------------
# 5. 결합 전 inventory_agg_df 키 중복 확인
# ------------------------------------------------------------

inventory_key_columns = [
    "date",
    "store_id",
    "product_id"
]

inventory_key_duplicate_count = (
    inventory_agg_df
    .duplicated(
        inventory_key_columns
    )
    .sum()
)

print(
    "재고 집계 테이블 날짜·점포·상품 중복 수:",
    inventory_key_duplicate_count
)

if inventory_key_duplicate_count > 0:
    raise ValueError(
        "inventory_agg_df의 조인 키가 중복되어 있습니다."
    )


# ------------------------------------------------------------
# 6. 결합 전 행 수 저장
# ------------------------------------------------------------

rows_before_inventory_merge = len(a1_base_df)


# ------------------------------------------------------------
# 7. 일별 재고 피처 결합
# 하루 재고 한 행이 해당 날짜의 모든 영업시간 행에 연결됨
# ------------------------------------------------------------

a1_base_df = a1_base_df.merge(
    inventory_agg_df[
        inventory_feature_columns
    ],
    on=[
        "date",
        "store_id",
        "product_id"
    ],
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 8. 재고 피처 미결합 여부 기록
# 결측치 처리 전에 확인
# ------------------------------------------------------------

inventory_merge_missing_mask = (
    a1_base_df["total_available_qty"].isna()
)

inventory_merge_missing_count = int(
    inventory_merge_missing_mask.sum()
)


# ------------------------------------------------------------
# 9. 품절 관련 컬럼 결측치 및 자료형 정리
# ------------------------------------------------------------

sold_out_count_columns = [
    "sold_out_lot_count"
]

sold_out_flag_columns = [
    "any_sold_out_lot_flag",
    "all_lots_sold_out_flag",
    "full_product_sold_out_flag",
    "partial_lot_sold_out_flag"
]


# 결합되지 않은 행의 품절 로트 수는 0 처리
a1_base_df[sold_out_count_columns] = (
    a1_base_df[sold_out_count_columns]
    .fillna(0)
    .astype("int16")
)


# 결합되지 않은 행의 품절 플래그는 0 처리
a1_base_df[sold_out_flag_columns] = (
    a1_base_df[sold_out_flag_columns]
    .fillna(0)
    .astype("int8")
)


# ------------------------------------------------------------
# 10. 시계열 단위에 맞게 정렬
# ------------------------------------------------------------

a1_base_df = (
    a1_base_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. 기본 결합 결과 점검
# ------------------------------------------------------------

rows_after_inventory_merge = len(a1_base_df)

print()
print("===== 재고 피처 결합 결과 =====")

print(
    "재고 피처 결합 전 행 수:",
    rows_before_inventory_merge
)

print(
    "재고 피처 결합 후 행 수:",
    rows_after_inventory_merge
)

print(
    "결합으로 증가·감소한 행 수:",
    rows_after_inventory_merge
    - rows_before_inventory_merge
)

print(
    "날짜·점포·상품·시간 중복 수:",
    a1_base_df.duplicated(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    ).sum()
)

print(
    "재고 피처 미결합 행 수:",
    inventory_merge_missing_count
)

print(
    "판매 가능 재고 음수 행 수:",
    (
        a1_base_df["total_available_qty"] < 0
    ).sum()
)

print(
    "근접 재고가 판매 가능 재고보다 큰 행 수:",
    (
        a1_base_df["near_expiry_qty"]
        >
        a1_base_df["total_available_qty"]
    ).sum()
)

print(
    "D-2 재고가 판매 가능 재고보다 큰 행 수:",
    (
        a1_base_df["markdown_window_2d_qty"]
        >
        a1_base_df["total_available_qty"]
    ).sum()
)

print(
    "near_expiry_ratio 0~1 범위 이탈:",
    (
        a1_base_df[
            "near_expiry_ratio"
        ].notna()
        &
        ~a1_base_df[
            "near_expiry_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "markdown_window_2d_ratio 0~1 범위 이탈:",
    (
        a1_base_df[
            "markdown_window_2d_ratio"
        ].notna()
        &
        ~a1_base_df[
            "markdown_window_2d_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "현재 할인율 0~0.4 범위 이탈:",
    (
        a1_base_df[
            "current_discount_rate"
        ].notna()
        &
        ~a1_base_df[
            "current_discount_rate"
        ].between(0, 0.4)
    ).sum()
)


# ------------------------------------------------------------
# 12. 품절 관련 결과 점검
# ------------------------------------------------------------

print()
print("===== 시간별 A-1 테이블 품절 플래그 결과 =====")

print(
    "품절 로트가 하나 이상 존재하는 시간 행 수:",
    int(
        a1_base_df[
            "any_sold_out_lot_flag"
        ].sum()
    )
)

print(
    "모든 로트가 SOLD_OUT인 시간 행 수:",
    int(
        a1_base_df[
            "all_lots_sold_out_flag"
        ].sum()
    )
)

print(
    "상품 전체 품절로 표시된 시간 행 수:",
    int(
        a1_base_df[
            "full_product_sold_out_flag"
        ].sum()
    )
)

print(
    "일부 로트만 품절인 시간 행 수:",
    int(
        a1_base_df[
            "partial_lot_sold_out_flag"
        ].sum()
    )
)

print(
    "상품 전체 품절인데 판매 가능 재고가 남은 행 수:",
    (
        a1_base_df[
            "full_product_sold_out_flag"
        ].eq(1)
        &
        a1_base_df[
            "total_available_qty"
        ].gt(0)
    ).sum()
)

print(
    "일부 로트 품절인데 판매 가능 재고가 없는 행 수:",
    (
        a1_base_df[
            "partial_lot_sold_out_flag"
        ].eq(1)
        &
        a1_base_df[
            "total_available_qty"
        ].le(0)
    ).sum()
)

print(
    "품절 로트 수가 전체 로트 수보다 큰 행 수:",
    (
        a1_base_df[
            "sold_out_lot_count"
        ]
        >
        a1_base_df[
            "lot_count"
        ]
    ).sum()
)

print(
    "품절 플래그가 0·1 범위를 벗어난 행 수:",
    (
        ~a1_base_df[
            sold_out_flag_columns
        ].isin([0, 1])
    ).any(axis=1).sum()
)


# ------------------------------------------------------------
# 13. 결과 확인
# ------------------------------------------------------------

print()
print(
    "재고 피처 결합 후 A-1 테이블 크기:",
    a1_base_df.shape
)

display(
    a1_base_df[
        [
            "forecast_datetime",
            "date",
            "hour",
            "store_id",
            "product_id",

            # 재고
            "total_current_stock_qty",
            "total_physical_available_qty",
            "total_available_qty",
            "excluded_unsaleable_qty",

            # 로트
            "lot_count",
            "saleable_lot_count",

            # 품절
            "sold_out_lot_count",
            "any_sold_out_lot_flag",
            "all_lots_sold_out_flag",
            "full_product_sold_out_flag",
            "partial_lot_sold_out_flag",

            # 유통기한 및 신선도
            "min_days_to_expiry",
            "weighted_days_to_expiry",
            "weighted_freshness_score",

            # 임박·경과 재고
            "expiry_today_qty",
            "near_expiry_qty",
            "near_expiry_ratio",
            "markdown_window_2d_qty",
            "markdown_window_2d_ratio",
            "expired_qty",

            # 현재 할인율
            "current_discount_rate"
        ]
    ].head(20)
)

재고 집계 테이블 날짜·점포·상품 중복 수: 0

===== 재고 피처 결합 결과 =====
재고 피처 결합 전 행 수: 492404
재고 피처 결합 후 행 수: 492404
결합으로 증가·감소한 행 수: 0
날짜·점포·상품·시간 중복 수: 0
재고 피처 미결합 행 수: 0
판매 가능 재고 음수 행 수: 0
근접 재고가 판매 가능 재고보다 큰 행 수: 0
D-2 재고가 판매 가능 재고보다 큰 행 수: 0
near_expiry_ratio 0~1 범위 이탈: 0
markdown_window_2d_ratio 0~1 범위 이탈: 0
현재 할인율 0~0.4 범위 이탈: 0

===== 시간별 A-1 테이블 품절 플래그 결과 =====
품절 로트가 하나 이상 존재하는 시간 행 수: 165724
모든 로트가 SOLD_OUT인 시간 행 수: 427
상품 전체 품절로 표시된 시간 행 수: 427
일부 로트만 품절인 시간 행 수: 165297
상품 전체 품절인데 판매 가능 재고가 남은 행 수: 0
일부 로트 품절인데 판매 가능 재고가 없는 행 수: 0
품절 로트 수가 전체 로트 수보다 큰 행 수: 0
품절 플래그가 0·1 범위를 벗어난 행 수: 0

재고 피처 결합 후 A-1 테이블 크기: (492404, 80)


,forecast_datetime,date,hour,store_id,product_id,total_current_stock_qty,total_physical_available_qty,total_available_qty,excluded_unsaleable_qty,lot_count,saleable_lot_count,sold_out_lot_count,any_sold_out_lot_flag,all_lots_sold_out_flag,full_product_sold_out_flag,partial_lot_sold_out_flag,min_days_to_expiry,weighted_days_to_expiry,weighted_freshness_score,expiry_today_qty,near_expiry_qty,near_expiry_ratio,markdown_window_2d_qty,markdown_window_2d_ratio,expired_qty,current_discount_rate
0,2025-01-01 10:00:00,2025-01-01,10,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
1,2025-01-01 11:00:00,2025-01-01,11,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
2,2025-01-01 12:00:00,2025-01-01,12,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
3,2025-01-01 13:00:00,2025-01-01,13,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
4,2025-01-01 14:00:00,2025-01-01,14,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
5,2025-01-01 15:00:00,2025-01-01,15,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
6,2025-01-01 16:00:00,2025-01-01,16,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
7,2025-01-01 17:00:00,2025-01-01,17,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
8,2025-01-01 18:00:00,2025-01-01,18,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545
9,2025-01-01 19:00:00,2025-01-01,19,S01,P001,33,33,33,0,3,3,0,0,0,0,0,0.0000,5.6364,0.6033,5,5,0.1515,5,0.1515,0,0.0545


In [214]:
# ============================================================
# 1-8-1. 통합 기본 피처 테이블 정리
# ============================================================

a1_feature_base_df = a1_base_df.copy()

# ------------------------------------------------------------
# 1. 결합 키 및 숫자형 컬럼 정리
# ------------------------------------------------------------

a1_feature_base_df["date"] = pd.to_datetime(
    a1_feature_base_df["date"],
    errors="coerce"
).dt.normalize()

a1_feature_base_df["store_id"] = (
    a1_feature_base_df["store_id"].astype(str)
)

a1_feature_base_df["product_id"] = (
    a1_feature_base_df["product_id"].astype(str)
)

numeric_base_columns = [
    "hour",
    "open_hour",
    "close_hour",
    "total_current_stock_qty",
    "total_available_qty",
    "total_reserved_qty",
    "near_expiry_qty",
    "markdown_window_2d_qty",
    "expired_qty",
    "excluded_unsaleable_qty",
    "current_discount_rate"
]

for col in numeric_base_columns:
    a1_feature_base_df[col] = pd.to_numeric(
        a1_feature_base_df[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 2. 방문 데이터 보유 여부
# 1-6-3에서 만든 값을 그대로 사용
# ------------------------------------------------------------

a1_feature_base_df["visitor_data_available"] = (
    pd.to_numeric(
        a1_feature_base_df["visitor_data_available"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

# 방문 데이터가 없는 점포·날짜는
# 실제 방문객 관련 피처를 결측으로 유지
visitor_actual_features = [
    "visitor_count",
    "lag_1h_visitor_count",
    "rolling_3h_visitor_mean",
    "cumulative_daily_visitors"
]

a1_feature_base_df.loc[
    a1_feature_base_df["visitor_data_available"] == 0,
    visitor_actual_features
] = np.nan

# ------------------------------------------------------------
# 3. 점포-상품별 시계열 식별자
# ------------------------------------------------------------

a1_feature_base_df["series_id"] = (
    a1_feature_base_df["store_id"]
    + "_"
    + a1_feature_base_df["product_id"]
)

# ------------------------------------------------------------
# 4. 영업시간 진행 피처
# ------------------------------------------------------------

# 개점 후 경과 시간
a1_feature_base_df["hours_since_open"] = (
    a1_feature_base_df["hour"]
    - a1_feature_base_df["open_hour"]
)

# 하루 전체 영업시간
a1_feature_base_df["business_hours"] = (
    a1_feature_base_df["close_hour"]
    - a1_feature_base_df["open_hour"]
)

# 영업 진행 비율
# 예: 10~22시 영업이면 10시는 0, 21시는 11/12
a1_feature_base_df["business_progress_ratio"] = np.where(
    a1_feature_base_df["business_hours"] > 0,
    (
        a1_feature_base_df["hours_since_open"]
        / a1_feature_base_df["business_hours"]
    ),
    np.nan
)

# ------------------------------------------------------------
# 5. 재고 데이터 존재 여부
# ------------------------------------------------------------

a1_feature_base_df["inventory_data_available"] = (
    a1_feature_base_df["total_current_stock_qty"]
    .notna()
    .astype(int)
)

# ------------------------------------------------------------
# 6. 재고 비율 피처
# ------------------------------------------------------------

# 현재 전체 재고 중 실제 판매 가능한 수량 비율
a1_feature_base_df["available_stock_ratio"] = np.where(
    a1_feature_base_df["inventory_data_available"] == 0,
    np.nan,
    np.where(
        a1_feature_base_df["total_current_stock_qty"] > 0,
        (
            a1_feature_base_df["total_available_qty"]
            / a1_feature_base_df["total_current_stock_qty"]
        ),
        0
    )
)

# 현재 전체 재고 중 예약 재고 비율
a1_feature_base_df["reserved_stock_ratio"] = np.where(
    a1_feature_base_df["inventory_data_available"] == 0,
    np.nan,
    np.where(
        a1_feature_base_df["total_current_stock_qty"] > 0,
        (
            a1_feature_base_df["total_reserved_qty"]
            / a1_feature_base_df["total_current_stock_qty"]
        ),
        0
    )
)

# 현재 전체 재고 중 판매 불가능 재고 비율
a1_feature_base_df["unsaleable_stock_ratio"] = np.where(
    a1_feature_base_df["inventory_data_available"] == 0,
    np.nan,
    np.where(
        a1_feature_base_df["total_current_stock_qty"] > 0,
        (
            a1_feature_base_df["excluded_unsaleable_qty"]
            / a1_feature_base_df["total_current_stock_qty"]
        ),
        0
    )
)

# ------------------------------------------------------------
# 7. 재고 상태 여부 피처
# ------------------------------------------------------------

# 실제 판매 가능한 재고 보유 여부
a1_feature_base_df["has_available_stock"] = (
    a1_feature_base_df["total_available_qty"] > 0
).astype(int)

# D-Day~D-1 근접 유통기한 재고 보유 여부
a1_feature_base_df["has_near_expiry_stock"] = (
    a1_feature_base_df["near_expiry_qty"] > 0
).astype(int)

# D-Day~D-2 할인 검토 대상 재고 보유 여부
a1_feature_base_df["has_markdown_window_2d_stock"] = (
    a1_feature_base_df["markdown_window_2d_qty"] > 0
).astype(int)

# 유통기한 경과 재고 보유 여부
a1_feature_base_df["has_expired_stock"] = (
    a1_feature_base_df["expired_qty"] > 0
).astype(int)

# 해당 일자 재고에 할인 적용 여부
a1_feature_base_df["has_current_discount"] = (
    a1_feature_base_df["current_discount_rate"] > 0
).astype(int)

# ------------------------------------------------------------
# 8. 시계열 순서로 정렬
# ------------------------------------------------------------

a1_feature_base_df = (
    a1_feature_base_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 9. 결과 점검
# ------------------------------------------------------------

print(
    "날짜·점포·상품·시간 중복 수:",
    a1_feature_base_df.duplicated(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    ).sum()
)

print(
    "series_id 결측 수:",
    a1_feature_base_df["series_id"].isna().sum()
)

print(
    "영업시간 길이 오류 수:",
    (
        a1_feature_base_df["business_hours"] <= 0
    ).sum()
)

print(
    "영업 진행 비율 범위 이탈:",
    (
        a1_feature_base_df[
            "business_progress_ratio"
        ].notna()
        & ~a1_feature_base_df[
            "business_progress_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "판매 가능 재고 비율 범위 이탈:",
    (
        a1_feature_base_df[
            "available_stock_ratio"
        ].notna()
        & ~a1_feature_base_df[
            "available_stock_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "예약 재고 비율 범위 이탈:",
    (
        a1_feature_base_df[
            "reserved_stock_ratio"
        ].notna()
        & ~a1_feature_base_df[
            "reserved_stock_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "판매 불가능 재고 비율 범위 이탈:",
    (
        a1_feature_base_df[
            "unsaleable_stock_ratio"
        ].notna()
        & ~a1_feature_base_df[
            "unsaleable_stock_ratio"
        ].between(0, 1)
    ).sum()
)

print(
    "방문 데이터 보유 행 수:",
    (
        a1_feature_base_df[
            "visitor_data_available"
        ] == 1
    ).sum()
)

print(
    "재고 데이터 보유 행 수:",
    (
        a1_feature_base_df[
            "inventory_data_available"
        ] == 1
    ).sum()
)

print(
    "통합 기본 피처 테이블 크기:",
    a1_feature_base_df.shape
)

print("통합 기본 피처 생성 완료")

날짜·점포·상품·시간 중복 수: 0
series_id 결측 수: 0
영업시간 길이 오류 수: 0
영업 진행 비율 범위 이탈: 0
판매 가능 재고 비율 범위 이탈: 0
예약 재고 비율 범위 이탈: 0
판매 불가능 재고 비율 범위 이탈: 0
방문 데이터 보유 행 수: 492404
재고 데이터 보유 행 수: 492404
통합 기본 피처 테이블 크기: (492404, 93)
통합 기본 피처 생성 완료


In [215]:
# ============================================================
# 1-8-2. A-1 기본 피처 테이블 정렬
# ============================================================

# ------------------------------------------------------------
# 1. A-1 예측 단위 식별 컬럼을 앞으로 배치
# ------------------------------------------------------------

key_columns = [
    "series_id",
    "forecast_datetime",
    "date",
    "hour",
    "store_id",
    "product_id"
]

remaining_columns = [
    col
    for col in a1_feature_base_df.columns
    if col not in key_columns
]

a1_feature_base_df = a1_feature_base_df[
    key_columns + remaining_columns
].copy()

# ------------------------------------------------------------
# 2. 점포-상품별 시간순 정렬
# 이후 판매량 lag·rolling 피처 생성을 위한 순서
# ------------------------------------------------------------

a1_feature_base_df = (
    a1_feature_base_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. 결과 점검
# ------------------------------------------------------------

print(
    "A-1 판매 연결 전 기본 테이블 크기:",
    a1_feature_base_df.shape
)

print(
    "날짜·점포·상품·시간 중복 수:",
    a1_feature_base_df.duplicated(
        [
            "date",
            "store_id",
            "product_id",
            "hour"
        ]
    ).sum()
)

print(
    "series_id 불일치 행 수:",
    (
        a1_feature_base_df["series_id"]
        != (
            a1_feature_base_df["store_id"].astype(str)
            + "_"
            + a1_feature_base_df["product_id"].astype(str)
        )
    ).sum()
)

print(
    "forecast_datetime 결측 수:",
    a1_feature_base_df[
        "forecast_datetime"
    ].isna().sum()
)

display(
    a1_feature_base_df[
        [
            "series_id",
            "forecast_datetime",
            "store_id",
            "product_id",
            "category",
            "time_slot",
            "visitor_ratio",
            "visitor_count",
            "total_available_qty",
            "min_days_to_expiry",
            "near_expiry_qty",
            "markdown_window_2d_qty",
            "current_discount_rate"
        ]
    ].head(20)
)

A-1 판매 연결 전 기본 테이블 크기: (492404, 93)
날짜·점포·상품·시간 중복 수: 0
series_id 불일치 행 수: 0
forecast_datetime 결측 수: 0


,series_id,forecast_datetime,store_id,product_id,category,time_slot,visitor_ratio,visitor_count,total_available_qty,min_days_to_expiry,near_expiry_qty,markdown_window_2d_qty,current_discount_rate
0,S01_P001,2025-01-01 10:00:00,S01,P001,produce,morning,0.1800,51.0000,33,0.0000,5,5,0.0545
1,S01_P001,2025-01-01 11:00:00,S01,P001,produce,morning,0.1800,63.0000,33,0.0000,5,5,0.0545
2,S01_P001,2025-01-01 12:00:00,S01,P001,produce,lunch,0.2200,55.0000,33,0.0000,5,5,0.0545
3,S01_P001,2025-01-01 13:00:00,S01,P001,produce,lunch,0.2200,50.0000,33,0.0000,5,5,0.0545
4,S01_P001,2025-01-01 14:00:00,S01,P001,produce,lunch,0.2200,58.0000,33,0.0000,5,5,0.0545
5,S01_P001,2025-01-01 15:00:00,S01,P001,produce,afternoon,0.2500,46.0000,33,0.0000,5,5,0.0545
6,S01_P001,2025-01-01 16:00:00,S01,P001,produce,afternoon,0.2500,53.0000,33,0.0000,5,5,0.0545
7,S01_P001,2025-01-01 17:00:00,S01,P001,produce,afternoon,0.2500,42.0000,33,0.0000,5,5,0.0545
8,S01_P001,2025-01-01 18:00:00,S01,P001,produce,evening,0.2500,46.0000,33,0.0000,5,5,0.0545
9,S01_P001,2025-01-01 19:00:00,S01,P001,produce,evening,0.2500,58.0000,33,0.0000,5,5,0.0545


In [216]:
# ============================================================
# 1-8-3. A-1 예측 단위 확인
# ============================================================

# ------------------------------------------------------------
# 1. A-1 예측 단위 중복 확인
# ------------------------------------------------------------

a1_key_columns = [
    "store_id",
    "product_id",
    "date",
    "hour"
]

duplicate_key_count = (
    a1_feature_base_df
    .duplicated(a1_key_columns)
    .sum()
)

# forecast_datetime를 사용한 동일 의미의 예측 단위 확인
datetime_key_columns = [
    "store_id",
    "product_id",
    "forecast_datetime"
]

duplicate_datetime_key_count = (
    a1_feature_base_df
    .duplicated(datetime_key_columns)
    .sum()
)

# ------------------------------------------------------------
# 2. series_id 일치 여부 확인
# ------------------------------------------------------------

expected_series_id = (
    a1_feature_base_df["store_id"].astype(str)
    + "_"
    + a1_feature_base_df["product_id"].astype(str)
)

series_id_mismatch_count = (
    a1_feature_base_df["series_id"]
    != expected_series_id
).sum()

# ------------------------------------------------------------
# 3. 기본 규모 확인
# ------------------------------------------------------------

print(
    "전체 행 수:",
    len(a1_feature_base_df)
)

print(
    "시계열 수:",
    a1_feature_base_df["series_id"].nunique()
)

print(
    "예측 단위 중복 행 수:",
    duplicate_key_count
)

print(
    "forecast_datetime 기준 중복 행 수:",
    duplicate_datetime_key_count
)

print(
    "series_id 불일치 행 수:",
    series_id_mismatch_count
)

print(
    "날짜 범위:",
    a1_feature_base_df["date"].min(),
    "~",
    a1_feature_base_df["date"].max()
)

print(
    "시간 범위:",
    a1_feature_base_df["hour"].min(),
    "~",
    a1_feature_base_df["hour"].max()
)

# ------------------------------------------------------------
# 4. 데이터 보유 비율 확인
# ------------------------------------------------------------

print(
    "실제 방문객 데이터 보유 행 비율:",
    round(
        a1_feature_base_df[
            "visitor_data_available"
        ].mean(),
        4
    )
)

print(
    "재고 데이터 보유 행 비율:",
    round(
        a1_feature_base_df[
            "inventory_data_available"
        ].mean(),
        4
    )
)

# ------------------------------------------------------------
# 5. 데이터 보유 여부 컬럼 값 점검
# ------------------------------------------------------------

print(
    "visitor_data_available의 0/1 이외 값 수:",
    (
        ~a1_feature_base_df[
            "visitor_data_available"
        ].isin([0, 1])
    ).sum()
)

print(
    "inventory_data_available의 0/1 이외 값 수:",
    (
        ~a1_feature_base_df[
            "inventory_data_available"
        ].isin([0, 1])
    ).sum()
)

# ------------------------------------------------------------
# 6. 점포별·상품별 행 수 요약
# ------------------------------------------------------------

series_summary_df = (
    a1_feature_base_df
    .groupby(
        [
            "store_id",
            "product_id",
            "series_id"
        ],
        as_index=False
    )
    .agg(
        start_datetime=(
            "forecast_datetime",
            "min"
        ),
        end_datetime=(
            "forecast_datetime",
            "max"
        ),
        row_count=(
            "forecast_datetime",
            "size"
        ),
        date_count=(
            "date",
            "nunique"
        )
    )
)

print(
    "시계열 요약 행 수:",
    len(series_summary_df)
)

display(series_summary_df.head(10))

전체 행 수: 492404
시계열 수: 114
예측 단위 중복 행 수: 0
forecast_datetime 기준 중복 행 수: 0
series_id 불일치 행 수: 0
날짜 범위: 2025-01-01 00:00:00 ~ 2025-12-31 00:00:00
시간 범위: 10 ~ 22
실제 방문객 데이터 보유 행 비율: 1.0
재고 데이터 보유 행 비율: 1.0
visitor_data_available의 0/1 이외 값 수: 0
inventory_data_available의 0/1 이외 값 수: 0
시계열 요약 행 수: 114


,store_id,product_id,series_id,start_datetime,end_datetime,row_count,date_count
0,S01,P001,S01_P001,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
1,S01,P002,S01_P002,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
2,S01,P003,S01_P003,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
3,S01,P004,S01_P004,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
4,S01,P005,S01_P005,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
5,S01,P006,S01_P006,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
6,S01,P007,S01_P007,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
7,S01,P008,S01_P008,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
8,S01,P009,S01_P009,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341
9,S01,P010,S01_P010,2025-01-01 10:00:00,2025-12-31 22:00:00,4433,341


In [217]:
# # ============================================================
# # 1-8-4. 판매 연결 전 기본 피처 테이블 저장
# # Google Drive에 저장
# # ============================================================

# from pathlib import Path
# from google.colab import drive

# # Google Drive 연결
# drive.mount("/content/drive")

# # 저장 폴더
# OUTPUT_DIR = Path(
#     "/content/drive/MyDrive/fresh_food_demand/a1_outputs"
# )

# OUTPUT_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

# # 저장 파일 경로
# base_feature_path = (
#     OUTPUT_DIR
#     / "a1_feature_base_without_sales.parquet"
# )

# # Parquet 저장
# a1_feature_base_df.to_parquet(
#     base_feature_path,
#     index=False
# )

# print("저장 완료:", base_feature_path)
# print("저장 행 수:", len(a1_feature_base_df))
# print("저장 컬럼 수:", len(a1_feature_base_df.columns))

In [218]:
# # ============================================================
# # 저장된 A-1 기본 피처 테이블 불러오기
# # ============================================================

# import pandas as pd

# a1_feature_base_df = pd.read_parquet(
#     "/content/drive/MyDrive/fresh_food_demand/"
#     "a1_outputs/a1_feature_base_without_sales.parquet"
# )

# print(
#     "불러오기 완료:",
#     a1_feature_base_df.shape
# )

In [219]:
# ============================================================
# 2-1. 판매 데이터 시간별 집계
# ============================================================

# 판매 일시 변환
receipt_df["sale_datetime"] = pd.to_datetime(
    receipt_df["sale_datetime"],
    errors="coerce"
)

# 필수 키 또는 판매 일시가 없는 행 제외
receipt_df = receipt_df.dropna(
    subset=[
        "store_id",
        "product_id",
        "sale_datetime"
    ]
).copy()

# 1시간 단위 시각 생성
receipt_df["datetime"] = (
    receipt_df["sale_datetime"].dt.floor("h")
)

# 숫자형 컬럼 변환
numeric_cols = [
    "quantity",
    "unit_price",
    "sale_unit_price",
    "line_amount",
    "discount_rate"
]

receipt_df[numeric_cols] = receipt_df[numeric_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# 할인율 결측값은 무할인으로 처리
receipt_df["discount_rate"] = (
    receipt_df["discount_rate"].fillna(0)
)

# 할인율이 10, 20처럼 퍼센트 단위로 저장된 경우
# 0.1, 0.2 비율 단위로 변환
receipt_df["discount_rate"] = np.where(
    receipt_df["discount_rate"] > 1,
    receipt_df["discount_rate"] / 100,
    receipt_df["discount_rate"]
)

# 할인율 범위를 0~1로 제한
receipt_df["discount_rate"] = (
    receipt_df["discount_rate"]
    .clip(0, 1)
)

# 판매수량이 없는 행 제외
receipt_df = receipt_df.dropna(
    subset=[
        "quantity",
        "unit_price",
        "sale_unit_price",
        "line_amount"
    ]
).copy()

# 판매수량이 0 이하인 행 제외
# 현재 데이터 구조에서는 환불·취소 거래를 사용하지 않으므로 제외
receipt_df = receipt_df[
    receipt_df["quantity"] > 0
].copy()


# ------------------------------------------------------------
# 금액 및 판매유형 계산
# ------------------------------------------------------------

# 할인 전 상품 판매금액
receipt_df["gross_line_amount"] = (
    receipt_df["quantity"]
    * receipt_df["unit_price"]
)

# 상품 할인금액
# line_amount 대신 상품별 실제 판매단가를 기준으로 계산
receipt_df["line_discount_amount"] = (
    receipt_df["quantity"]
    * (
        receipt_df["unit_price"]
        - receipt_df["sale_unit_price"]
    )
)

# 정가 판매량
receipt_df["regular_sales_qty"] = np.where(
    receipt_df["discount_rate"] <= 0,
    receipt_df["quantity"],
    0
)

# 할인 판매량
receipt_df["discount_sales_qty"] = np.where(
    receipt_df["discount_rate"] > 0,
    receipt_df["quantity"],
    0
)


# ------------------------------------------------------------
# 판매수량 가중평균 계산용 컬럼
# ------------------------------------------------------------

receipt_df["discount_rate_x_qty"] = (
    receipt_df["discount_rate"]
    * receipt_df["quantity"]
)

receipt_df["unit_price_x_qty"] = (
    receipt_df["unit_price"]
    * receipt_df["quantity"]
)

receipt_df["sale_unit_price_x_qty"] = (
    receipt_df["sale_unit_price"]
    * receipt_df["quantity"]
)


# ------------------------------------------------------------
# 점포 × 상품 × 시간 단위 집계
# ------------------------------------------------------------

hourly_sales_df = (
    receipt_df
    .groupby(
        [
            "store_id",
            "product_id",
            "datetime"
        ],
        as_index=False
    )
    .agg(
        sales_qty=("quantity", "sum"),

        regular_sales_qty=(
            "regular_sales_qty",
            "sum"
        ),

        discount_sales_qty=(
            "discount_sales_qty",
            "sum"
        ),

        gross_sales_amount=(
            "gross_line_amount",
            "sum"
        ),

        sales_amount=(
            "line_amount",
            "sum"
        ),

        discount_amount=(
            "line_discount_amount",
            "sum"
        ),

        discount_rate_x_qty=(
            "discount_rate_x_qty",
            "sum"
        ),

        unit_price_x_qty=(
            "unit_price_x_qty",
            "sum"
        ),

        sale_unit_price_x_qty=(
            "sale_unit_price_x_qty",
            "sum"
        ),

        receipt_count=(
            "receipt_id",
            "nunique"
        )
    )
)


# ------------------------------------------------------------
# 가중평균 계산
# ------------------------------------------------------------

# 0으로 나누는 것을 방지하기 위한 분모
sales_qty_denominator = (
    hourly_sales_df["sales_qty"]
    .replace(0, np.nan)
)

# 판매수량 기준 가중평균 할인율
hourly_sales_df["weighted_discount_rate"] = (
    hourly_sales_df["discount_rate_x_qty"]
    / sales_qty_denominator
)

# 판매수량 기준 평균 정가
hourly_sales_df["avg_unit_price"] = (
    hourly_sales_df["unit_price_x_qty"]
    / sales_qty_denominator
)

# 판매수량 기준 평균 실제 판매단가
hourly_sales_df["avg_sale_unit_price"] = (
    hourly_sales_df["sale_unit_price_x_qty"]
    / sales_qty_denominator
)


# ------------------------------------------------------------
# A-1 기본 피처 테이블 연결용 키 생성
# ------------------------------------------------------------

hourly_sales_df["date"] = (
    hourly_sales_df["datetime"].dt.normalize()
)

hourly_sales_df["hour"] = (
    hourly_sales_df["datetime"].dt.hour
)


# 계산용 임시 컬럼 제거
hourly_sales_df = hourly_sales_df.drop(
    columns=[
        "discount_rate_x_qty",
        "unit_price_x_qty",
        "sale_unit_price_x_qty"
    ]
)

# 시계열 순서 정렬
hourly_sales_df = (
    hourly_sales_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "date",
            "hour"
        ]
    )
    .reset_index(drop=True)
)

display(hourly_sales_df.head())

print(
    "시간별 판매 집계 행 수:",
    len(hourly_sales_df)
)

,store_id,product_id,datetime,sales_qty,regular_sales_qty,discount_sales_qty,gross_sales_amount,sales_amount,discount_amount,receipt_count,weighted_discount_rate,avg_unit_price,avg_sale_unit_price,date,hour
0,S01,P001,2025-01-01 11:00:00,4,0,4,26000,16640,9360,2,0.3600,"6,500.0000","4,160.0000",2025-01-01,11
1,S01,P001,2025-01-01 19:00:00,2,0,2,13000,8320,4680,1,0.3600,"6,500.0000","4,160.0000",2025-01-01,19
2,S01,P001,2025-01-01 22:00:00,1,0,1,6500,4160,2340,1,0.3600,"6,500.0000","4,160.0000",2025-01-01,22
3,S01,P001,2025-01-02 10:00:00,1,1,0,6500,6500,0,1,0.0000,"6,500.0000","6,500.0000",2025-01-02,10
4,S01,P001,2025-01-02 11:00:00,1,1,0,6500,6500,0,1,0.0000,"6,500.0000","6,500.0000",2025-01-02,11


시간별 판매 집계 행 수: 101973


In [220]:
# ============================================================
# 2-2. 영업시간 무판매 행 sales_qty=0 처리
# ============================================================

# 학습 테이블과 판매 테이블 시간 형식 통일
a1_feature_base_df["forecast_datetime"] = pd.to_datetime(
    a1_feature_base_df["forecast_datetime"]
).dt.floor("h")

# 병합에 필요한 컬럼만 선택
# date, hour는 a1_feature_base_df에 이미 있으므로 제외
hourly_sales_merge_df = (
    hourly_sales_df[
        [
            "store_id",
            "product_id",
            "datetime",
            "sales_qty",
            "regular_sales_qty",
            "discount_sales_qty",
            "gross_sales_amount",
            "sales_amount",
            "discount_amount",
            "receipt_count",
            "weighted_discount_rate",
            "avg_unit_price",
            "avg_sale_unit_price"
        ]
    ]
    .rename(
        columns={
            "datetime": "forecast_datetime"
        }
    )
    .copy()
)

hourly_sales_merge_df["forecast_datetime"] = pd.to_datetime(
    hourly_sales_merge_df["forecast_datetime"]
).dt.floor("h")


# ------------------------------------------------------------
# 1-8 통합 테이블에 시간별 판매 데이터 연결
# ------------------------------------------------------------

a1_training_df = a1_feature_base_df.merge(
    hourly_sales_merge_df,
    on=[
        "store_id",
        "product_id",
        "forecast_datetime"
    ],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 판매 기록이 없는 영업시간 처리
# ------------------------------------------------------------

# 무판매 시간은 수량·매출·할인 관련 값을 0으로 처리
zero_fill_cols = [
    "sales_qty",
    "regular_sales_qty",
    "discount_sales_qty",
    "gross_sales_amount",
    "sales_amount",
    "discount_amount",
    "receipt_count",
    "weighted_discount_rate"
]

a1_training_df[zero_fill_cols] = (
    a1_training_df[zero_fill_cols]
    .fillna(0)
)

# 영수증 수만 정수형으로 변환
# 판매수량은 중량 상품의 소수 수량을 보존
a1_training_df["receipt_count"] = (
    a1_training_df["receipt_count"]
    .astype(int)
)


# ------------------------------------------------------------
# 시계열 순서 정렬
# ------------------------------------------------------------

a1_training_df = (
    a1_training_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)

display(a1_training_df.head())

print("판매 연결 전 행 수:", len(a1_feature_base_df))
print("판매 연결 후 행 수:", len(a1_training_df))
print(
    "판매 발생 행 수:",
    (a1_training_df["sales_qty"] > 0).sum()
)
print(
    "무판매 행 수:",
    (a1_training_df["sales_qty"] == 0).sum()
)

,series_id,forecast_datetime,date,hour,store_id,product_id,open_hour,close_hour,hours_until_close,year,month,day,day_of_week,week_of_month,day_type,is_weekend,is_holiday,holiday_name,season,season_index,is_event_day,event_type,event_index,store_name,area_type,resident_pop,floating_idx,order_error_sigma,product_name,category,subcategory,sales_type,unit,standard_weight_kg,shelf_life_days,base_cost,base_price,margin_rate,freshness_decay_type,baseline_waste_rate,hour_num,month_num,day_of_week_num,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,time_slot,visitor_ratio,visitor_count,lag_1h_visitor_count,rolling_3h_visitor_mean,cumulative_daily_visitors,visitor_data_available,total_current_stock_qty,total_physical_available_qty,total_available_qty,total_reserved_qty,excluded_unsaleable_qty,lot_count,saleable_lot_count,sold_out_lot_count,any_sold_out_lot_flag,all_lots_sold_out_flag,full_product_sold_out_flag,partial_lot_sold_out_flag,min_days_to_expiry,mean_days_to_expiry,weighted_days_to_expiry,mean_freshness_score,weighted_freshness_score,expiry_today_qty,near_expiry_qty,near_expiry_ratio,markdown_window_2d_qty,markdown_window_2d_ratio,expired_qty,expired_qty_ratio,current_discount_rate,hours_since_open,business_hours,business_progress_ratio,inventory_data_available,available_stock_ratio,reserved_stock_ratio,unsaleable_stock_ratio,has_available_stock,has_near_expiry_stock,has_markdown_window_2d_stock,has_expired_stock,has_current_discount,sales_qty,regular_sales_qty,discount_sales_qty,gross_sales_amount,sales_amount,discount_amount,receipt_count,weighted_discount_rate,avg_unit_price,avg_sale_unit_price
0,S01_P001,2025-01-01 10:00:00,2025-01-01,10,S01,P001,10,23,13,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560,10,1,2,0.5000,-0.8660,0.9749,-0.2225,0.0000,1.0000,morning,0.1800,51.0000,NaN,NaN,0.0000,1,33,33,33,0,0,3,3,0,0,0,0,0,0.0000,4.3333,5.6364,0.4849,0.6033,5,5,0.1515,5,0.1515,0,0.0000,0.0545,0,13,0.0000,1,1.0000,0.0000,0.0000,1,1,1,0,1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,NaN,NaN
1,S01_P001,2025-01-01 11:00:00,2025-01-01,11,S01,P001,10,23,12,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560,11,1,2,0.2588,-0.9659,0.9749,-0.2225,0.0000,1.0000,morning,0.1800,63.0000,51.0000,51.0000,51.0000,1,33,33,33,0,0,3,3,0,0,0,0,0,0.0000,4.3333,5.6364,0.4849,0.6033,5,5,0.1515,5,0.1515,0,0.0000,0.0545,1,13,0.0769,1,1.0000,0.0000,0.0000,1,1,1,0,1,4.0000,0.0000,4.0000,"26,000.0000","16,640.0000","9,360.0000",2,0.3600,"6,500.0000","4,160.0000"
2,S01_P001,2025-01-01 12:00:00,2025-01-01,12,S01,P001,10,23,11,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560,12,1,2,0.0000,-1.0000,0.9749,-0.2225,0.0000,1.0000,lunch,0.2200,55.0000,63.0000,57.0000,114.0000,1,33,33,33,0,0,3,3,0,0,0,0,0,0.0000,4.3333,5.6364,0.4849,0.6033,5,5,0.1515,5,0.1515,0,0.0000,0.0545,2,13,0.1538,1,1.0000,0.0000,0.0000,1,1,1,0,1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,NaN,NaN
3,S01_P001,2025-01-01 13:00:00,2025-01-01,13,S01,P001,10,23,10,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price,bag,1.5000,10,5200,6500,0.2000,medium,0.0560,13,1,2,-0.2588,-0.9659,0.9749,-0.2225,0.0000,1.0000,lunch,0.2200,50.0000,55.0000,56.3333,169.0000,1,33,33,33,0,0,3,3,0,0,0,0,0,0.0000,4.3333,5.6364,0.4849,0.6033,5,5,0.1515,5,0.1515,0,0.0000,0.0545,3,13,0.2308,1,1.0000,0.0000,0.0000,1,1,1,0,1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,NaN,NaN
4,S01_P001,2025-01-01 14:00:00,2025-01-01,14,S01,P001,10,23,9,2025,1,1,WED,1,holiday,0,1,신정,winter,1.0000,0,NaN,1.0000,가상주거형점,residential,25000,0.8500,0.1200,사과,produce,apple,fixed_price

판매 연결 전 행 수: 492404
판매 연결 후 행 수: 492404
판매 발생 행 수: 101973
무판매 행 수: 390431


In [221]:
# ============================================================
# 2-3. 할인 거래 및 정가 기준 처리
# ============================================================

# ------------------------------------------------------------
# 1. 판매 발생 여부
# ------------------------------------------------------------

a1_training_df["has_sales"] = (
    a1_training_df["sales_qty"] > 0
).astype("int8")


# ------------------------------------------------------------
# 2. 할인율 형식 통일
# ------------------------------------------------------------

discount_rate_raw = pd.to_numeric(
    a1_training_df["weighted_discount_rate"],
    errors="coerce"
)

# 할인율이 10, 20처럼 저장된 경우
# 0.1, 0.2 비율 단위로 변환
discount_rate_normalized = np.where(
    discount_rate_raw > 1,
    discount_rate_raw / 100,
    discount_rate_raw
)

discount_rate_normalized = pd.Series(
    discount_rate_normalized,
    index=a1_training_df.index
).clip(0, 1)

# 판매가 발생한 행에서만 실제 할인율이 관측된 것으로 처리
# 무판매 행은 영수증만으로 할인 여부를 알 수 없으므로 NaN 처리
a1_training_df["discount_rate_model"] = (
    discount_rate_normalized.where(
        a1_training_df["has_sales"] == 1,
        np.nan
    )
)

# 부동소수점 계산 오차 대응
discount_threshold = 1e-8


# ------------------------------------------------------------
# 3. 정가·할인·혼합 판매 조건 구분
# ------------------------------------------------------------

# 실제 정가 판매만 발생한 시간
a1_training_df["is_regular_condition"] = (
    (a1_training_df["sales_qty"] > 0)
    & (a1_training_df["regular_sales_qty"] > 0)
    & (a1_training_df["discount_sales_qty"] == 0)
).astype("int8")

# 할인 판매가 하나라도 발생한 시간
# 정가와 할인이 섞인 시간도 포함
a1_training_df["is_discounted"] = (
    (a1_training_df["sales_qty"] > 0)
    & (a1_training_df["discount_sales_qty"] > 0)
).astype("int8")

# 같은 시간대에 정가 판매와 할인 판매가 모두 발생한 경우
a1_training_df["is_mixed_price_condition"] = (
    (a1_training_df["regular_sales_qty"] > 0)
    & (a1_training_df["discount_sales_qty"] > 0)
).astype("int8")

# 판매가 없어 할인 조건을 확인할 수 없는 시간
a1_training_df["is_discount_condition_unknown"] = (
    a1_training_df["sales_qty"] == 0
).astype("int8")


# ------------------------------------------------------------
# 4. 전체 판매량 중 할인 판매량 비중
# ------------------------------------------------------------

sales_qty_denominator = (
    pd.to_numeric(
        a1_training_df["sales_qty"],
        errors="coerce"
    )
    .replace(0, np.nan)
)

a1_training_df["discount_sales_share"] = (
    pd.to_numeric(
        a1_training_df["discount_sales_qty"],
        errors="coerce"
    )
    / sales_qty_denominator
)

a1_training_df["discount_sales_share"] = (
    a1_training_df["discount_sales_share"]
    .fillna(0)
    .clip(0, 1)
)

# discount_sales_share는 판매 결과로 계산된 값이므로
# 분석·검사용으로만 사용하고 모델 입력 피처에서는 제외


# ------------------------------------------------------------
# 5. 모델용 가격 변수
# ------------------------------------------------------------

# 상품 정상가
a1_training_df["regular_price_model"] = pd.to_numeric(
    a1_training_df["base_price"],
    errors="coerce"
)

# 0원 이하의 비정상 가격은 결측값 처리
a1_training_df["regular_price_model"] = (
    a1_training_df["regular_price_model"]
    .where(
        a1_training_df["regular_price_model"] > 0,
        np.nan
    )
)

# 정상가 대비 판매가격 비율
# 정가 판매는 1, 10% 할인은 0.9
# 무판매 행은 할인 조건을 알 수 없으므로 NaN
a1_training_df["price_ratio"] = (
    1 - a1_training_df["discount_rate_model"]
)

# 할인 조건을 반영한 모델용 판매가격
a1_training_df["sale_price_model"] = (
    a1_training_df["regular_price_model"]
    * a1_training_df["price_ratio"]
)


# ------------------------------------------------------------
# 6. A-1 기본수요 모델 목표값
# ------------------------------------------------------------

# 할인·정가를 포함한 실제 전체 판매량을 목표값으로 사용
# 무판매 시간도 수요 0으로 포함
a1_training_df["demand_target"] = pd.to_numeric(
    a1_training_df["sales_qty"],
    errors="coerce"
).fillna(0)


# ------------------------------------------------------------
# 7. 시계열 순서 정렬
# ------------------------------------------------------------

a1_training_df = (
    a1_training_df
    .sort_values(
        [
            "store_id",
            "product_id",
            "forecast_datetime"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. 결과 확인
# ------------------------------------------------------------

display(
    a1_training_df[
        [
            "store_id",
            "product_id",
            "forecast_datetime",
            "sales_qty",
            "regular_sales_qty",
            "discount_sales_qty",
            "discount_rate_model",
            "discount_sales_share",
            "regular_price_model",
            "sale_price_model",
            "price_ratio",
            "has_sales",
            "is_regular_condition",
            "is_discounted",
            "is_mixed_price_condition",
            "is_discount_condition_unknown",
            "demand_target"
        ]
    ].head()
)

print(
    "실제 정가 판매 행 수:",
    a1_training_df["is_regular_condition"].sum()
)

print(
    "할인 판매 포함 행 수:",
    a1_training_df["is_discounted"].sum()
)

print(
    "정가·할인 혼합 판매 행 수:",
    a1_training_df["is_mixed_price_condition"].sum()
)

print(
    "무판매로 할인 여부를 확인할 수 없는 행 수:",
    a1_training_df[
        "is_discount_condition_unknown"
    ].sum()
)

print(
    "전체 판매 발생 행 수:",
    a1_training_df["has_sales"].sum()
)

print(
    "전체 무판매 행 수:",
    (a1_training_df["sales_qty"] == 0).sum()
)

,store_id,product_id,forecast_datetime,sales_qty,regular_sales_qty,discount_sales_qty,discount_rate_model,discount_sales_share,regular_price_model,sale_price_model,price_ratio,has_sales,is_regular_condition,is_discounted,is_mixed_price_condition,is_discount_condition_unknown,demand_target
0,S01,P001,2025-01-01 10:00:00,0.0000,0.0000,0.0000,NaN,0.0000,6500,NaN,NaN,0,0,0,0,1,0.0000
1,S01,P001,2025-01-01 11:00:00,4.0000,0.0000,4.0000,0.3600,1.0000,6500,"4,160.0000",0.6400,1,0,1,0,0,4.0000
2,S01,P001,2025-01-01 12:00:00,0.0000,0.0000,0.0000,NaN,0.0000,6500,NaN,NaN,0,0,0,0,1,0.0000
3,S01,P001,2025-01-01 13:00:00,0.0000,0.0000,0.0000,NaN,0.0000,6500,NaN,NaN,0,0,0,0,1,0.0000
4,S01,P001,2025-01-01 14:00:00,0.0000,0.0000,0.0000,NaN,0.0000,6500,NaN,NaN,0,0,0,0,1,0.0000


실제 정가 판매 행 수: 70268
할인 판매 포함 행 수: 31705
정가·할인 혼합 판매 행 수: 1239
무판매로 할인 여부를 확인할 수 없는 행 수: 390431
전체 판매 발생 행 수: 101973
전체 무판매 행 수: 390431


In [222]:
# ============================================================
# 2-4. 판매량 lag·rolling 피처 생성
# ============================================================

# ------------------------------------------------------------
# 1. 기본 설정
# ------------------------------------------------------------

series_key_cols = [
    "store_id",
    "product_id"
]

datetime_col = "forecast_datetime"
target_col = "demand_target"

# 재실행 시 기존 lag·rolling 컬럼 제거
existing_sales_history_cols = [
    col
    for col in a1_training_df.columns
    if (
        col.startswith("sales_qty_lag_")
        or col.startswith("sales_qty_roll_")
    )
]

a1_training_df = a1_training_df.drop(
    columns=existing_sales_history_cols,
    errors="ignore"
)

# 시간 형식 통일
a1_training_df[datetime_col] = pd.to_datetime(
    a1_training_df[datetime_col]
)

# 목표값 숫자형 통일
a1_training_df[target_col] = pd.to_numeric(
    a1_training_df[target_col],
    errors="coerce"
).fillna(0)

# 시계열 순서 정렬
a1_training_df = (
    a1_training_df
    .sort_values(
        series_key_cols + [datetime_col]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. 실제 시간 기준 lag 피처 생성
# ------------------------------------------------------------

# 1·2·3시간 전:
# 단기 판매 흐름
#
# 24·48시간 전:
# 전날 및 이틀 전 같은 시간
#
# 168시간 전:
# 전주 같은 요일·시간
lag_hours = [
    1,
    2,
    3,
    24,
    48,
    168
]

# lag 계산용 원본 테이블
lag_source_df = (
    a1_training_df[
        series_key_cols
        + [
            datetime_col,
            target_col
        ]
    ]
    .copy()
)

for lag_hour in lag_hours:

    lag_col = f"sales_qty_lag_{lag_hour}h"

    lag_lookup_df = lag_source_df.rename(
        columns={
            target_col: lag_col
        }
    ).copy()

    # 과거 시각의 판매량을 미래 시각에 연결
    # 예: 13시 판매량을 14시 행에 연결하면 lag_1h
    lag_lookup_df[datetime_col] = (
        lag_lookup_df[datetime_col]
        + pd.Timedelta(hours=lag_hour)
    )

    a1_training_df = a1_training_df.merge(
        lag_lookup_df,
        on=series_key_cols + [datetime_col],
        how="left",
        validate="one_to_one"
    )


# ------------------------------------------------------------
# 3. 실제 시간 기준 rolling 피처 생성
# ------------------------------------------------------------

# 현재 시점은 제외하고 과거 데이터만 사용
rolling_windows = {
    "3h": "3h",
    "6h": "6h",
    "12h": "12h",
    "24h": "24h",
    "168h": "168h"
}

rolling_source_df = (
    a1_training_df[
        series_key_cols
        + [
            datetime_col,
            target_col
        ]
    ]
    .sort_values(
        series_key_cols + [datetime_col]
    )
    .set_index(datetime_col)
)

rolling_feature_frames = []

for window_name, window_size in rolling_windows.items():

    rolling_stats_df = (
        rolling_source_df
        .groupby(
            series_key_cols,
            observed=True
        )[target_col]
        .rolling(
            window=window_size,
            min_periods=1,
            closed="left"
        )
        .agg(
            [
                "mean",
                "std",
                "sum",
                "max"
            ]
        )
        .rename(
            columns={
                "mean": (
                    f"sales_qty_roll_mean_{window_name}"
                ),
                "std": (
                    f"sales_qty_roll_std_{window_name}"
                ),
                "sum": (
                    f"sales_qty_roll_sum_{window_name}"
                ),
                "max": (
                    f"sales_qty_roll_max_{window_name}"
                )
            }
        )
    )

    rolling_feature_frames.append(
        rolling_stats_df
    )

# 각 rolling 구간의 결과를 하나로 결합
rolling_feature_df = (
    pd.concat(
        rolling_feature_frames,
        axis=1
    )
    .reset_index()
)

# 기존 학습 테이블에 rolling 피처 연결
a1_training_df = a1_training_df.merge(
    rolling_feature_df,
    on=series_key_cols + [datetime_col],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 4. lag·rolling 피처의 결측 여부 표시
# ------------------------------------------------------------

# 초기 시점처럼 과거 데이터가 없어 lag가 없는 행 표시
for lag_hour in lag_hours:

    lag_col = f"sales_qty_lag_{lag_hour}h"

    a1_training_df[
        f"{lag_col}_missing"
    ] = (
        a1_training_df[lag_col]
        .isna()
        .astype("int8")
    )


# ------------------------------------------------------------
# 5. 최종 정렬
# ------------------------------------------------------------

a1_training_df = (
    a1_training_df
    .sort_values(
        series_key_cols + [datetime_col]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. 결과 확인
# ------------------------------------------------------------

lag_feature_cols = [
    f"sales_qty_lag_{lag_hour}h"
    for lag_hour in lag_hours
]

rolling_feature_cols = [
    col
    for col in a1_training_df.columns
    if col.startswith("sales_qty_roll_")
]

display(
    a1_training_df[
        [
            "store_id",
            "product_id",
            "forecast_datetime",
            "demand_target"
        ]
        + lag_feature_cols
        + [
            "sales_qty_roll_mean_3h",
            "sales_qty_roll_mean_24h",
            "sales_qty_roll_mean_168h"
        ]
    ].head(20)
)

print(
    "생성된 lag 피처 수:",
    len(lag_feature_cols)
)

print(
    "생성된 rolling 피처 수:",
    len(rolling_feature_cols)
)

print(
    "lag·rolling 생성 후 전체 행 수:",
    len(a1_training_df)
)

print(
    "lag_24h 결측 행 수:",
    a1_training_df[
        "sales_qty_lag_24h"
    ].isna().sum()
)

print(
    "lag_168h 결측 행 수:",
    a1_training_df[
        "sales_qty_lag_168h"
    ].isna().sum()
)

,store_id,product_id,forecast_datetime,demand_target,sales_qty_lag_1h,sales_qty_lag_2h,sales_qty_lag_3h,sales_qty_lag_24h,sales_qty_lag_48h,sales_qty_lag_168h,sales_qty_roll_mean_3h,sales_qty_roll_mean_24h,sales_qty_roll_mean_168h
0,S01,P001,2025-01-01 10:00:00,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S01,P001,2025-01-01 11:00:00,4.0000,0.0000,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000
2,S01,P001,2025-01-01 12:00:00,0.0000,4.0000,0.0000,NaN,NaN,NaN,NaN,2.0000,2.0000,2.0000
3,S01,P001,2025-01-01 13:00:00,0.0000,0.0000,4.0000,0.0000,NaN,NaN,NaN,1.3333,1.3333,1.3333
4,S01,P001,2025-01-01 14:00:00,0.0000,0.0000,0.0000,4.0000,NaN,NaN,NaN,1.3333,1.0000,1.0000
5,S01,P001,2025-01-01 15:00:00,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,0.8000,0.8000
6,S01,P001,2025-01-01 16:00:00,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,0.6667,0.6667
7,S01,P001,2025-01-01 17:00:00,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,0.5714,0.5714
8,S01,P001,2025-01-01 18:00:00,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,0.5000,0.5000
9,S01,P001,2025-01-01 19:00:00,2.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,0.4444,0.4444


생성된 lag 피처 수: 6
생성된 rolling 피처 수: 20
lag·rolling 생성 후 전체 행 수: 492404
lag_24h 결측 행 수: 36100
lag_168h 결측 행 수: 43776


In [223]:
# ============================================================
# receipt 판매 데이터 기간 확인
# ============================================================

receipt_period_check_df = receipt_df.copy()

receipt_period_check_df["sale_datetime"] = pd.to_datetime(
    receipt_period_check_df["sale_datetime"],
    errors="coerce"
)

receipt_period_check_df = receipt_period_check_df.dropna(
    subset=["sale_datetime"]
)

print(
    "판매 데이터 시작:",
    receipt_period_check_df["sale_datetime"].min()
)

print(
    "판매 데이터 종료:",
    receipt_period_check_df["sale_datetime"].max()
)

print(
    "판매 데이터 날짜 수:",
    receipt_period_check_df["sale_datetime"]
    .dt.normalize()
    .nunique()
)

print(
    "판매 데이터 월 수:",
    receipt_period_check_df["sale_datetime"]
    .dt.to_period("M")
    .nunique()
)

판매 데이터 시작: 2025-01-01 10:01:31
판매 데이터 종료: 2025-12-31 22:59:56
판매 데이터 날짜 수: 365
판매 데이터 월 수: 12


In [224]:
# ============================================================
# 2-5. 시간순 Train·Validation·Test 분할
# Train(~9/30) / Valid(~11/15) / Test(~12/31) 지금은1년치라 이렇게
# ============================================================

# ------------------------------------------------------------
# 1. 기본 설정
# ------------------------------------------------------------

datetime_col = "forecast_datetime"
target_col = "demand_target"

# 시간 형식 통일
a1_training_df[datetime_col] = pd.to_datetime(
    a1_training_df[datetime_col],
    errors="coerce"
)

# 시간값이 없는 행 제외
a1_training_df = (
    a1_training_df
    .dropna(subset=[datetime_col])
    .copy()
)

# 날짜 단위 분할용 컬럼
a1_training_df["split_date"] = (
    a1_training_df[datetime_col]
    .dt.normalize()
)

# 분할 기준일
validation_start_date = pd.Timestamp("2025-10-01")
test_start_date = pd.Timestamp("2025-11-16")


# ------------------------------------------------------------
# 2. 시간순 데이터 분할
# ------------------------------------------------------------

# 2025-09-30까지
train_df = (
    a1_training_df[
        a1_training_df["split_date"]
        < validation_start_date
    ]
    .copy()
)

# 2025-10-01 ~ 2025-11-15
validation_df = (
    a1_training_df[
        (
            a1_training_df["split_date"]
            >= validation_start_date
        )
        & (
            a1_training_df["split_date"]
            < test_start_date
        )
    ]
    .copy()
)

# 2025-11-16 이후
test_df = (
    a1_training_df[
        a1_training_df["split_date"]
        >= test_start_date
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. 분할 구분 컬럼 추가
# ------------------------------------------------------------

train_df["dataset_split"] = "train"
validation_df["dataset_split"] = "validation"
test_df["dataset_split"] = "test"

a1_training_df["dataset_split"] = np.select(
    [
        a1_training_df["split_date"]
        < validation_start_date,

        (
            a1_training_df["split_date"]
            >= validation_start_date
        )
        & (
            a1_training_df["split_date"]
            < test_start_date
        ),

        a1_training_df["split_date"]
        >= test_start_date
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="unknown"
)


# ------------------------------------------------------------
# 4. 시계열 순서 정렬
# ------------------------------------------------------------

sort_cols = [
    "store_id",
    "product_id",
    datetime_col
]

train_df = (
    train_df
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

validation_df = (
    validation_df
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

test_df = (
    test_df
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

a1_training_df = (
    a1_training_df
    .sort_values(sort_cols)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. 분할 결과 확인
# ------------------------------------------------------------

def print_split_summary(
    split_name,
    split_df
):
    print(f"\n[{split_name}]")
    print("행 수:", len(split_df))
    print(
        "날짜 수:",
        split_df["split_date"].nunique()
    )
    print(
        "시계열 수:",
        split_df["series_id"].nunique()
    )
    print(
        "기간:",
        split_df[datetime_col].min(),
        "~",
        split_df[datetime_col].max()
    )
    print(
        "판매 발생 행 수:",
        (split_df[target_col] > 0).sum()
    )
    print(
        "무판매 행 비율:",
        round(
            (split_df[target_col] == 0).mean(),
            4
        )
    )


print_split_summary(
    "Train",
    train_df
)

print_split_summary(
    "Validation",
    validation_df
)

print_split_summary(
    "Test",
    test_df
)


# ------------------------------------------------------------
# 6. 시간순 분할 검증
# ------------------------------------------------------------

print(
    "\nTrain 종료 < Validation 시작:",
    train_df[datetime_col].max()
    < validation_df[datetime_col].min()
)

print(
    "Validation 종료 < Test 시작:",
    validation_df[datetime_col].max()
    < test_df[datetime_col].min()
)

print(
    "전체 행 수 일치:",
    len(a1_training_df)
    == (
        len(train_df)
        + len(validation_df)
        + len(test_df)
    )
)


[Train]
행 수: 368220
날짜 수: 273
시계열 수: 114
기간: 2025-01-01 10:00:00 ~ 2025-09-30 22:00:00
판매 발생 행 수: 76401
무판매 행 비율: 0.7925

[Validation]
행 수: 62092
날짜 수: 46
시계열 수: 114
기간: 2025-10-01 10:00:00 ~ 2025-11-15 22:00:00
판매 발생 행 수: 12752
무판매 행 비율: 0.7946

[Test]
행 수: 62092
날짜 수: 46
시계열 수: 114
기간: 2025-11-16 10:00:00 ~ 2025-12-31 22:00:00
판매 발생 행 수: 12820
무판매 행 비율: 0.7935

Train 종료 < Validation 시작: True
Validation 종료 < Test 시작: True
전체 행 수 일치: True


In [225]:
# # ============================================================
# # 2-6. Train·Validation·Test 시간대별 수요 분포 점검
# # 특히 10~11시, 15~17시 확인
# # ============================================================

# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# # ------------------------------------------------------------
# # 1. 기본 설정
# # ------------------------------------------------------------

# target_col = "demand_target"
# hour_col = "hour"

# # 2-5에서 만든 데이터프레임 이름에 맞게 연결
# split_df_dict = {
#     "Train": train_df,
#     "Validation": validation_df,
#     "Test": test_df
# }


# # ------------------------------------------------------------
# # 2. 시간 컬럼 숫자형 통일
# # ------------------------------------------------------------

# for split_name, split_df in split_df_dict.items():

#     split_df[hour_col] = pd.to_numeric(
#         split_df[hour_col],
#         errors="coerce"
#     )

#     split_df[target_col] = pd.to_numeric(
#         split_df[target_col],
#         errors="coerce"
#     )


# # ------------------------------------------------------------
# # 3. 데이터셋별 기본 정보 확인
# # ------------------------------------------------------------

# print("=" * 70)
# print("Train·Validation·Test 기본 정보")
# print("=" * 70)

# basic_summary_list = []

# for split_name, split_df in split_df_dict.items():

#     basic_summary_list.append({
#         "dataset": split_name,
#         "row_count": len(split_df),
#         "min_hour": split_df[hour_col].min(),
#         "max_hour": split_df[hour_col].max(),
#         "target_missing_count": split_df[target_col].isna().sum(),
#         "target_zero_ratio": (split_df[target_col] == 0).mean(),
#         "target_positive_ratio": (split_df[target_col] > 0).mean(),
#         "target_mean": split_df[target_col].mean(),
#         "target_sum": split_df[target_col].sum()
#     })

# basic_summary_df = pd.DataFrame(basic_summary_list)

# display(
#     basic_summary_df.style.format({
#         "target_zero_ratio": "{:.2%}",
#         "target_positive_ratio": "{:.2%}",
#         "target_mean": "{:.4f}",
#         "target_sum": "{:,.2f}"
#     })
# )


# # ------------------------------------------------------------
# # 4. 데이터셋별 시간대 수요 통계 생성
# # ------------------------------------------------------------

# hourly_summary_list = []

# for split_name, split_df in split_df_dict.items():

#     hourly_summary = (
#         split_df
#         .groupby(hour_col, observed=True)
#         .agg(
#             row_count=(target_col, "size"),
#             demand_sum=(target_col, "sum"),
#             demand_mean=(target_col, "mean"),
#             demand_median=(target_col, "median"),
#             demand_std=(target_col, "std"),
#             demand_max=(target_col, "max"),
#             zero_count=(target_col, lambda x: (x == 0).sum()),
#             positive_count=(target_col, lambda x: (x > 0).sum())
#         )
#         .reset_index()
#     )

#     hourly_summary["zero_ratio"] = (
#         hourly_summary["zero_count"]
#         / hourly_summary["row_count"]
#     )

#     hourly_summary["positive_ratio"] = (
#         hourly_summary["positive_count"]
#         / hourly_summary["row_count"]
#     )

#     hourly_summary["dataset"] = split_name

#     hourly_summary_list.append(hourly_summary)


# hourly_demand_summary_df = pd.concat(
#     hourly_summary_list,
#     ignore_index=True
# )

# hourly_demand_summary_df = hourly_demand_summary_df[
#     [
#         "dataset",
#         hour_col,
#         "row_count",
#         "demand_sum",
#         "demand_mean",
#         "demand_median",
#         "demand_std",
#         "demand_max",
#         "zero_count",
#         "zero_ratio",
#         "positive_count",
#         "positive_ratio"
#     ]
# ].sort_values(
#     ["dataset", hour_col]
# ).reset_index(drop=True)


# print("\n")
# print("=" * 70)
# print("전체 시간대별 수요 분포")
# print("=" * 70)

# display(
#     hourly_demand_summary_df.style.format({
#         "demand_sum": "{:,.2f}",
#         "demand_mean": "{:.4f}",
#         "demand_median": "{:.4f}",
#         "demand_std": "{:.4f}",
#         "demand_max": "{:,.2f}",
#         "zero_ratio": "{:.2%}",
#         "positive_ratio": "{:.2%}"
#     })
# )


# # ------------------------------------------------------------
# # 5. 주요 점검 시간대 추출
# # 10~11시, 15~17시
# # ------------------------------------------------------------

# focus_hours = [10, 11, 15, 16, 17]

# focus_hour_summary_df = (
#     hourly_demand_summary_df[
#         hourly_demand_summary_df[hour_col].isin(focus_hours)
#     ]
#     .copy()
#     .sort_values([hour_col, "dataset"])
#     .reset_index(drop=True)
# )


# print("\n")
# print("=" * 70)
# print("주요 시간대 수요 분포: 10~11시, 15~17시")
# print("=" * 70)

# display(
#     focus_hour_summary_df.style.format({
#         "demand_sum": "{:,.2f}",
#         "demand_mean": "{:.4f}",
#         "demand_median": "{:.4f}",
#         "demand_std": "{:.4f}",
#         "demand_max": "{:,.2f}",
#         "zero_ratio": "{:.2%}",
#         "positive_ratio": "{:.2%}"
#     })
# )


# # ------------------------------------------------------------
# # 6. 주요 시간대 데이터셋 간 차이 비교
# # Train을 기준으로 Validation·Test 차이 계산
# # ------------------------------------------------------------

# focus_comparison_df = (
#     focus_hour_summary_df
#     .pivot(
#         index=hour_col,
#         columns="dataset",
#         values=[
#             "row_count",
#             "demand_sum",
#             "demand_mean",
#             "zero_ratio",
#             "positive_ratio"
#         ]
#     )
# )

# focus_comparison_df.columns = [
#     f"{metric}_{dataset}"
#     for metric, dataset in focus_comparison_df.columns
# ]

# focus_comparison_df = (
#     focus_comparison_df
#     .reset_index()
#     .sort_values(hour_col)
# )


# for compare_dataset in ["Validation", "Test"]:

#     # 평균 수요 차이
#     focus_comparison_df[
#         f"demand_mean_diff_{compare_dataset}_vs_Train"
#     ] = (
#         focus_comparison_df[f"demand_mean_{compare_dataset}"]
#         - focus_comparison_df["demand_mean_Train"]
#     )

#     # 평균 수요 변화율
#     focus_comparison_df[
#         f"demand_mean_change_rate_{compare_dataset}_vs_Train"
#     ] = np.where(
#         focus_comparison_df["demand_mean_Train"] != 0,
#         (
#             focus_comparison_df[f"demand_mean_{compare_dataset}"]
#             - focus_comparison_df["demand_mean_Train"]
#         )
#         / focus_comparison_df["demand_mean_Train"],
#         np.nan
#     )

#     # 0수요 비율 차이
#     focus_comparison_df[
#         f"zero_ratio_diff_{compare_dataset}_vs_Train"
#     ] = (
#         focus_comparison_df[f"zero_ratio_{compare_dataset}"]
#         - focus_comparison_df["zero_ratio_Train"]
#     )

#     # 수요 발생 비율 차이
#     focus_comparison_df[
#         f"positive_ratio_diff_{compare_dataset}_vs_Train"
#     ] = (
#         focus_comparison_df[f"positive_ratio_{compare_dataset}"]
#         - focus_comparison_df["positive_ratio_Train"]
#     )


# print("\n")
# print("=" * 70)
# print("주요 시간대 Train 대비 Validation·Test 차이")
# print("=" * 70)

# display(
#     focus_comparison_df.style.format({
#         col: "{:.2%}"
#         for col in focus_comparison_df.columns
#         if (
#             "ratio" in col
#             or "change_rate" in col
#         )
#     })
# )


# # ------------------------------------------------------------
# # 7. 시간대별 행 수 구성 비율 확인
# # 특정 시간이 Validation·Test에서 빠졌는지 확인
# # ------------------------------------------------------------

# hour_row_ratio_list = []

# for split_name, split_df in split_df_dict.items():

#     temp = (
#         split_df[hour_col]
#         .value_counts(dropna=False, normalize=True)
#         .rename("row_ratio")
#         .reset_index()
#     )

#     temp.columns = [hour_col, "row_ratio"]
#     temp["dataset"] = split_name

#     hour_row_ratio_list.append(temp)


# hour_row_ratio_df = pd.concat(
#     hour_row_ratio_list,
#     ignore_index=True
# )

# focus_hour_row_ratio_df = (
#     hour_row_ratio_df[
#         hour_row_ratio_df[hour_col].isin(focus_hours)
#     ]
#     .sort_values([hour_col, "dataset"])
#     .reset_index(drop=True)
# )


# print("\n")
# print("=" * 70)
# print("주요 시간대 행 구성 비율")
# print("=" * 70)

# display(
#     focus_hour_row_ratio_df.style.format({
#         "row_ratio": "{:.2%}"
#     })
# )


# # ------------------------------------------------------------
# # 8. 그래프 1
# # 시간대별 평균 수요 비교
# # ------------------------------------------------------------

# plt.figure(figsize=(13, 5))

# for split_name in ["Train", "Validation", "Test"]:

#     plot_df = hourly_demand_summary_df[
#         hourly_demand_summary_df["dataset"] == split_name
#     ]

#     plt.plot(
#         plot_df[hour_col],
#         plot_df["demand_mean"],
#         marker="o",
#         label=split_name
#     )

# for focus_hour in focus_hours:
#     plt.axvline(
#         x=focus_hour,
#         linestyle="--",
#         alpha=0.25
#     )

# plt.title("Train·Validation·Test 시간대별 평균 수요")
# plt.xlabel("Hour")
# plt.ylabel("Mean Demand")
# plt.xticks(
#     sorted(
#         hourly_demand_summary_df[hour_col]
#         .dropna()
#         .unique()
#     )
# )
# plt.legend()
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()


# # ------------------------------------------------------------
# # 9. 그래프 2
# # 시간대별 수요 발생 비율 비교
# # ------------------------------------------------------------

# plt.figure(figsize=(13, 5))

# for split_name in ["Train", "Validation", "Test"]:

#     plot_df = hourly_demand_summary_df[
#         hourly_demand_summary_df["dataset"] == split_name
#     ]

#     plt.plot(
#         plot_df[hour_col],
#         plot_df["positive_ratio"],
#         marker="o",
#         label=split_name
#     )

# for focus_hour in focus_hours:
#     plt.axvline(
#         x=focus_hour,
#         linestyle="--",
#         alpha=0.25
#     )

# plt.title("Train·Validation·Test 시간대별 수요 발생 비율")
# plt.xlabel("Hour")
# plt.ylabel("Positive Demand Ratio")
# plt.xticks(
#     sorted(
#         hourly_demand_summary_df[hour_col]
#         .dropna()
#         .unique()
#     )
# )
# plt.ylim(0, 1)
# plt.legend()
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()


# # ------------------------------------------------------------
# # 10. 그래프 3
# # 시간대별 0수요 비율 비교
# # ------------------------------------------------------------

# plt.figure(figsize=(13, 5))

# for split_name in ["Train", "Validation", "Test"]:

#     plot_df = hourly_demand_summary_df[
#         hourly_demand_summary_df["dataset"] == split_name
#     ]

#     plt.plot(
#         plot_df[hour_col],
#         plot_df["zero_ratio"],
#         marker="o",
#         label=split_name
#     )

# for focus_hour in focus_hours:
#     plt.axvline(
#         x=focus_hour,
#         linestyle="--",
#         alpha=0.25
#     )

# plt.title("Train·Validation·Test 시간대별 0수요 비율")
# plt.xlabel("Hour")
# plt.ylabel("Zero Demand Ratio")
# plt.xticks(
#     sorted(
#         hourly_demand_summary_df[hour_col]
#         .dropna()
#         .unique()
#     )
# )
# plt.ylim(0, 1)
# plt.legend()
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()


# # ------------------------------------------------------------
# # 11. 그래프 4
# # 주요 시간대 평균 수요 막대그래프
# # ------------------------------------------------------------

# focus_mean_pivot_df = (
#     focus_hour_summary_df
#     .pivot(
#         index=hour_col,
#         columns="dataset",
#         values="demand_mean"
#     )
#     .reindex(columns=["Train", "Validation", "Test"])
# )

# focus_mean_pivot_df.plot(
#     kind="bar",
#     figsize=(11, 5)
# )

# plt.title("주요 시간대 평균 수요 비교")
# plt.xlabel("Hour")
# plt.ylabel("Mean Demand")
# plt.xticks(rotation=0)
# plt.grid(axis="y", alpha=0.3)
# plt.tight_layout()
# plt.show()


# # ------------------------------------------------------------
# # 12. 주요 시간대 원자료 분포 요약
# # 수요가 발생한 행만 별도 확인
# # ------------------------------------------------------------

# positive_focus_summary_list = []

# for split_name, split_df in split_df_dict.items():

#     positive_focus_df = split_df[
#         split_df[hour_col].isin(focus_hours)
#         & (split_df[target_col] > 0)
#     ].copy()

#     positive_summary = (
#         positive_focus_df
#         .groupby(hour_col, observed=True)[target_col]
#         .agg(
#             positive_row_count="size",
#             positive_demand_mean="mean",
#             positive_demand_median="median",
#             positive_demand_std="std",
#             positive_demand_min="min",
#             positive_demand_max="max"
#         )
#         .reset_index()
#     )

#     positive_summary["dataset"] = split_name

#     positive_focus_summary_list.append(positive_summary)


# positive_focus_summary_df = pd.concat(
#     positive_focus_summary_list,
#     ignore_index=True
# ).sort_values(
#     [hour_col, "dataset"]
# ).reset_index(drop=True)


# print("\n")
# print("=" * 70)
# print("주요 시간대 수요 발생 행만 비교")
# print("=" * 70)

# display(
#     positive_focus_summary_df.style.format({
#         "positive_demand_mean": "{:.4f}",
#         "positive_demand_median": "{:.4f}",
#         "positive_demand_std": "{:.4f}",
#         "positive_demand_min": "{:.2f}",
#         "positive_demand_max": "{:.2f}"
#     })
# )


# # ------------------------------------------------------------
# # 13. 간단한 자동 경고
# # Train 대비 평균 수요 또는 수요 발생률 차이가 큰 시간 표시
# # ------------------------------------------------------------

# warning_records = []

# mean_change_threshold = 0.30
# positive_ratio_diff_threshold = 0.05

# for _, row in focus_comparison_df.iterrows():

#     current_hour = int(row[hour_col])

#     for compare_dataset in ["Validation", "Test"]:

#         mean_change_col = (
#             f"demand_mean_change_rate_{compare_dataset}_vs_Train"
#         )

#         positive_diff_col = (
#             f"positive_ratio_diff_{compare_dataset}_vs_Train"
#         )

#         mean_change = row.get(mean_change_col, np.nan)
#         positive_diff = row.get(positive_diff_col, np.nan)

#         if (
#             pd.notna(mean_change)
#             and abs(mean_change) >= mean_change_threshold
#         ):
#             warning_records.append({
#                 "hour": current_hour,
#                 "dataset": compare_dataset,
#                 "check_type": "평균 수요 변화율",
#                 "value": mean_change,
#                 "criterion": "Train 대비 절댓값 30% 이상"
#             })

#         if (
#             pd.notna(positive_diff)
#             and abs(positive_diff) >= positive_ratio_diff_threshold
#         ):
#             warning_records.append({
#                 "hour": current_hour,
#                 "dataset": compare_dataset,
#                 "check_type": "수요 발생률 차이",
#                 "value": positive_diff,
#                 "criterion": "Train 대비 5%p 이상"
#             })


# warning_df = pd.DataFrame(warning_records)


# print("\n")
# print("=" * 70)
# print("주요 시간대 자동 점검 결과")
# print("=" * 70)

# if len(warning_df) == 0:

#     print(
#         "설정한 기준에서 큰 분포 차이가 발견되지 않았습니다.\n"
#         "- 평균 수요 변화율: Train 대비 30% 미만\n"
#         "- 수요 발생률 차이: Train 대비 5%p 미만"
#     )

# else:

#     display(
#         warning_df.style.format({
#             "value": "{:.2%}"
#         })
#     )


# print("\n점검 완료")

In [42]:
# ============================================================
# 3-1. Seasonal Naive 기준선 모델 생성
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 기준 컬럼 설정
# ------------------------------------------------------------

SERIES_KEY_COLS = [
    "store_id",
    "product_id"
]

TIME_COL = "forecast_datetime"
TARGET_COL = "demand_target"


# ------------------------------------------------------------
# 2. 날짜형 정리
# ------------------------------------------------------------

a1_training_df[TIME_COL] = pd.to_datetime(
    a1_training_df[TIME_COL]
)

train_df[TIME_COL] = pd.to_datetime(
    train_df[TIME_COL]
)

validation_df[TIME_COL] = pd.to_datetime(
    validation_df[TIME_COL]
)


# ------------------------------------------------------------
# 3. 과거 실제 판매량 Lookup 테이블 생성
# ------------------------------------------------------------

history_df = (
    a1_training_df[
        SERIES_KEY_COLS
        + [TIME_COL, TARGET_COL]
    ]
    .copy()
)


# 7일 전 동일 시간 판매량
weekly_lookup_df = history_df.rename(
    columns={
        TARGET_COL: "weekly_naive_pred"
    }
)

weekly_lookup_df[TIME_COL] = (
    weekly_lookup_df[TIME_COL]
    + pd.Timedelta(days=7)
)


# 1일 전 동일 시간 판매량
daily_lookup_df = history_df.rename(
    columns={
        TARGET_COL: "daily_naive_pred"
    }
)

daily_lookup_df[TIME_COL] = (
    daily_lookup_df[TIME_COL]
    + pd.Timedelta(days=1)
)


# ------------------------------------------------------------
# 4. Train 기준 점포·상품별 중앙값 생성
# ------------------------------------------------------------

series_median_df = (
    train_df
    .groupby(
        SERIES_KEY_COLS,
        as_index=False
    )[TARGET_COL]
    .median()
    .rename(
        columns={
            TARGET_COL: "train_series_median"
        }
    )
)

train_global_median = (
    train_df[TARGET_COL]
    .median()
)


# ------------------------------------------------------------
# 5. Validation 데이터에 과거 판매량 연결
# ------------------------------------------------------------

validation_baseline_df = (
    validation_df
    .copy()
    .merge(
        weekly_lookup_df,
        on=SERIES_KEY_COLS + [TIME_COL],
        how="left"
    )
    .merge(
        daily_lookup_df,
        on=SERIES_KEY_COLS + [TIME_COL],
        how="left"
    )
    .merge(
        series_median_df,
        on=SERIES_KEY_COLS,
        how="left"
    )
)


# ------------------------------------------------------------
# 6. Seasonal Naive 최종 예측값 생성
# ------------------------------------------------------------

validation_baseline_df["seasonal_naive_source"] = np.select(
    [
        validation_baseline_df[
            "weekly_naive_pred"
        ].notna(),

        validation_baseline_df[
            "daily_naive_pred"
        ].notna(),

        validation_baseline_df[
            "train_series_median"
        ].notna()
    ],
    [
        "previous_week",
        "previous_day",
        "train_series_median"
    ],
    default="train_global_median"
)


validation_baseline_df["seasonal_naive_pred"] = (
    validation_baseline_df[
        "weekly_naive_pred"
    ]
    .fillna(
        validation_baseline_df[
            "daily_naive_pred"
        ]
    )
    .fillna(
        validation_baseline_df[
            "train_series_median"
        ]
    )
    .fillna(
        train_global_median
    )
    .clip(lower=0)
)


# ------------------------------------------------------------
# 7. 결과 확인
# ------------------------------------------------------------

print(
    "Validation 전체 행 수:",
    len(validation_baseline_df)
)

print(
    "Seasonal Naive 예측 결측 수:",
    validation_baseline_df[
        "seasonal_naive_pred"
    ].isna().sum()
)

print("\n예측값 생성 기준:")
print(
    validation_baseline_df[
        "seasonal_naive_source"
    ].value_counts(dropna=False)
)

display(
    validation_baseline_df[
        [
            "store_id",
            "product_id",
            "date",
            "hour",
            "forecast_datetime",
            "demand_target",
            "weekly_naive_pred",
            "daily_naive_pred",
            "seasonal_naive_pred",
            "seasonal_naive_source"
        ]
    ].head(10)
)
# Validation 전체 행: 62,092행
# 최종 기준선 예측 결측: 0행
# 7일 전 동일 시간 판매량 사용: 57,760행
# 7일 전 값이 없어 1일 전 동일 시간 판매량 사용: 4,332행

Validation 전체 행 수: 62092
Seasonal Naive 예측 결측 수: 0

예측값 생성 기준:
seasonal_naive_source
previous_week    57760
previous_day      4332
Name: count, dtype: int64


,store_id,product_id,date,hour,forecast_datetime,demand_target,weekly_naive_pred,daily_naive_pred,seasonal_naive_pred,seasonal_naive_source
0,S01,P001,2025-10-01,10,2025-10-01 10:00:00,0.0000,1.0000,0.0000,1.0000,previous_week
1,S01,P001,2025-10-01,11,2025-10-01 11:00:00,0.0000,3.0000,0.0000,3.0000,previous_week
2,S01,P001,2025-10-01,12,2025-10-01 12:00:00,3.0000,0.0000,0.0000,0.0000,previous_week
3,S01,P001,2025-10-01,13,2025-10-01 13:00:00,0.0000,0.0000,4.0000,0.0000,previous_week
4,S01,P001,2025-10-01,14,2025-10-01 14:00:00,0.0000,0.0000,1.0000,0.0000,previous_week
5,S01,P001,2025-10-01,15,2025-10-01 15:00:00,0.0000,1.0000,0.0000,1.0000,previous_week
6,S01,P001,2025-10-01,16,2025-10-01 16:00:00,0.0000,2.0000,3.0000,2.0000,previous_week
7,S01,P001,2025-10-01,17,2025-10-01 17:00:00,0.0000,0.0000,0.0000,0.0000,previous_week
8,S01,P001,2025-10-01,18,2025-10-01 18:00:00,1.0000,1.0000,1.0000,1.0000,previous_week
9,S01,P001,2025-10-01,19,2025-10-01 19:00:00,0.0000,1.0000,0.0000,1.0000,previous_week


In [43]:
# ============================================================
# 3-2. Moving Average
# - 최근 7일 동일 시간대 수요 평균
# - 예측값이 없으면 Train 시계열 중앙값으로 대체
# ============================================================

MA_WINDOW_DAYS = 7

# 재실행 시 기존 결과 컬럼 제거
existing_ma_columns = [
    "moving_average_pred",
    "moving_average_available_days",
    "moving_average_source",
    "ma_series_median"
]

validation_baseline_df = validation_baseline_df.drop(
    columns=existing_ma_columns,
    errors="ignore"
)


# ------------------------------------------------------------
# 1. Moving Average 계산에 사용할 과거 데이터
# ------------------------------------------------------------

moving_average_history_df = pd.concat(
    [
        train_df[
            SERIES_KEY_COLS
            + [TIME_COL, TARGET_COL]
        ],
        validation_df[
            SERIES_KEY_COLS
            + [TIME_COL, TARGET_COL]
        ]
    ],
    ignore_index=True
).copy()

moving_average_history_df[TIME_COL] = pd.to_datetime(
    moving_average_history_df[TIME_COL]
)

moving_average_history_df = moving_average_history_df.sort_values(
    SERIES_KEY_COLS + [TIME_COL]
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. 최근 1~7일 동일 시간대 판매량 연결
# ------------------------------------------------------------

ma_lag_columns = []

for lag_day in range(1, MA_WINDOW_DAYS + 1):

    lag_column = f"ma_lag_{lag_day}d"
    ma_lag_columns.append(lag_column)

    lag_lookup_df = moving_average_history_df[
        SERIES_KEY_COLS
        + [TIME_COL, TARGET_COL]
    ].copy()

    # 과거 판매량을 미래 시점의 예측값으로 이동
    lag_lookup_df[TIME_COL] = (
        lag_lookup_df[TIME_COL]
        + pd.Timedelta(days=lag_day)
    )

    lag_lookup_df = lag_lookup_df.rename(
        columns={
            TARGET_COL: lag_column
        }
    )

    validation_baseline_df = validation_baseline_df.merge(
        lag_lookup_df,
        on=SERIES_KEY_COLS + [TIME_COL],
        how="left"
    )


# ------------------------------------------------------------
# 3. 최근 7일 동일 시간대 평균 계산
# ------------------------------------------------------------

validation_baseline_df[
    "moving_average_available_days"
] = validation_baseline_df[
    ma_lag_columns
].notna().sum(axis=1)

validation_baseline_df[
    "moving_average_pred"
] = validation_baseline_df[
    ma_lag_columns
].mean(
    axis=1,
    skipna=True
)


# ------------------------------------------------------------
# 4. 과거값이 없는 시계열에 사용할 대체값
# ------------------------------------------------------------

ma_series_median_df = (
    train_df
    .groupby(
        SERIES_KEY_COLS,
        as_index=False
    )[TARGET_COL]
    .median()
    .rename(
        columns={
            TARGET_COL: "ma_series_median"
        }
    )
)

validation_baseline_df = validation_baseline_df.merge(
    ma_series_median_df,
    on=SERIES_KEY_COLS,
    how="left"
)

ma_global_median = train_df[TARGET_COL].median()


# ------------------------------------------------------------
# 5. 결측 예측값 대체
# ------------------------------------------------------------

validation_baseline_df[
    "moving_average_source"
] = np.select(
    [
        validation_baseline_df[
            "moving_average_available_days"
        ] > 0,

        validation_baseline_df[
            "ma_series_median"
        ].notna()
    ],
    [
        "recent_7day_same_hour_mean",
        "train_series_median"
    ],
    default="train_global_median"
)

validation_baseline_df[
    "moving_average_pred"
] = (
    validation_baseline_df[
        "moving_average_pred"
    ]
    .fillna(
        validation_baseline_df[
            "ma_series_median"
        ]
    )
    .fillna(ma_global_median)
    .clip(lower=0)
)


# ------------------------------------------------------------
# 6. 계산용 lag 컬럼 제거
# ------------------------------------------------------------

validation_baseline_df = validation_baseline_df.drop(
    columns=ma_lag_columns
)


# ------------------------------------------------------------
# 7. 결과 확인
# ------------------------------------------------------------

print(
    "Validation 전체 행 수:",
    len(validation_baseline_df)
)

print(
    "Moving Average 예측 결측 수:",
    validation_baseline_df[
        "moving_average_pred"
    ].isna().sum()
)

print("\n사용 가능한 과거 일수:")
print(
    validation_baseline_df[
        "moving_average_available_days"
    ]
    .value_counts()
    .sort_index()
)

print("\n예측값 생성 방식:")
print(
    validation_baseline_df[
        "moving_average_source"
    ]
    .value_counts()
)

validation_baseline_df[
    SERIES_KEY_COLS
    + [
        TIME_COL,
        TARGET_COL,
        "moving_average_pred",
        "moving_average_available_days",
        "moving_average_source"
    ]
].head(10)

Validation 전체 행 수: 62092
Moving Average 예측 결측 수: 0

사용 가능한 과거 일수:
moving_average_available_days
6    32908
7    29184
Name: count, dtype: int64

예측값 생성 방식:
moving_average_source
recent_7day_same_hour_mean    62092
Name: count, dtype: int64


,store_id,product_id,forecast_datetime,demand_target,moving_average_pred,moving_average_available_days,moving_average_source
0,S01,P001,2025-10-01 10:00:00,0.0000,0.3333,6,recent_7day_same_hour_mean
1,S01,P001,2025-10-01 11:00:00,0.0000,0.8333,6,recent_7day_same_hour_mean
2,S01,P001,2025-10-01 12:00:00,3.0000,0.0000,6,recent_7day_same_hour_mean
3,S01,P001,2025-10-01 13:00:00,0.0000,0.6667,6,recent_7day_same_hour_mean
4,S01,P001,2025-10-01 14:00:00,0.0000,0.3333,6,recent_7day_same_hour_mean
5,S01,P001,2025-10-01 15:00:00,0.0000,0.5000,6,recent_7day_same_hour_mean
6,S01,P001,2025-10-01 16:00:00,0.0000,1.3333,6,recent_7day_same_hour_mean
7,S01,P001,2025-10-01 17:00:00,0.0000,0.5000,6,recent_7day_same_hour_mean
8,S01,P001,2025-10-01 18:00:00,1.0000,0.8333,6,recent_7day_same_hour_mean
9,S01,P001,2025-10-01 19:00:00,0.0000,0.1667,6,recent_7day_same_hour_mean


In [44]:
# ============================================================
# 3-3. Croston
# - 간헐적 수요를 위한 기준선 모델
# - Train 데이터로 초기화
# - Validation을 시간순으로 순차 예측
# ============================================================

CROSTON_ALPHA = 0.1

# 재실행 시 기존 결과 컬럼 제거
validation_baseline_df = validation_baseline_df.drop(
    columns=[
        "croston_pred",
        "croston_train_nonzero_count"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 1. Croston 초기 상태 생성 함수
# ------------------------------------------------------------

def initialize_croston_state(
    demand_values,
    alpha=0.1
):
    """
    Train 판매량으로 Croston 상태를 초기화합니다.

    z: 0이 아닌 판매량의 평활값
    p: 판매 발생 간격의 평활값
    q: 마지막 판매 발생 이후 경과 시간
    """

    demand_values = np.asarray(
        demand_values,
        dtype=float
    )

    nonzero_positions = np.flatnonzero(
        demand_values > 0
    )

    # Train에서 판매가 한 번도 없었던 경우
    if len(nonzero_positions) == 0:
        return {
            "z": 0.0,
            "p": 1.0,
            "q": max(len(demand_values), 1),
            "forecast": 0.0,
            "nonzero_count": 0
        }

    first_nonzero_position = nonzero_positions[0]

    # 최초 판매량과 최초 판매 발생 간격으로 초기화
    z = float(
        demand_values[first_nonzero_position]
    )

    p = float(
        first_nonzero_position + 1
    )

    q = 1

    # 최초 판매 이후 Train 데이터를 순서대로 반영
    for demand in demand_values[
        first_nonzero_position + 1:
    ]:

        if demand > 0:
            z = (
                z
                + alpha * (demand - z)
            )

            p = (
                p
                + alpha * (q - p)
            )

            q = 1

        else:
            q += 1

    forecast = (
        z / p
        if p > 0
        else 0.0
    )

    return {
        "z": z,
        "p": p,
        "q": q,
        "forecast": forecast,
        "nonzero_count": len(nonzero_positions)
    }


# ------------------------------------------------------------
# 2. 시계열 키 형태 통일 함수
# ------------------------------------------------------------

def normalize_series_key(key):
    if isinstance(key, tuple):
        return key

    return (key,)


# ------------------------------------------------------------
# 3. Train 시계열별 데이터 저장
# ------------------------------------------------------------

croston_train_group_dict = {
    normalize_series_key(series_key): group_df.sort_values(
        TIME_COL
    )
    for series_key, group_df in train_df.groupby(
        SERIES_KEY_COLS,
        sort=False,
        dropna=False
    )
}


# ------------------------------------------------------------
# 4. Validation 예측용 데이터 준비
# ------------------------------------------------------------

croston_validation_df = validation_df[
    SERIES_KEY_COLS
    + [TIME_COL, TARGET_COL]
].copy()

croston_validation_df[TIME_COL] = pd.to_datetime(
    croston_validation_df[TIME_COL]
)

croston_validation_df = croston_validation_df.sort_values(
    SERIES_KEY_COLS + [TIME_COL]
).reset_index(drop=True)

croston_validation_df["croston_pred"] = np.nan
croston_validation_df[
    "croston_train_nonzero_count"
] = 0


# ------------------------------------------------------------
# 5. 시계열별 순차 예측
# ------------------------------------------------------------

for series_key, validation_group_df in (
    croston_validation_df.groupby(
        SERIES_KEY_COLS,
        sort=False,
        dropna=False
    )
):

    normalized_key = normalize_series_key(
        series_key
    )

    train_group_df = croston_train_group_dict.get(
        normalized_key
    )

    # 해당 시계열의 Train 판매량
    if train_group_df is not None:

        train_demand_values = (
            train_group_df[TARGET_COL]
            .fillna(0)
            .clip(lower=0)
            .to_numpy(dtype=float)
        )

    else:
        train_demand_values = np.array(
            [],
            dtype=float
        )

    croston_state = initialize_croston_state(
        demand_values=train_demand_values,
        alpha=CROSTON_ALPHA
    )

    z = croston_state["z"]
    p = croston_state["p"]
    q = croston_state["q"]

    validation_indices = (
        validation_group_df.index
    )

    validation_actual_values = (
        validation_group_df[TARGET_COL]
        .fillna(0)
        .clip(lower=0)
        .to_numpy(dtype=float)
    )

    series_predictions = []

    # 각 시점 예측 후 실제값으로 상태 갱신
    for actual_demand in validation_actual_values:

        current_forecast = (
            z / p
            if p > 0
            else 0.0
        )

        series_predictions.append(
            max(current_forecast, 0.0)
        )

        # 현재 시점 실제 판매량은
        # 다음 시점 예측부터 사용
        if actual_demand > 0:

            z = (
                z
                + CROSTON_ALPHA
                * (actual_demand - z)
            )

            p = (
                p
                + CROSTON_ALPHA
                * (q - p)
            )

            q = 1

        else:
            q += 1

    croston_validation_df.loc[
        validation_indices,
        "croston_pred"
    ] = series_predictions

    croston_validation_df.loc[
        validation_indices,
        "croston_train_nonzero_count"
    ] = croston_state["nonzero_count"]


# ------------------------------------------------------------
# 6. Validation 기준선 테이블에 결합
# ------------------------------------------------------------

croston_result_df = croston_validation_df[
    SERIES_KEY_COLS
    + [
        TIME_COL,
        "croston_pred",
        "croston_train_nonzero_count"
    ]
].copy()

validation_baseline_df = validation_baseline_df.merge(
    croston_result_df,
    on=SERIES_KEY_COLS + [TIME_COL],
    how="left"
)

validation_baseline_df[
    "croston_pred"
] = (
    validation_baseline_df["croston_pred"]
    .fillna(0)
    .clip(lower=0)
)


# ------------------------------------------------------------
# 7. 결과 확인
# ------------------------------------------------------------

print(
    "Validation 전체 행 수:",
    len(validation_baseline_df)
)

print(
    "Croston 예측 결측 수:",
    validation_baseline_df[
        "croston_pred"
    ].isna().sum()
)

print(
    "Croston 예측 최소값:",
    validation_baseline_df[
        "croston_pred"
    ].min()
)

print(
    "Croston 예측 최대값:",
    validation_baseline_df[
        "croston_pred"
    ].max()
)

print(
    "Train 판매 발생 이력이 없는 행 수:",
    (
        validation_baseline_df[
            "croston_train_nonzero_count"
        ] == 0
    ).sum()
)

validation_baseline_df[
    SERIES_KEY_COLS
    + [
        TIME_COL,
        TARGET_COL,
        "seasonal_naive_pred",
        "moving_average_pred",
        "croston_pred",
        "croston_train_nonzero_count"
    ]
].head(10)

# Croston 예측 최소값: 0.029208255371063457
# Croston 예측 최대값: 1.458416436860267
# => 시간당기대 판매량 0.029...~1.458...

Validation 전체 행 수: 62092
Croston 예측 결측 수: 0
Croston 예측 최소값: 0.029208255371063457
Croston 예측 최대값: 1.458416436860267
Train 판매 발생 이력이 없는 행 수: 0


,store_id,product_id,forecast_datetime,demand_target,seasonal_naive_pred,moving_average_pred,croston_pred,croston_train_nonzero_count
0,S01,P001,2025-10-01 10:00:00,0.0000,1.0000,0.3333,0.7150,731
1,S01,P001,2025-10-01 11:00:00,0.0000,3.0000,0.8333,0.7150,731
2,S01,P001,2025-10-01 12:00:00,3.0000,0.0000,0.0000,0.7150,731
3,S01,P001,2025-10-01 13:00:00,0.0000,0.0000,0.6667,0.6972,731
4,S01,P001,2025-10-01 14:00:00,0.0000,0.0000,0.3333,0.6972,731
5,S01,P001,2025-10-01 15:00:00,0.0000,1.0000,0.5000,0.6972,731
6,S01,P001,2025-10-01 16:00:00,0.0000,2.0000,1.3333,0.6972,731
7,S01,P001,2025-10-01 17:00:00,0.0000,0.0000,0.5000,0.6972,731
8,S01,P001,2025-10-01 18:00:00,1.0000,1.0000,0.8333,0.6972,731
9,S01,P001,2025-10-01 19:00:00,0.0000,1.0000,0.1667,0.6064,731


In [45]:
# ============================================================
# 4-1. LightGBM Quantile 모델 입력 피처 확정
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 4-1-1. 기본 컬럼 설정
# ------------------------------------------------------------

TARGET_COL = "demand_target"
TIME_COL = "forecast_datetime"

SERIES_KEY_COLS = [
    "store_id",
    "product_id"
]


# 기존 분할 데이터는 보존하고 별도 복사본 사용
train_lgb_df = train_df.copy()
validation_lgb_df = validation_df.copy()
test_lgb_df = test_df.copy()


# ------------------------------------------------------------
# 4-1-2. 날짜·시간 기본 피처 보완
# ------------------------------------------------------------

def add_lightgbm_time_features(df):

    df = df.copy()

    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL],
        errors="coerce"
    )

    # 이미 생성된 컬럼은 그대로 사용
    if "year" not in df.columns:
        df["year"] = df[TIME_COL].dt.year

    if "month" not in df.columns:
        df["month"] = df[TIME_COL].dt.month

    if "day" not in df.columns:
        df["day"] = df[TIME_COL].dt.day

    if "day_of_week" not in df.columns:
        df["day_of_week"] = df[TIME_COL].dt.dayofweek

    if "week_of_year" not in df.columns:
        df["week_of_year"] = (
            df[TIME_COL]
            .dt
            .isocalendar()
            .week
            .astype("Int64")
        )

    if "hour" not in df.columns:
        df["hour"] = df[TIME_COL].dt.hour

    if "is_weekend" not in df.columns:
        df["is_weekend"] = (
            df["day_of_week"]
            .isin([5, 6])
            .astype("int8")
        )

    return df


train_lgb_df = add_lightgbm_time_features(train_lgb_df)
validation_lgb_df = add_lightgbm_time_features(validation_lgb_df)
test_lgb_df = add_lightgbm_time_features(test_lgb_df)


# ------------------------------------------------------------
# 4-1-3. LightGBM 입력에서 제외할 컬럼 정의
# ------------------------------------------------------------

# 예측 대상 및 식별용 컬럼
BASE_EXCLUDE_COLS = {
    TARGET_COL,
    TIME_COL,
    "date",
    "series_id",
    "split",
    "data_split",
    "dataset_split",
    "fold"
}


# 해당 시간대의 실제 판매 결과
# 예측 시점에는 알 수 없으므로 입력 피처에서 제외
CURRENT_SALES_RESULT_COLS = {
    "sales_qty",
    "regular_sales_qty",
    "discount_sales_qty",
    "gross_sales_amount",
    "sales_amount",
    "discount_amount",
    "weighted_discount_rate",
    "receipt_count",
    "sales_observed_flag",
    "has_sales_record",
    "is_sales_observed"
}


# 기준선 모델 결과
BASELINE_RESULT_COLS = {
    "seasonal_naive_pred",
    "seasonal_naive_source",
    "moving_average_pred",
    "moving_average_available_days",
    "moving_average_source",
    "ma_series_median",
    "croston_pred",
    "croston_train_nonzero_count"
}


# ID로 충분하기 때문에 중복 정보가 되는 이름 컬럼 제외
RAW_NAME_COLS = {
    "store_name",
    "product_name",
    "item_name"
}


# ------------------------------------------------------------
# 추가 누수 및 예측 시점 미확보 컬럼
# ------------------------------------------------------------

LEAKAGE_COLS = {
    "split_date",

    "avg_unit_price",
    "avg_sale_unit_price",
    "has_sales",
    "is_regular_condition",
    "is_mixed_price_condition",
    "regular_price_model",
    "price_ratio",
    "sale_price_model",

    # 미래 시간대의 실제 방문객 수
    "visitor_count",
    "cumulative_daily_visitors",
}


EXCLUDE_COLS = (
    BASE_EXCLUDE_COLS
    | CURRENT_SALES_RESULT_COLS
    | BASELINE_RESULT_COLS
    | RAW_NAME_COLS
    | LEAKAGE_COLS
)


# ------------------------------------------------------------
# 4-1-4. 추가적인 누수 가능 컬럼 판별 함수
# ------------------------------------------------------------

def is_excluded_feature(column_name):

    column_lower = column_name.lower()

    # 명시적 제외 컬럼
    if column_name in EXCLUDE_COLS:
        return True

    # 할인 전 기본수요 예측이므로
    # 할인·프로모션 관련 변수는 LightGBM 입력에서 제외
    if (
        "discount" in column_lower
        or "promotion" in column_lower
        or "markdown" in column_lower
    ):
        return True

    # 미래 정보를 직접 포함하는 컬럼
    if column_lower.startswith(
        (
            "lead_",
            "future_",
            "next_"
        )
    ):
        return True

    # 이미 계산된 모델 예측값
    if column_lower.endswith(
        (
            "_pred",
            "_prediction",
            "_forecast"
        )
    ):
        return True

    # 판매 후 재고처럼 결과가 반영된 컬럼
    if any(
        keyword in column_lower
        for keyword in [
            "after_sales",
            "post_sales",
            "end_of_hour",
            "ending_inventory"
        ]
    ):
        return True

    # 원본 날짜·시각 컬럼
    # 파생된 hour, day_of_week 등은 제외되지 않음
    if column_lower.endswith(
        (
            "_datetime",
            "_timestamp"
        )
    ):
        return True

    return False


# ------------------------------------------------------------
# 4-1-5. 세 분할 데이터에 공통으로 존재하는 컬럼만 사용
# ------------------------------------------------------------

common_columns = [
    column
    for column in train_lgb_df.columns
    if (
        column in validation_lgb_df.columns
        and column in test_lgb_df.columns
    )
]


FEATURE_COLS = [
    column
    for column in common_columns
    if not is_excluded_feature(column)
]


# 전체가 결측인 컬럼 제거
FEATURE_COLS = [
    column
    for column in FEATURE_COLS
    if not train_lgb_df[column].isna().all()
]


# ------------------------------------------------------------
# 4-1-6. 범주형 피처 확정
# ------------------------------------------------------------

FORCED_CATEGORICAL_COLS = [
    "store_id",
    "product_id",
    "category",
    "category_id",
    "subcategory",
    "subcategory_id",
    "store_type",
    "region",
    "time_slot",
    "day_type",
    "weather_type"
]


CATEGORICAL_COLS = []

for column in FEATURE_COLS:

    if column in FORCED_CATEGORICAL_COLS:
        CATEGORICAL_COLS.append(column)

    elif (
        pd.api.types.is_object_dtype(train_lgb_df[column])
        or isinstance(
            train_lgb_df[column].dtype,
            pd.CategoricalDtype
        )
    ):
        CATEGORICAL_COLS.append(column)


# 중복 제거
CATEGORICAL_COLS = list(dict.fromkeys(CATEGORICAL_COLS))

# ------------------------------------------------------------
# 4-1-7. Train 기준으로 범주 수준 통일
# ------------------------------------------------------------

for column in CATEGORICAL_COLS:

    # Train 데이터에 존재하는 범주만 사용
    train_values = (
        train_lgb_df[column]
        .astype("string")
        .fillna("__MISSING__")
    )

    category_values = train_values.drop_duplicates().tolist()

    # 결측값과 처음 보는 범주 처리용
    for special_category in [
        "__MISSING__",
        "__UNKNOWN__"
    ]:
        if special_category not in category_values:
            category_values.append(special_category)

    category_dtype = pd.CategoricalDtype(
        categories=category_values
    )

    def convert_category(series):

        series = (
            series
            .astype("string")
            .fillna("__MISSING__")
        )

        # Train에 없던 Validation·Test 값
        series = series.where(
            series.isin(category_values),
            "__UNKNOWN__"
        )

        return series.astype(category_dtype)

    train_lgb_df[column] = convert_category(
        train_lgb_df[column]
    )

    validation_lgb_df[column] = convert_category(
        validation_lgb_df[column]
    )

    test_lgb_df[column] = convert_category(
        test_lgb_df[column]
    )

# ------------------------------------------------------------
# 4-1-8. 불리언·수치형 피처 정리
# ------------------------------------------------------------

NUMERIC_COLS = [
    column
    for column in FEATURE_COLS
    if column not in CATEGORICAL_COLS
]


for column in NUMERIC_COLS:

    # True/False는 0/1로 변환
    if pd.api.types.is_bool_dtype(train_lgb_df[column]):

        train_lgb_df[column] = (
            train_lgb_df[column]
            .astype("int8")
        )

        validation_lgb_df[column] = (
            validation_lgb_df[column]
            .astype("int8")
        )

        test_lgb_df[column] = (
            test_lgb_df[column]
            .astype("int8")
        )

    else:

        train_lgb_df[column] = pd.to_numeric(
            train_lgb_df[column],
            errors="coerce"
        )

        validation_lgb_df[column] = pd.to_numeric(
            validation_lgb_df[column],
            errors="coerce"
        )

        test_lgb_df[column] = pd.to_numeric(
            test_lgb_df[column],
            errors="coerce"
        )


# 무한대 값은 결측으로 처리
train_lgb_df[NUMERIC_COLS] = (
    train_lgb_df[NUMERIC_COLS]
    .replace([np.inf, -np.inf], np.nan)
)

validation_lgb_df[NUMERIC_COLS] = (
    validation_lgb_df[NUMERIC_COLS]
    .replace([np.inf, -np.inf], np.nan)
)

test_lgb_df[NUMERIC_COLS] = (
    test_lgb_df[NUMERIC_COLS]
    .replace([np.inf, -np.inf], np.nan)
)


# ------------------------------------------------------------
# 4-1-9. 타깃 결측 행 제거
# ------------------------------------------------------------

train_lgb_df = (
    train_lgb_df[
        train_lgb_df[TARGET_COL].notna()
    ]
    .copy()
)

validation_lgb_df = (
    validation_lgb_df[
        validation_lgb_df[TARGET_COL].notna()
    ]
    .copy()
)

test_lgb_df = (
    test_lgb_df[
        test_lgb_df[TARGET_COL].notna()
    ]
    .copy()
)


# ------------------------------------------------------------
# 4-1-10. 최종 X·y 생성
# ------------------------------------------------------------

X_train = train_lgb_df[FEATURE_COLS].copy()
y_train = (
    train_lgb_df[TARGET_COL]
    .astype(float)
    .clip(lower=0)
)

X_validation = validation_lgb_df[FEATURE_COLS].copy()
y_validation = (
    validation_lgb_df[TARGET_COL]
    .astype(float)
    .clip(lower=0)
)

X_test = test_lgb_df[FEATURE_COLS].copy()
y_test = (
    test_lgb_df[TARGET_COL]
    .astype(float)
    .clip(lower=0)
)


# ------------------------------------------------------------
# 4-1-11. 입력 피처 확정 결과 확인
# ------------------------------------------------------------

print("LightGBM Quantile 입력 피처 확정 완료")
print("-" * 60)

print("전체 입력 피처 수:", len(FEATURE_COLS))
print("범주형 피처 수:", len(CATEGORICAL_COLS))
print("수치형 피처 수:", len(NUMERIC_COLS))

print()
print("Train X / y:", X_train.shape, y_train.shape)
print(
    "Validation X / y:",
    X_validation.shape,
    y_validation.shape
)
print("Test X / y:", X_test.shape, y_test.shape)

print()
print("범주형 피처:")
print(CATEGORICAL_COLS)

print()
print("최종 입력 피처:")
print(FEATURE_COLS)

print()
print("Train 타깃 요약:")
print(y_train.describe())

LightGBM Quantile 입력 피처 확정 완료
------------------------------------------------------------
전체 입력 피처 수: 109
범주형 피처 수: 14
수치형 피처 수: 95

Train X / y: (368220, 109) (368220,)
Validation X / y: (62092, 109) (62092,)
Test X / y: (62092, 109) (62092,)

범주형 피처:
['store_id', 'product_id', 'day_of_week', 'day_type', 'holiday_name', 'season', 'event_type', 'area_type', 'category', 'subcategory', 'sales_type', 'unit', 'freshness_decay_type', 'time_slot']

최종 입력 피처:
['hour', 'store_id', 'product_id', 'open_hour', 'close_hour', 'hours_until_close', 'year', 'month', 'day', 'day_of_week', 'week_of_month', 'day_type', 'is_weekend', 'is_holiday', 'holiday_name', 'season', 'season_index', 'is_event_day', 'event_type', 'event_index', 'area_type', 'resident_pop', 'floating_idx', 'order_error_sigma', 'category', 'subcategory', 'sales_type', 'unit', 'standard_weight_kg', 'shelf_life_days', 'base_cost', 'base_price', 'margin_rate', 'freshness_decay_type', 'baseline_waste_rate', 'hour_num', 'month_num', 'day_o

In [46]:
# ============================================================
# 4-2. P10 LightGBM Quantile 모델
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_pinball_loss


# ------------------------------------------------------------
# 1. P10 모델 설정
# ------------------------------------------------------------

P10_ALPHA = 0.10

p10_model = lgb.LGBMRegressor(
    objective="quantile",
    alpha=P10_ALPHA,

    boosting_type="gbdt",
    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


# ------------------------------------------------------------
# 2. P10 모델 학습
# ------------------------------------------------------------

p10_model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_validation, y_validation)
    ],

    eval_names=[
        "validation"
    ],

    eval_metric="quantile",

    categorical_feature=CATEGORICAL_COLS,

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=100)
    ]
)


# ------------------------------------------------------------
# 3. Validation 구간 P10 예측
# ------------------------------------------------------------

p10_validation_pred_raw = p10_model.predict(
    X_validation,
    num_iteration=p10_model.best_iteration_
)

# 판매량은 음수가 될 수 없으므로 0 이상으로 보정
p10_validation_pred = np.clip(
    p10_validation_pred_raw,
    a_min=0,
    a_max=None
)


# ------------------------------------------------------------
# 4. P10 모델 평가
# ------------------------------------------------------------

p10_pinball_loss = mean_pinball_loss(
    y_validation,
    p10_validation_pred,
    alpha=P10_ALPHA
)

# 실제값이 P10 예측값 이하인 비율
p10_coverage = np.mean(
    y_validation.to_numpy() <= p10_validation_pred
)

negative_prediction_count = int(
    np.sum(p10_validation_pred_raw < 0)
)


# ------------------------------------------------------------
# 5. 결과 출력
# ------------------------------------------------------------

print("P10 모델 학습 완료")
print("-" * 50)
print("최적 반복 횟수:", p10_model.best_iteration_)
print(f"Validation Pinball Loss: {p10_pinball_loss:.6f}")
print(f"Validation P10 Coverage: {p10_coverage:.4f}")
print("보정 전 음수 예측 개수:", negative_prediction_count)

print("\nP10 예측값 요약")
print(
    pd.Series(
        p10_validation_pred,
        name="predicted_p10"
    ).describe()
)


# ------------------------------------------------------------
# 6. Validation 예측 결과 저장
# ------------------------------------------------------------

p10_validation_result_df = validation_lgb_df[
    [
        "forecast_datetime",
        "store_id",
        "product_id"
    ]
].copy()

p10_validation_result_df["actual_demand"] = (
    y_validation.to_numpy()
)

p10_validation_result_df["predicted_p10"] = (
    p10_validation_pred
)

display(
    p10_validation_result_df.head(10)
)


# ------------------------------------------------------------
# 7. 피처 중요도 확인
# ------------------------------------------------------------

p10_feature_importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_gain": (
        p10_model.booster_
        .feature_importance(importance_type="gain")
    )
}).sort_values(
    "importance_gain",
    ascending=False
).reset_index(drop=True)

print("\nP10 모델 피처 중요도 상위 20개")
display(
    p10_feature_importance_df.head(20)
)

Training until validation scores don't improve for 100 rounds
[100]	validation's quantile: 0.0334085
[200]	validation's quantile: 0.0333938
[300]	validation's quantile: 0.0333918
[400]	validation's quantile: 0.0333913
[500]	validation's quantile: 0.0333911
[600]	validation's quantile: 0.0333911
[700]	validation's quantile: 0.0333911
[800]	validation's quantile: 0.0333911
[900]	validation's quantile: 0.0333911
[1000]	validation's quantile: 0.0333911
Early stopping, best iteration is:
[944]	validation's quantile: 0.033391
Evaluated only: quantile
P10 모델 학습 완료
--------------------------------------------------
최적 반복 횟수: 944
Validation Pinball Loss: 0.033391
Validation P10 Coverage: 0.7947
보정 전 음수 예측 개수: 0

P10 예측값 요약
count   62,092.0000
mean         0.0010
std          0.0306
min          0.0000
25%          0.0000
50%          0.0000
75%          0.0000
max          1.0096
Name: predicted_p10, dtype: float64


,forecast_datetime,store_id,product_id,actual_demand,predicted_p10
0,2025-10-01 10:00:00,S01,P001,0.0000,0.0000
1,2025-10-01 11:00:00,S01,P001,0.0000,0.0000
2,2025-10-01 12:00:00,S01,P001,3.0000,0.0000
3,2025-10-01 13:00:00,S01,P001,0.0000,0.0000
4,2025-10-01 14:00:00,S01,P001,0.0000,0.0000
5,2025-10-01 15:00:00,S01,P001,0.0000,0.0000
6,2025-10-01 16:00:00,S01,P001,0.0000,0.0000
7,2025-10-01 17:00:00,S01,P001,0.0000,0.0000
8,2025-10-01 18:00:00,S01,P001,1.0000,0.0000
9,2025-10-01 19:00:00,S01,P001,0.0000,0.0000



P10 모델 피처 중요도 상위 20개


,feature,importance_gain
0,total_current_stock_qty,"2,143,294.8196"
1,total_physical_available_qty,"441,697.4170"
2,visitor_ratio,"303,258.9409"
3,sales_qty_roll_std_12h,"212,447.9559"
4,hour,"138,399.0736"
5,sales_qty_roll_mean_12h,"129,882.2417"
6,total_available_qty,"112,422.6075"
7,lot_count,"97,986.2542"
8,baseline_waste_rate,"80,593.7742"
9,product_id,"70,702.5080"


In [47]:
# ============================================================
# 4-2-1. P10 희소성 및 Zero Baseline 진단
# ============================================================

y_validation_array = np.asarray(y_validation)
p10_prediction_array = np.asarray(p10_validation_pred)

p10_strict_below_rate = np.mean(
    y_validation_array < p10_prediction_array
)

p10_at_or_below_rate = np.mean(
    y_validation_array <= p10_prediction_array
)

actual_zero_rate = np.mean(
    y_validation_array == 0
)

predicted_zero_rate = np.mean(
    np.isclose(p10_prediction_array, 0)
)

p10_equal_rate = np.mean(
    np.isclose(
        y_validation_array,
        p10_prediction_array
    )
)

# 항상 0을 예측하는 단순 기준선
zero_baseline_prediction = np.zeros_like(
    y_validation_array,
    dtype=float
)

zero_baseline_pinball_loss = mean_pinball_loss(
    y_validation_array,
    zero_baseline_prediction,
    alpha=0.10
)

if zero_baseline_pinball_loss > 0:
    p10_improvement_rate = (
        zero_baseline_pinball_loss - p10_pinball_loss
    ) / zero_baseline_pinball_loss * 100
else:
    p10_improvement_rate = np.nan


print("실제 판매량 0 비율:", round(actual_zero_rate, 4))
print("P10 예측값 0 비율:", round(predicted_zero_rate, 4))
print("실제값 < P10 비율:", round(p10_strict_below_rate, 4))
print("실제값 <= P10 비율:", round(p10_at_or_below_rate, 4))
print("실제값 = P10 비율:", round(p10_equal_rate, 4))

print("\nP10 Pinball Loss:", round(p10_pinball_loss, 6))
print(
    "Zero Baseline Pinball Loss:",
    round(zero_baseline_pinball_loss, 6)
)
print(
    "Zero Baseline 대비 개선율:",
    round(p10_improvement_rate, 2),
    "%"
)

print(
    "\n이산형 분위수 조건 충족 여부:",
    p10_strict_below_rate <= 0.10 <= p10_at_or_below_rate
)

실제 판매량 0 비율: 0.7946
P10 예측값 0 비율: 0.9988
실제값 < P10 비율: 0.0001
실제값 <= P10 비율: 0.7947
실제값 = P10 비율: 0.7946

P10 Pinball Loss: 0.033391
Zero Baseline Pinball Loss: 0.033489
Zero Baseline 대비 개선율: 0.29 %

이산형 분위수 조건 충족 여부: True


In [48]:
# # ============================================================
# # 4-2-2. P10 학습 후 임시 메모리 정리
# # ============================================================

# import gc

# temporary_variables = [
#     "p10_validation_pred_raw",
#     "y_validation_array",
#     "p10_prediction_array",
#     "zero_baseline_prediction"
# ]

# for variable_name in temporary_variables:
#     if variable_name in globals():
#         del globals()[variable_name]

# gc.collect()

# print("임시 메모리 정리 완료")
# print("P10 모델 유지 여부:", "p10_model" in globals())
# print(
#     "P10 예측 결과 유지 여부:",
#     "p10_validation_result_df" in globals()
# )

임시 메모리 정리 완료
P10 모델 유지 여부: True
P10 예측 결과 유지 여부: True


In [49]:
# import psutil

# memory = psutil.virtual_memory()

# print(f"전체 RAM: {memory.total / 1024**3:.2f} GB")
# print(f"사용 RAM: {memory.used / 1024**3:.2f} GB")
# print(f"사용 가능 RAM: {memory.available / 1024**3:.2f} GB")
# print(f"RAM 사용률: {memory.percent:.1f}%")

전체 RAM: 12.67 GB
사용 RAM: 4.97 GB
사용 가능 RAM: 7.39 GB
RAM 사용률: 41.7%


In [50]:
# ============================================================
# 4-3. P50 LightGBM Quantile 모델
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_pinball_loss


# ------------------------------------------------------------
# 1. P50 모델 설정
# ------------------------------------------------------------

P50_ALPHA = 0.50

p50_model = lgb.LGBMRegressor(
    objective="quantile",
    alpha=P50_ALPHA,

    boosting_type="gbdt",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


# ------------------------------------------------------------
# 2. P50 모델 학습
# ------------------------------------------------------------

p50_model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_validation, y_validation)
    ],

    eval_metric="quantile",

    categorical_feature=CATEGORICAL_COLS,

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),

        lgb.log_evaluation(
            period=100
        )
    ]
)


# ------------------------------------------------------------
# 3. Validation 데이터 P50 예측
# ------------------------------------------------------------

p50_validation_pred_raw = p50_model.predict(
    X_validation,
    num_iteration=p50_model.best_iteration_
)

# 클리핑하기 전에 음수 예측 개수 확인
p50_negative_prediction_count_raw = int(
    np.sum(p50_validation_pred_raw < 0)
)

# 판매량은 음수가 될 수 없으므로 0 미만을 0으로 변환
p50_validation_pred = np.clip(
    p50_validation_pred_raw,
    a_min=0,
    a_max=None
)

# Series 또는 배열 모두 처리 가능하도록 변환
y_validation_array = np.asarray(
    y_validation
).reshape(-1)


# ------------------------------------------------------------
# 4. P50 기본 성능 평가
# ------------------------------------------------------------

p50_pinball_loss = mean_pinball_loss(
    y_validation_array,
    p50_validation_pred,
    alpha=P50_ALPHA
)

p50_coverage = np.mean(
    y_validation_array <= p50_validation_pred
)


print("P50 최적 반복 횟수:", p50_model.best_iteration_)

print(
    "P50 Pinball Loss:",
    f"{p50_pinball_loss:.6f}"
)

print(
    "P50 이하 실제값 비율 Coverage:",
    f"{p50_coverage:.4f}"
)

print(
    "클리핑 전 P50 음수 예측 개수:",
    p50_negative_prediction_count_raw
)


# ------------------------------------------------------------
# 5. P50 Coverage 상세 진단
# ------------------------------------------------------------

p50_coverage_lt = np.mean(
    y_validation_array < p50_validation_pred
)

p50_coverage_le = np.mean(
    y_validation_array <= p50_validation_pred
)

validation_actual_zero_ratio = np.mean(
    y_validation_array == 0
)

p50_prediction_zero_ratio = np.mean(
    p50_validation_pred == 0
)

p50_prediction_positive_ratio = np.mean(
    p50_validation_pred > 0
)

p50_quantile_condition = (
    p50_coverage_lt
    <= P50_ALPHA
    <= p50_coverage_le
)


print("\n[P50 Coverage 상세 진단]")

print(
    "실제값 < P50 비율:",
    f"{p50_coverage_lt:.4f}"
)

print(
    "실제값 <= P50 비율:",
    f"{p50_coverage_le:.4f}"
)

print(
    "Validation 실제 판매량 0 비율:",
    f"{validation_actual_zero_ratio:.4f}"
)

print(
    "P50 예측값 0 비율:",
    f"{p50_prediction_zero_ratio:.4f}"
)

print(
    "P50 예측값 양수 비율:",
    f"{p50_prediction_positive_ratio:.4f}"
)

print(
    "P50 분위수 조건 충족 여부:",
    p50_quantile_condition
)


# ------------------------------------------------------------
# 6. P50 Validation 예측 결과 테이블 생성
# ------------------------------------------------------------

p50_validation_result_df = (
    validation_lgb_df[
        [
            TIME_COL,
            "store_id",
            "product_id"
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

p50_validation_result_df["actual_demand"] = (
    y_validation_array
)

p50_validation_result_df["predicted_p50"] = (
    p50_validation_pred
)

# 양수: 실제값이 예측보다 큼
# 음수: 실제값이 예측보다 작음
p50_validation_result_df["prediction_error_p50"] = (
    p50_validation_result_df["actual_demand"]
    - p50_validation_result_df["predicted_p50"]
)


print(
    "\nP50 Validation 결과 행 수:",
    len(p50_validation_result_df)
)

display(
    p50_validation_result_df.head(10)
)


# ------------------------------------------------------------
# 7. P50 예측값 요약 통계
# ------------------------------------------------------------

p50_prediction_summary_df = pd.DataFrame({
    "actual_demand": y_validation_array,
    "predicted_p50": p50_validation_pred
}).describe().T


print("\nP50 실제값·예측값 요약 통계")

display(
    p50_prediction_summary_df
)


# ------------------------------------------------------------
# 8. P50 피처 중요도
# ------------------------------------------------------------

p50_feature_importance_df = pd.DataFrame({
    "feature": p50_model.feature_name_,
    "importance": p50_model.feature_importances_
})

p50_feature_importance_df = (
    p50_feature_importance_df
    .sort_values(
        by="importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nP50 주요 피처 중요도 상위 20개")

display(
    p50_feature_importance_df.head(20)
)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's quantile: 0.16661
[200]	valid_0's quantile: 0.166543
[300]	valid_0's quantile: 0.166482
[400]	valid_0's quantile: 0.166436
[500]	valid_0's quantile: 0.166397
[600]	valid_0's quantile: 0.166374
[700]	valid_0's quantile: 0.16636
[800]	valid_0's quantile: 0.16636
Early stopping, best iteration is:
[726]	valid_0's quantile: 0.16636
Evaluated only: quantile
P50 최적 반복 횟수: 726
P50 Pinball Loss: 0.166356
P50 이하 실제값 비율 Coverage: 0.7951
클리핑 전 P50 음수 예측 개수: 551

[P50 Coverage 상세 진단]
실제값 < P50 비율: 0.0557
실제값 <= P50 비율: 0.7951
Validation 실제 판매량 0 비율: 0.7946
P50 예측값 0 비율: 0.9011
P50 예측값 양수 비율: 0.0989
P50 분위수 조건 충족 여부: True

P50 Validation 결과 행 수: 62092


,forecast_datetime,store_id,product_id,actual_demand,predicted_p50,prediction_error_p50
0,2025-10-01 10:00:00,S01,P001,0.0000,0.0000,0.0000
1,2025-10-01 11:00:00,S01,P001,0.0000,0.0000,0.0000
2,2025-10-01 12:00:00,S01,P001,3.0000,0.0000,3.0000
3,2025-10-01 13:00:00,S01,P001,0.0000,0.0000,0.0000
4,2025-10-01 14:00:00,S01,P001,0.0000,0.0000,0.0000
5,2025-10-01 15:00:00,S01,P001,0.0000,0.0000,0.0000
6,2025-10-01 16:00:00,S01,P001,0.0000,0.0000,0.0000
7,2025-10-01 17:00:00,S01,P001,0.0000,0.0000,0.0000
8,2025-10-01 18:00:00,S01,P001,1.0000,0.0000,1.0000
9,2025-10-01 19:00:00,S01,P001,0.0000,0.0000,0.0000



P50 실제값·예측값 요약 통계


,count,mean,std,min,25%,50%,75%,max
actual_demand,"62,092.0000",0.3349,0.7756,0.0000,0.0000,0.0000,0.0000,8.0000
predicted_p50,"62,092.0000",0.0212,0.1298,0.0000,0.0000,0.0000,0.0000,1.9418



P50 주요 피처 중요도 상위 20개


,feature,importance
0,total_current_stock_qty,4142
1,product_id,2428
2,visitor_ratio,2339
3,sales_qty_roll_std_12h,2233
4,lot_count,1997
5,hour,1769
6,sales_qty_roll_mean_12h,1404
7,total_physical_available_qty,854
8,time_slot,635
9,sales_qty_roll_max_12h,610


In [51]:
# ============================================================
# 4-4. P90 LightGBM Quantile 모델
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_pinball_loss


# ------------------------------------------------------------
# 1. P90 모델 설정
# ------------------------------------------------------------

P90_ALPHA = 0.90

p90_model = lgb.LGBMRegressor(
    objective="quantile",
    alpha=P90_ALPHA,

    boosting_type="gbdt",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


# ------------------------------------------------------------
# 2. P90 모델 학습
# ------------------------------------------------------------

p90_model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_validation, y_validation)
    ],

    eval_metric="quantile",

    categorical_feature=CATEGORICAL_COLS,

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),

        lgb.log_evaluation(
            period=100
        )
    ]
)


# ------------------------------------------------------------
# 3. Validation 데이터 P90 예측
# ------------------------------------------------------------

p90_validation_pred_raw = p90_model.predict(
    X_validation,
    num_iteration=p90_model.best_iteration_
)

# 클리핑 전 음수 예측 개수
p90_negative_prediction_count_raw = int(
    np.sum(p90_validation_pred_raw < 0)
)

# 수요는 음수가 될 수 없으므로 0 미만을 0으로 변환
p90_validation_pred = np.clip(
    p90_validation_pred_raw,
    a_min=0,
    a_max=None
)

# 실제값을 NumPy 배열로 변환
y_validation_array = y_validation.to_numpy()


# ------------------------------------------------------------
# 4. P90 기본 성능 평가
# ------------------------------------------------------------

p90_pinball_loss = mean_pinball_loss(
    y_validation_array,
    p90_validation_pred,
    alpha=P90_ALPHA
)

p90_coverage = np.mean(
    y_validation_array <= p90_validation_pred
)


print("P90 최적 반복 횟수:", p90_model.best_iteration_)

print(
    "P90 Pinball Loss:",
    f"{p90_pinball_loss:.6f}"
)

print(
    "P90 이하 실제값 비율 Coverage:",
    f"{p90_coverage:.4f}"
)

print(
    "클리핑 전 P90 음수 예측 개수:",
    p90_negative_prediction_count_raw
)


# ------------------------------------------------------------
# 5. P90 Coverage 상세 진단
# ------------------------------------------------------------

p90_coverage_lt = np.mean(
    y_validation_array < p90_validation_pred
)

p90_coverage_le = np.mean(
    y_validation_array <= p90_validation_pred
)

validation_actual_zero_ratio = np.mean(
    y_validation_array == 0
)

p90_prediction_zero_ratio = np.mean(
    p90_validation_pred == 0
)

p90_prediction_positive_ratio = np.mean(
    p90_validation_pred > 0
)

p90_quantile_condition = (
    p90_coverage_lt
    <= P90_ALPHA
    <= p90_coverage_le
)


print("\n[P90 Coverage 상세 진단]")

print(
    "실제값 < P90 비율:",
    f"{p90_coverage_lt:.4f}"
)

print(
    "실제값 <= P90 비율:",
    f"{p90_coverage_le:.4f}"
)

print(
    "Validation 실제 판매량 0 비율:",
    f"{validation_actual_zero_ratio:.4f}"
)

print(
    "P90 예측값 0 비율:",
    f"{p90_prediction_zero_ratio:.4f}"
)

print(
    "P90 예측값 양수 비율:",
    f"{p90_prediction_positive_ratio:.4f}"
)

print(
    "P90 분위수 조건 충족 여부:",
    p90_quantile_condition
)


# ------------------------------------------------------------
# 6. P90 Validation 예측 결과 테이블
# ------------------------------------------------------------

p90_validation_result_df = validation_lgb_df[
    [
        TIME_COL,
        "store_id",
        "product_id"
    ]
].copy()

p90_validation_result_df["actual_demand"] = (
    y_validation_array
)

p90_validation_result_df["predicted_p90"] = (
    p90_validation_pred
)

p90_validation_result_df[
    "prediction_error_p90"
] = (
    p90_validation_result_df["actual_demand"]
    - p90_validation_result_df["predicted_p90"]
)


print(
    "\nP90 Validation 결과 행 수:",
    len(p90_validation_result_df)
)

display(
    p90_validation_result_df.head(10)
)


# ------------------------------------------------------------
# 7. P90 피처 중요도
# ------------------------------------------------------------

p90_feature_importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": p90_model.feature_importances_
})

p90_feature_importance_df = (
    p90_feature_importance_df
    .sort_values(
        by="importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nP90 주요 피처 중요도")

display(
    p90_feature_importance_df.head(20)
)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's quantile: 0.160964
[200]	valid_0's quantile: 0.158934
[300]	valid_0's quantile: 0.158561
[400]	valid_0's quantile: 0.158441
[500]	valid_0's quantile: 0.158337
[600]	valid_0's quantile: 0.158252
[700]	valid_0's quantile: 0.158203
[800]	valid_0's quantile: 0.158189
[900]	valid_0's quantile: 0.158182
[1000]	valid_0's quantile: 0.158186
Early stopping, best iteration is:
[910]	valid_0's quantile: 0.158181
Evaluated only: quantile
P90 최적 반복 횟수: 910
P90 Pinball Loss: 0.158119
P90 이하 실제값 비율 Coverage: 0.9075
클리핑 전 P90 음수 예측 개수: 631

[P90 Coverage 상세 진단]
실제값 < P90 비율: 0.8974
실제값 <= P90 비율: 0.9075
Validation 실제 판매량 0 비율: 0.7946
P90 예측값 0 비율: 0.0102
P90 예측값 양수 비율: 0.9898
P90 분위수 조건 충족 여부: True

P90 Validation 결과 행 수: 62092


,forecast_datetime,store_id,product_id,actual_demand,predicted_p90,prediction_error_p90
0,2025-10-01 10:00:00,S01,P001,0.0000,1.0406,-1.0406
1,2025-10-01 11:00:00,S01,P001,0.0000,1.0273,-1.0273
2,2025-10-01 12:00:00,S01,P001,3.0000,1.0264,1.9736
3,2025-10-01 13:00:00,S01,P001,0.0000,0.8801,-0.8801
4,2025-10-01 14:00:00,S01,P001,0.0000,0.8748,-0.8748
5,2025-10-01 15:00:00,S01,P001,0.0000,0.9041,-0.9041
6,2025-10-01 16:00:00,S01,P001,0.0000,0.9148,-0.9148
7,2025-10-01 17:00:00,S01,P001,0.0000,0.9204,-0.9204
8,2025-10-01 18:00:00,S01,P001,1.0000,1.1018,-0.1018
9,2025-10-01 19:00:00,S01,P001,0.0000,1.0567,-1.0567



P90 주요 피처 중요도


,feature,importance
0,product_id,3765
1,total_current_stock_qty,2570
2,sales_qty_roll_mean_168h,1872
3,sales_qty_roll_mean_12h,1786
4,sales_qty_roll_std_12h,1513
5,mean_freshness_score,1402
6,visitor_ratio,1055
7,hours_until_close,863
8,sales_qty_roll_std_168h,711
9,subcategory,709


In [52]:
# ============================================================
# 4-4-1. P10·P50·P90 통합 최종 점검
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import mean_pinball_loss


# ------------------------------------------------------------
# 1. 실제값·분위수 예측값 배열 통일
# ------------------------------------------------------------

y_validation_array = np.asarray(
    y_validation
).reshape(-1)

p10_validation_pred_array = np.asarray(
    p10_validation_pred
).reshape(-1)

p50_validation_pred_array = np.asarray(
    p50_validation_pred
).reshape(-1)

p90_validation_pred_array = np.asarray(
    p90_validation_pred
).reshape(-1)


# 행 수가 서로 같은지 확인
validation_length = len(y_validation_array)

assert len(p10_validation_pred_array) == validation_length, (
    "P10 예측값과 Validation 실제값의 행 수가 다릅니다."
)

assert len(p50_validation_pred_array) == validation_length, (
    "P50 예측값과 Validation 실제값의 행 수가 다릅니다."
)

assert len(p90_validation_pred_array) == validation_length, (
    "P90 예측값과 Validation 실제값의 행 수가 다릅니다."
)

assert len(validation_lgb_df) == validation_length, (
    "validation_lgb_df와 Validation 실제값의 행 수가 다릅니다."
)


print("Validation 전체 행 수:", validation_length)


# ------------------------------------------------------------
# 2. P10·P50·P90 통합 결과 테이블 생성
# ------------------------------------------------------------

quantile_validation_result_df = (
    validation_lgb_df[
        [
            TIME_COL,
            "store_id",
            "product_id"
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

quantile_validation_result_df["actual_demand"] = (
    y_validation_array
)

quantile_validation_result_df["predicted_p10"] = (
    p10_validation_pred_array
)

quantile_validation_result_df["predicted_p50"] = (
    p50_validation_pred_array
)

quantile_validation_result_df["predicted_p90"] = (
    p90_validation_pred_array
)


# ------------------------------------------------------------
# 3. 분위수 교차 여부 확인
# ------------------------------------------------------------

# P10이 P50보다 큰 경우
quantile_validation_result_df["cross_p10_p50"] = (
    quantile_validation_result_df["predicted_p10"]
    >
    quantile_validation_result_df["predicted_p50"]
)

# P50이 P90보다 큰 경우
quantile_validation_result_df["cross_p50_p90"] = (
    quantile_validation_result_df["predicted_p50"]
    >
    quantile_validation_result_df["predicted_p90"]
)

# P10이 P90보다 큰 심한 교차
quantile_validation_result_df["cross_p10_p90"] = (
    quantile_validation_result_df["predicted_p10"]
    >
    quantile_validation_result_df["predicted_p90"]
)

# 하나라도 교차가 발생한 경우
quantile_validation_result_df["cross_any"] = (
    quantile_validation_result_df[
        [
            "cross_p10_p50",
            "cross_p50_p90",
            "cross_p10_p90"
        ]
    ]
    .any(axis=1)
)


p10_p50_cross_count = int(
    quantile_validation_result_df["cross_p10_p50"].sum()
)

p50_p90_cross_count = int(
    quantile_validation_result_df["cross_p50_p90"].sum()
)

p10_p90_cross_count = int(
    quantile_validation_result_df["cross_p10_p90"].sum()
)

any_cross_count = int(
    quantile_validation_result_df["cross_any"].sum()
)


p10_p50_cross_rate = (
    p10_p50_cross_count
    / validation_length
)

p50_p90_cross_rate = (
    p50_p90_cross_count
    / validation_length
)

p10_p90_cross_rate = (
    p10_p90_cross_count
    / validation_length
)

any_cross_rate = (
    any_cross_count
    / validation_length
)


print("\n[분위수 교차 점검]")

print(
    "P10 > P50 교차:",
    f"{p10_p50_cross_count:,}건",
    f"({p10_p50_cross_rate:.4%})"
)

print(
    "P50 > P90 교차:",
    f"{p50_p90_cross_count:,}건",
    f"({p50_p90_cross_rate:.4%})"
)

print(
    "P10 > P90 교차:",
    f"{p10_p90_cross_count:,}건",
    f"({p10_p90_cross_rate:.4%})"
)

print(
    "하나 이상의 분위수 교차:",
    f"{any_cross_count:,}건",
    f"({any_cross_rate:.4%})"
)

print(
    "전체 분위수 순서 정상 여부:",
    any_cross_count == 0
)


# ------------------------------------------------------------
# 4. 분위수별 Coverage와 분위수 조건 점검
# ------------------------------------------------------------

quantile_settings = {
    "P10": {
        "alpha": 0.10,
        "prediction": p10_validation_pred_array
    },
    "P50": {
        "alpha": 0.50,
        "prediction": p50_validation_pred_array
    },
    "P90": {
        "alpha": 0.90,
        "prediction": p90_validation_pred_array
    }
}

quantile_diagnostic_rows = []

for quantile_name, setting in quantile_settings.items():

    alpha = setting["alpha"]
    prediction = setting["prediction"]

    coverage_lt = np.mean(
        y_validation_array < prediction
    )

    coverage_le = np.mean(
        y_validation_array <= prediction
    )

    pinball_loss = mean_pinball_loss(
        y_validation_array,
        prediction,
        alpha=alpha
    )

    zero_prediction_ratio = np.mean(
        prediction == 0
    )

    positive_prediction_ratio = np.mean(
        prediction > 0
    )

    quantile_condition = (
        coverage_lt
        <= alpha
        <= coverage_le
    )

    quantile_diagnostic_rows.append({
        "quantile": quantile_name,
        "alpha": alpha,
        "pinball_loss": pinball_loss,
        "actual_lt_prediction": coverage_lt,
        "actual_le_prediction": coverage_le,
        "target_coverage": alpha,
        "zero_prediction_ratio": zero_prediction_ratio,
        "positive_prediction_ratio": positive_prediction_ratio,
        "quantile_condition": quantile_condition
    })


quantile_diagnostic_df = pd.DataFrame(
    quantile_diagnostic_rows
)


print("\n[분위수별 통합 진단]")

display(
    quantile_diagnostic_df
)


# ------------------------------------------------------------
# 5. P10~P90 예측구간 점검
# ------------------------------------------------------------

# P10과 P90 사이에 실제값이 포함되었는지
inside_p10_p90_interval = (
    (y_validation_array >= p10_validation_pred_array)
    &
    (y_validation_array <= p90_validation_pred_array)
)

# 실제값이 P10보다 낮은 경우
below_p10 = (
    y_validation_array < p10_validation_pred_array
)

# 실제값이 P90보다 높은 경우
above_p90 = (
    y_validation_array > p90_validation_pred_array
)


p10_p90_interval_coverage = np.mean(
    inside_p10_p90_interval
)

below_p10_rate = np.mean(
    below_p10
)

above_p90_rate = np.mean(
    above_p90
)

p10_p90_interval_width = (
    p90_validation_pred_array
    - p10_validation_pred_array
)


print("\n[P10~P90 예측구간 진단]")

print(
    "P10~P90 구간 목표 Coverage:",
    "약 0.8000"
)

print(
    "P10~P90 실제 Coverage:",
    f"{p10_p90_interval_coverage:.4f}"
)

print(
    "실제값이 P10보다 낮은 비율:",
    f"{below_p10_rate:.4f}"
)

print(
    "실제값이 P90보다 높은 비율:",
    f"{above_p90_rate:.4f}"
)

print(
    "P10~P90 평균 구간 너비:",
    f"{np.mean(p10_p90_interval_width):.6f}"
)

print(
    "P10~P90 중앙 구간 너비:",
    f"{np.median(p10_p90_interval_width):.6f}"
)


# ------------------------------------------------------------
# 6. 분위수 교차 보정
#    각 행의 P10·P50·P90 예측값을 작은 순서로 정렬
# ------------------------------------------------------------

quantile_prediction_matrix = np.column_stack([
    p10_validation_pred_array,
    p50_validation_pred_array,
    p90_validation_pred_array
])

quantile_prediction_sorted = np.sort(
    quantile_prediction_matrix,
    axis=1
)


# 4-5에서 참고할 최종 Validation 분위수 예측값
p10_validation_pred_final = (
    quantile_prediction_sorted[:, 0]
)

p50_validation_pred_final = (
    quantile_prediction_sorted[:, 1]
)

p90_validation_pred_final = (
    quantile_prediction_sorted[:, 2]
)


quantile_validation_result_df[
    "predicted_p10_final"
] = p10_validation_pred_final

quantile_validation_result_df[
    "predicted_p50_final"
] = p50_validation_pred_final

quantile_validation_result_df[
    "predicted_p90_final"
] = p90_validation_pred_final


# 보정 후 교차 여부
cross_after_correction = (
    (p10_validation_pred_final > p50_validation_pred_final)
    |
    (p50_validation_pred_final > p90_validation_pred_final)
)

cross_after_correction_count = int(
    np.sum(cross_after_correction)
)


print("\n[분위수 교차 보정 결과]")

print(
    "보정 전 교차 행 수:",
    f"{any_cross_count:,}"
)

print(
    "보정 후 교차 행 수:",
    f"{cross_after_correction_count:,}"
)

print(
    "보정 후 순서 정상 여부:",
    cross_after_correction_count == 0
)


# ------------------------------------------------------------
# 7. 교차 보정 전후 Pinball Loss 비교
# ------------------------------------------------------------

pinball_comparison_rows = []

for quantile_name, alpha, before_pred, after_pred in [
    (
        "P10",
        0.10,
        p10_validation_pred_array,
        p10_validation_pred_final
    ),
    (
        "P50",
        0.50,
        p50_validation_pred_array,
        p50_validation_pred_final
    ),
    (
        "P90",
        0.90,
        p90_validation_pred_array,
        p90_validation_pred_final
    )
]:

    loss_before = mean_pinball_loss(
        y_validation_array,
        before_pred,
        alpha=alpha
    )

    loss_after = mean_pinball_loss(
        y_validation_array,
        after_pred,
        alpha=alpha
    )

    pinball_comparison_rows.append({
        "quantile": quantile_name,
        "pinball_loss_before": loss_before,
        "pinball_loss_after": loss_after,
        "loss_change": loss_after - loss_before
    })


pinball_comparison_df = pd.DataFrame(
    pinball_comparison_rows
)


print("\n[교차 보정 전후 Pinball Loss]")

display(
    pinball_comparison_df
)


# ------------------------------------------------------------
# 8. 보정 후 Coverage 재확인
# ------------------------------------------------------------

final_quantile_settings = {
    "P10": {
        "alpha": 0.10,
        "prediction": p10_validation_pred_final
    },
    "P50": {
        "alpha": 0.50,
        "prediction": p50_validation_pred_final
    },
    "P90": {
        "alpha": 0.90,
        "prediction": p90_validation_pred_final
    }
}

final_diagnostic_rows = []

for quantile_name, setting in final_quantile_settings.items():

    alpha = setting["alpha"]
    prediction = setting["prediction"]

    coverage_lt = np.mean(
        y_validation_array < prediction
    )

    coverage_le = np.mean(
        y_validation_array <= prediction
    )

    final_diagnostic_rows.append({
        "quantile": quantile_name,
        "actual_lt_prediction": coverage_lt,
        "target_coverage": alpha,
        "actual_le_prediction": coverage_le,
        "quantile_condition": (
            coverage_lt
            <= alpha
            <= coverage_le
        )
    })


final_quantile_diagnostic_df = pd.DataFrame(
    final_diagnostic_rows
)


print("\n[보정 후 분위수 조건 재확인]")

display(
    final_quantile_diagnostic_df
)


# ------------------------------------------------------------
# 9. 분위수 교차 사례 확인
# ------------------------------------------------------------

if any_cross_count > 0:

    print("\n[분위수 교차 발생 사례 상위 20개]")

    display(
        quantile_validation_result_df.loc[
            quantile_validation_result_df["cross_any"],
            [
                TIME_COL,
                "store_id",
                "product_id",
                "actual_demand",
                "predicted_p10",
                "predicted_p50",
                "predicted_p90",
                "predicted_p10_final",
                "predicted_p50_final",
                "predicted_p90_final"
            ]
        ].head(20)
    )

else:

    print("\n분위수 교차 사례가 없습니다.")


# ------------------------------------------------------------
# 10. 최종 통합 결과 확인
# ------------------------------------------------------------

print("\n[P10·P50·P90 최종 통합 결과 상위 10개]")

display(
    quantile_validation_result_df[
        [
            TIME_COL,
            "store_id",
            "product_id",
            "actual_demand",
            "predicted_p10_final",
            "predicted_p50_final",
            "predicted_p90_final",
            "cross_any"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 11. 최종 판정
# ------------------------------------------------------------

all_quantile_conditions_true = bool(
    final_quantile_diagnostic_df[
        "quantile_condition"
    ].all()
)

print("\n[4-4-1 최종 판정]")

print(
    "보정 후 P10 <= P50 <= P90:",
    cross_after_correction_count == 0
)

print(
    "보정 후 모든 분위수 조건 충족:",
    all_quantile_conditions_true
)

print(
    "4-5 진행 가능 여부:",
    (
        cross_after_correction_count == 0
        and all_quantile_conditions_true
    )
)

Validation 전체 행 수: 62092

[분위수 교차 점검]
P10 > P50 교차: 0건 (0.0000%)
P50 > P90 교차: 0건 (0.0000%)
P10 > P90 교차: 0건 (0.0000%)
하나 이상의 분위수 교차: 0건 (0.0000%)
전체 분위수 순서 정상 여부: True

[분위수별 통합 진단]


,quantile,alpha,pinball_loss,actual_lt_prediction,actual_le_prediction,target_coverage,zero_prediction_ratio,positive_prediction_ratio,quantile_condition
0,P10,0.1000,0.0334,0.0001,0.7947,0.1000,0.9988,0.0012,True
1,P50,0.5000,0.1664,0.0557,0.7951,0.5000,0.9011,0.0989,True
2,P90,0.9000,0.1581,0.8974,0.9075,0.9000,0.0102,0.9898,True



[P10~P90 예측구간 진단]
P10~P90 구간 목표 Coverage: 약 0.8000
P10~P90 실제 Coverage: 0.9074
실제값이 P10보다 낮은 비율: 0.0001
실제값이 P90보다 높은 비율: 0.0925
P10~P90 평균 구간 너비: 1.201301
P10~P90 중앙 구간 너비: 1.055352

[분위수 교차 보정 결과]
보정 전 교차 행 수: 0
보정 후 교차 행 수: 0
보정 후 순서 정상 여부: True

[교차 보정 전후 Pinball Loss]


,quantile,pinball_loss_before,pinball_loss_after,loss_change
0,P10,0.0334,0.0334,0.0000
1,P50,0.1664,0.1664,0.0000
2,P90,0.1581,0.1581,0.0000



[보정 후 분위수 조건 재확인]


,quantile,actual_lt_prediction,target_coverage,actual_le_prediction,quantile_condition
0,P10,0.0001,0.1000,0.7947,True
1,P50,0.0557,0.5000,0.7951,True
2,P90,0.8974,0.9000,0.9075,True



분위수 교차 사례가 없습니다.

[P10·P50·P90 최종 통합 결과 상위 10개]


,forecast_datetime,store_id,product_id,actual_demand,predicted_p10_final,predicted_p50_final,predicted_p90_final,cross_any
0,2025-10-01 10:00:00,S01,P001,0.0000,0.0000,0.0000,1.0406,False
1,2025-10-01 11:00:00,S01,P001,0.0000,0.0000,0.0000,1.0273,False
2,2025-10-01 12:00:00,S01,P001,3.0000,0.0000,0.0000,1.0264,False
3,2025-10-01 13:00:00,S01,P001,0.0000,0.0000,0.0000,0.8801,False
4,2025-10-01 14:00:00,S01,P001,0.0000,0.0000,0.0000,0.8748,False
5,2025-10-01 15:00:00,S01,P001,0.0000,0.0000,0.0000,0.9041,False
6,2025-10-01 16:00:00,S01,P001,0.0000,0.0000,0.0000,0.9148,False
7,2025-10-01 17:00:00,S01,P001,0.0000,0.0000,0.0000,0.9204,False
8,2025-10-01 18:00:00,S01,P001,1.0000,0.0000,0.0000,1.1018,False
9,2025-10-01 19:00:00,S01,P001,0.0000,0.0000,0.0000,1.0567,False



[4-4-1 최종 판정]
보정 후 P10 <= P50 <= P90: True
보정 후 모든 분위수 조건 충족: True
4-5 진행 가능 여부: True


In [53]:
# ============================================================
# 4-5. 정가 조건 기본수요 추론
# ============================================================
# 목적
# - 학습 완료된 P10·P50·P90 LightGBM Quantile 모델을 이용해
#   Test 기간의 정가 기준 기본수요를 추론
#
# 최종 생성 컬럼
# - base_demand_p10
# - base_demand_p50
# - base_demand_p90
#
# 현재 LightGBM 입력 피처에서는
# discount / promotion / markdown 관련 변수를 이미 제외했으므로
# X_test 자체를 정가 조건 입력으로 사용
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 정가 조건 입력 피처 확인
# ------------------------------------------------------------

regular_condition_keywords = [
    "discount",
    "promotion",
    "markdown",
    "sale_price",
    "discount_rate",
    "price_ratio"
]

regular_condition_excluded_features = [
    col
    for col in FEATURE_COLS
    if any(
        keyword in col.lower()
        for keyword in regular_condition_keywords
    )
]

print("정가 조건에서 제외되어야 하는 할인·프로모션 피처:")
print(regular_condition_excluded_features)

if len(regular_condition_excluded_features) == 0:
    print("정가 조건 입력 확인 완료")
    print("할인·프로모션 관련 피처가 LightGBM 입력에 포함되어 있지 않습니다.")
else:
    print("주의: 정가 조건과 관련된 피처가 입력에 포함되어 있습니다.")
    print("해당 피처가 실제로 필요한 피처인지 다시 확인해야 합니다.")


# ------------------------------------------------------------
# 2. Test 데이터 정가 조건 입력 생성
# ------------------------------------------------------------

# 4-1에서 이미 생성한 X_test 사용
X_regular_price_test = X_test.copy()

print()
print("정가 조건 추론 대상 행 수:", len(X_regular_price_test))
print("정가 조건 입력 피처 수:", X_regular_price_test.shape[1])


# ------------------------------------------------------------
# 3. P10·P50·P90 기본수요 추론
# ------------------------------------------------------------

base_demand_p10_raw = p10_model.predict(
    X_regular_price_test
)

base_demand_p50_raw = p50_model.predict(
    X_regular_price_test
)

base_demand_p90_raw = p90_model.predict(
    X_regular_price_test
)


# ------------------------------------------------------------
# 4. 음수 수요 예측값 보정
# ------------------------------------------------------------
# 수요량은 0보다 작을 수 없으므로 음수값을 0으로 보정
# 소수점은 기대수요이므로 반올림하지 않음

base_demand_p10_nonnegative = np.clip(
    base_demand_p10_raw,
    a_min=0,
    a_max=None
)

base_demand_p50_nonnegative = np.clip(
    base_demand_p50_raw,
    a_min=0,
    a_max=None
)

base_demand_p90_nonnegative = np.clip(
    base_demand_p90_raw,
    a_min=0,
    a_max=None
)


print()
print("음수 예측값 개수")

print(
    "P10:",
    int((base_demand_p10_raw < 0).sum())
)

print(
    "P50:",
    int((base_demand_p50_raw < 0).sum())
)

print(
    "P90:",
    int((base_demand_p90_raw < 0).sum())
)


# ------------------------------------------------------------
# 5. 분위수 교차 여부 확인
# ------------------------------------------------------------

quantile_crossing_before = (
    (base_demand_p10_nonnegative > base_demand_p50_nonnegative)
    |
    (base_demand_p50_nonnegative > base_demand_p90_nonnegative)
)

quantile_crossing_before_count = int(
    quantile_crossing_before.sum()
)

print()
print("분위수 순서 보정 전 교차 행 수:",
      quantile_crossing_before_count)

print(
    "분위수 순서 보정 전 교차 비율:",
    round(
        quantile_crossing_before_count
        / len(X_regular_price_test) * 100,
        4
    ),
    "%"
)


# ------------------------------------------------------------
# 6. 분위수 순서 예방 보정
# ------------------------------------------------------------
# 각 행의 P10·P50·P90을 오름차순으로 정렬
#
# 교차가 없다면 기존 값이 그대로 유지됨
# 교차가 있다면 P10 <= P50 <= P90이 되도록 보정됨

base_demand_matrix = np.column_stack([
    base_demand_p10_nonnegative,
    base_demand_p50_nonnegative,
    base_demand_p90_nonnegative
])

base_demand_matrix_sorted = np.sort(
    base_demand_matrix,
    axis=1
)

base_demand_p10 = base_demand_matrix_sorted[:, 0]
base_demand_p50 = base_demand_matrix_sorted[:, 1]
base_demand_p90 = base_demand_matrix_sorted[:, 2]


# ------------------------------------------------------------
# 7. 보정 후 분위수 순서 최종 확인
# ------------------------------------------------------------

quantile_crossing_after = (
    (base_demand_p10 > base_demand_p50)
    |
    (base_demand_p50 > base_demand_p90)
)

quantile_crossing_after_count = int(
    quantile_crossing_after.sum()
)

print()
print("분위수 순서 보정 후 교차 행 수:",
      quantile_crossing_after_count)

print(
    "최종 분위수 순서:",
    "P10 <= P50 <= P90"
    if quantile_crossing_after_count == 0
    else "분위수 순서 이상 있음"
)


# ------------------------------------------------------------
# 8. 정가 조건 기본수요 결과 테이블 생성
# ------------------------------------------------------------

output_key_candidates = [
    TIME_COL,
    "date",
    "hour",
    "store_id",
    "product_id",
    "series_id",
    "category",
    "subcategory",
    "time_slot"
]

# 중복 컬럼명을 제거하면서 기존 순서 유지
output_key_candidates = list(
    dict.fromkeys(output_key_candidates)
)

# 실제 test_lgb_df에 존재하는 컬럼만 선택
OUTPUT_KEY_COLS = [
    col
    for col in output_key_candidates
    if col in test_lgb_df.columns
]

a1_base_demand_test_result_df = (
    test_lgb_df[OUTPUT_KEY_COLS]
    .copy()
    .reset_index(drop=True)
)

# 실제 수요량
a1_base_demand_test_result_df["actual_demand"] = (
    y_test
    .to_numpy()
)

# 정가 조건 기본수요 분위수
a1_base_demand_test_result_df["base_demand_p10"] = (
    base_demand_p10
)

a1_base_demand_test_result_df["base_demand_p50"] = (
    base_demand_p50
)

a1_base_demand_test_result_df["base_demand_p90"] = (
    base_demand_p90
)


# ------------------------------------------------------------
# 9. A-1 최종 출력용 테이블 생성
# ------------------------------------------------------------
# 실제 수요량은 모델 평가용이므로 제외하고,
# B-2 시뮬레이션 등에 전달할 기본수요 출력만 별도 생성

a1_base_demand_output_df = (
    a1_base_demand_test_result_df[
        OUTPUT_KEY_COLS
        + [
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 10. 최종 결과 무결성 점검
# ------------------------------------------------------------

base_demand_columns = [
    "base_demand_p10",
    "base_demand_p50",
    "base_demand_p90"
]

missing_value_count = (
    a1_base_demand_output_df[
        base_demand_columns
    ]
    .isna()
    .sum()
)

negative_value_count = (
    a1_base_demand_output_df[
        base_demand_columns
    ]
    .lt(0)
    .sum()
)

final_crossing_count = int(
    (
        (
            a1_base_demand_output_df["base_demand_p10"]
            >
            a1_base_demand_output_df["base_demand_p50"]
        )
        |
        (
            a1_base_demand_output_df["base_demand_p50"]
            >
            a1_base_demand_output_df["base_demand_p90"]
        )
    )
    .sum()
)


print()
print("=" * 60)
print("4-5. 정가 조건 기본수요 추론 결과")
print("=" * 60)

print("최종 출력 행 수:",
      len(a1_base_demand_output_df))

print("예측 시계열 수:",
      a1_base_demand_output_df["series_id"].nunique()
      if "series_id" in a1_base_demand_output_df.columns
      else "series_id 컬럼 없음")

print()
print("결측값 개수")
print(missing_value_count)

print()
print("음수값 개수")
print(negative_value_count)

print()
print("최종 분위수 교차 행 수:",
      final_crossing_count)


# ------------------------------------------------------------
# 11. 기본수요 예측 분포 확인
# ------------------------------------------------------------

print()
print("정가 조건 기본수요 예측 분포")

display(
    a1_base_demand_output_df[
        base_demand_columns
    ]
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# 12. 결과 일부 확인
# ------------------------------------------------------------

display(
    a1_base_demand_test_result_df.head(20)
)


# ------------------------------------------------------------
# 13. 최종 완료 판정
# ------------------------------------------------------------

if (
    missing_value_count.sum() == 0
    and negative_value_count.sum() == 0
    and final_crossing_count == 0
):
    print()
    print("4-5 정가 조건 기본수요 추론 완료")
    print("base_demand_p10, base_demand_p50, base_demand_p90 생성 완료")
else:
    print()
    print("4-5 결과에 이상이 있습니다.")
    print("결측값·음수값·분위수 교차 결과를 확인하세요.")

정가 조건에서 제외되어야 하는 할인·프로모션 피처:
[]
정가 조건 입력 확인 완료
할인·프로모션 관련 피처가 LightGBM 입력에 포함되어 있지 않습니다.

정가 조건 추론 대상 행 수: 62092
정가 조건 입력 피처 수: 109

음수 예측값 개수
P10: 0
P50: 508
P90: 622

분위수 순서 보정 전 교차 행 수: 0
분위수 순서 보정 전 교차 비율: 0.0 %

분위수 순서 보정 후 교차 행 수: 0
최종 분위수 순서: P10 <= P50 <= P90

4-5. 정가 조건 기본수요 추론 결과
최종 출력 행 수: 62092
예측 시계열 수: 114

결측값 개수
base_demand_p10    0
base_demand_p50    0
base_demand_p90    0
dtype: int64

음수값 개수
base_demand_p10    0
base_demand_p50    0
base_demand_p90    0
dtype: int64

최종 분위수 교차 행 수: 0

정가 조건 기본수요 예측 분포


,base_demand_p10,base_demand_p50,base_demand_p90
count,"62,092.0000","62,092.0000","62,092.0000"
mean,0.0015,0.0251,1.2249
std,0.0382,0.1449,0.7537
min,0.0000,0.0000,0.0000
25%,0.0000,0.0000,0.8638
50%,0.0000,0.0000,1.0654
75%,0.0000,0.0000,1.7310
max,1.0096,1.9453,4.0899


,forecast_datetime,date,hour,store_id,product_id,series_id,category,subcategory,time_slot,actual_demand,base_demand_p10,base_demand_p50,base_demand_p90
0,2025-11-16 10:00:00,2025-11-16,10,S01,P001,S01_P001,produce,apple,morning,0.0000,0.0000,0.0000,1.3992
1,2025-11-16 11:00:00,2025-11-16,11,S01,P001,S01_P001,produce,apple,morning,0.0000,0.0000,0.0000,1.3976
2,2025-11-16 12:00:00,2025-11-16,12,S01,P001,S01_P001,produce,apple,lunch,0.0000,0.0000,0.0000,1.3229
3,2025-11-16 13:00:00,2025-11-16,13,S01,P001,S01_P001,produce,apple,lunch,0.0000,0.0000,0.0000,1.3104
4,2025-11-16 14:00:00,2025-11-16,14,S01,P001,S01_P001,produce,apple,lunch,0.0000,0.0000,0.0000,1.2990
5,2025-11-16 15:00:00,2025-11-16,15,S01,P001,S01_P001,produce,apple,afternoon,1.0000,0.0000,0.0000,2.0310
6,2025-11-16 16:00:00,2025-11-16,16,S01,P001,S01_P001,produce,apple,afternoon,0.0000,0.0000,0.0000,1.5354
7,2025-11-16 17:00:00,2025-11-16,17,S01,P001,S01_P001,produce,apple,afternoon,0.0000,0.0000,0.0000,1.5091
8,2025-11-16 18:00:00,2025-11-16,18,S01,P001,S01_P001,produce,apple,evening,0.0000,0.0000,0.0085,2.1199
9,2025-11-16 19:00:00,2025-11-16,19,S01,P001,S01_P001,produce,apple,evening,1.0000,0.0000,0.0945,2.2051



4-5 정가 조건 기본수요 추론 완료
base_demand_p10, base_demand_p50, base_demand_p90 생성 완료


In [54]:
# ============================================================
# 5-1. MAE·RMSE·WAPE 평가
# - 평가 데이터: Test
# - 점예측 대표값: P50
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# ------------------------------------------------------------
# 1. 평가 데이터 준비
# ------------------------------------------------------------

point_evaluation_df = (
    a1_base_demand_test_result_df[
        [
            "actual_demand",
            "base_demand_p50"
        ]
    ]
    .dropna()
    .copy()
)

y_true = point_evaluation_df["actual_demand"].to_numpy()
y_pred = point_evaluation_df["base_demand_p50"].to_numpy()


# ------------------------------------------------------------
# 2. MAE
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_true,
    y_pred
)


# ------------------------------------------------------------
# 3. RMSE
# ------------------------------------------------------------

rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)


# ------------------------------------------------------------
# 4. WAPE
# ------------------------------------------------------------
# WAPE = 전체 절대오차 합 / 전체 실제 수요량 합 × 100

actual_demand_sum = np.abs(y_true).sum()
absolute_error_sum = np.abs(y_true - y_pred).sum()

if actual_demand_sum > 0:
    wape = (
        absolute_error_sum
        / actual_demand_sum
        * 100
    )
else:
    wape = np.nan


# ------------------------------------------------------------
# 5. 평가 결과 정리
# ------------------------------------------------------------

a1_point_metric_result_df = pd.DataFrame(
    {
        "model": ["LightGBM Quantile P50"],
        "evaluation_data": ["Test"],
        "evaluation_rows": [len(point_evaluation_df)],
        "actual_demand_sum": [actual_demand_sum],
        "MAE": [mae],
        "RMSE": [rmse],
        "WAPE_percent": [wape]
    }
)


# ------------------------------------------------------------
# 6. 결과 출력
# ------------------------------------------------------------

print("=" * 60)
print("5-1. LightGBM P50 점예측 평가 결과")
print("=" * 60)

print(f"평가 행 수       : {len(point_evaluation_df):,}")
print(f"실제 수요량 합계 : {actual_demand_sum:,.4f}")
print(f"MAE              : {mae:,.4f}")
print(f"RMSE             : {rmse:,.4f}")

if np.isnan(wape):
    print("WAPE             : 계산 불가 (실제 수요량 합계가 0)")
else:
    print(f"WAPE             : {wape:,.2f}%")

print("=" * 60)

display(a1_point_metric_result_df)

5-1. LightGBM P50 점예측 평가 결과
평가 행 수       : 62,092
실제 수요량 합계 : 20,982.0000
MAE              : 0.3343
RMSE             : 0.8263
WAPE             : 98.94%


,model,evaluation_data,evaluation_rows,actual_demand_sum,MAE,RMSE,WAPE_percent
0,LightGBM Quantile P50,Test,62092,"20,982.0000",0.3343,0.8263,98.9390


In [55]:
# ============================================================
# 5-1-1. P50 점예측 결과 추가 진단
# - 0수요 비율
# - 실제 수요 발생 행 성능
# - 과대·과소 예측
# - 전체 0 예측 기준선과 비교
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# ------------------------------------------------------------
# 1. 평가 데이터 준비
# ------------------------------------------------------------

diagnostic_df = (
    a1_base_demand_test_result_df[
        [
            "actual_demand",
            "base_demand_p50"
        ]
    ]
    .dropna()
    .copy()
)

diagnostic_df["absolute_error"] = np.abs(
    diagnostic_df["actual_demand"]
    - diagnostic_df["base_demand_p50"]
)

diagnostic_df["error"] = (
    diagnostic_df["base_demand_p50"]
    - diagnostic_df["actual_demand"]
)


# ------------------------------------------------------------
# 2. 기본 현황
# ------------------------------------------------------------

total_rows = len(diagnostic_df)

zero_actual_rows = (
    diagnostic_df["actual_demand"] == 0
).sum()

positive_actual_rows = (
    diagnostic_df["actual_demand"] > 0
).sum()

zero_actual_rate = (
    zero_actual_rows
    / total_rows
    * 100
)

actual_sum = diagnostic_df["actual_demand"].sum()
prediction_sum = diagnostic_df["base_demand_p50"].sum()

mean_actual = diagnostic_df["actual_demand"].mean()
mean_prediction = diagnostic_df["base_demand_p50"].mean()

prediction_bias = (
    diagnostic_df["base_demand_p50"]
    - diagnostic_df["actual_demand"]
).mean()


# ------------------------------------------------------------
# 3. 전체 0 예측 기준선
# ------------------------------------------------------------

zero_prediction = np.zeros(total_rows)

zero_baseline_mae = mean_absolute_error(
    diagnostic_df["actual_demand"],
    zero_prediction
)

zero_baseline_rmse = np.sqrt(
    mean_squared_error(
        diagnostic_df["actual_demand"],
        zero_prediction
    )
)

zero_baseline_wape = 100.0 if actual_sum > 0 else np.nan


# ------------------------------------------------------------
# 4. 실제 수요가 발생한 행만 평가
# ------------------------------------------------------------

positive_demand_df = diagnostic_df[
    diagnostic_df["actual_demand"] > 0
].copy()

if len(positive_demand_df) > 0:

    positive_mae = mean_absolute_error(
        positive_demand_df["actual_demand"],
        positive_demand_df["base_demand_p50"]
    )

    positive_rmse = np.sqrt(
        mean_squared_error(
            positive_demand_df["actual_demand"],
            positive_demand_df["base_demand_p50"]
        )
    )

    positive_wape = (
        positive_demand_df["absolute_error"].sum()
        / positive_demand_df["actual_demand"].sum()
        * 100
    )

else:

    positive_mae = np.nan
    positive_rmse = np.nan
    positive_wape = np.nan


# ------------------------------------------------------------
# 5. 실제 수요가 0인 행의 예측 확인
# ------------------------------------------------------------

zero_demand_df = diagnostic_df[
    diagnostic_df["actual_demand"] == 0
].copy()

if len(zero_demand_df) > 0:

    zero_row_prediction_mean = (
        zero_demand_df["base_demand_p50"].mean()
    )

    zero_row_positive_prediction_rate = (
        (
            zero_demand_df["base_demand_p50"] > 0
        ).mean()
        * 100
    )

else:

    zero_row_prediction_mean = np.nan
    zero_row_positive_prediction_rate = np.nan


# ------------------------------------------------------------
# 6. 결과 정리
# ------------------------------------------------------------

diagnostic_result_df = pd.DataFrame(
    {
        "항목": [
            "전체 평가 행 수",
            "실제 수요 0인 행 수",
            "실제 수요 0 비율",
            "실제 수요 발생 행 수",
            "실제 수요량 합계",
            "P50 예측량 합계",
            "행당 평균 실제 수요",
            "행당 평균 P50 예측",
            "평균 예측 편향",
            "0 예측 기준선 MAE",
            "0 예측 기준선 RMSE",
            "0 예측 기준선 WAPE",
            "수요 발생 행 MAE",
            "수요 발생 행 RMSE",
            "수요 발생 행 WAPE",
            "0수요 행 평균 P50 예측",
            "0수요 행 양수 예측 비율"
        ],
        "결과": [
            total_rows,
            zero_actual_rows,
            zero_actual_rate,
            positive_actual_rows,
            actual_sum,
            prediction_sum,
            mean_actual,
            mean_prediction,
            prediction_bias,
            zero_baseline_mae,
            zero_baseline_rmse,
            zero_baseline_wape,
            positive_mae,
            positive_rmse,
            positive_wape,
            zero_row_prediction_mean,
            zero_row_positive_prediction_rate
        ]
    }
)


# ------------------------------------------------------------
# 7. 출력
# ------------------------------------------------------------

print("=" * 65)
print("5-1-1. P50 점예측 추가 진단")
print("=" * 65)

print(f"전체 평가 행 수            : {total_rows:,}")
print(f"실제 수요 0 비율           : {zero_actual_rate:,.2f}%")
print(f"실제 수요 발생 행 수       : {positive_actual_rows:,}")

print("-" * 65)

print(f"실제 수요량 합계           : {actual_sum:,.4f}")
print(f"P50 예측량 합계            : {prediction_sum:,.4f}")
print(f"평균 예측 편향             : {prediction_bias:,.4f}")

print("-" * 65)

print(f"전체 0 예측 기준선 MAE     : {zero_baseline_mae:,.4f}")
print(f"전체 0 예측 기준선 RMSE    : {zero_baseline_rmse:,.4f}")
print(f"전체 0 예측 기준선 WAPE    : {zero_baseline_wape:,.2f}%")

print("-" * 65)

print(f"수요 발생 행 MAE           : {positive_mae:,.4f}")
print(f"수요 발생 행 RMSE          : {positive_rmse:,.4f}")
print(f"수요 발생 행 WAPE          : {positive_wape:,.2f}%")

print("-" * 65)

print(
    f"0수요 행 평균 P50 예측     : "
    f"{zero_row_prediction_mean:,.4f}"
)

print(
    f"0수요 행 양수 예측 비율    : "
    f"{zero_row_positive_prediction_rate:,.2f}%"
)

print("=" * 65)

display(diagnostic_result_df)

5-1-1. P50 점예측 추가 진단
전체 평가 행 수            : 62,092
실제 수요 0 비율           : 79.35%
실제 수요 발생 행 수       : 12,820
-----------------------------------------------------------------
실제 수요량 합계           : 20,982.0000
P50 예측량 합계            : 1,560.6053
평균 예측 편향             : -0.3128
-----------------------------------------------------------------
전체 0 예측 기준선 MAE     : 0.3379
전체 0 예측 기준선 RMSE    : 0.8492
전체 0 예측 기준선 WAPE    : 100.00%
-----------------------------------------------------------------
수요 발생 행 MAE           : 1.5687
수요 발생 행 RMSE          : 1.8080
수요 발생 행 WAPE          : 95.85%
-----------------------------------------------------------------
0수요 행 평균 P50 예측     : 0.0132
0수요 행 양수 예측 비율    : 6.89%


,항목,결과
0,전체 평가 행 수,"62,092.0000"
1,실제 수요 0인 행 수,"49,272.0000"
2,실제 수요 0 비율,79.3532
3,실제 수요 발생 행 수,"12,820.0000"
4,실제 수요량 합계,"20,982.0000"
5,P50 예측량 합계,"1,560.6053"
6,행당 평균 실제 수요,0.3379
7,행당 평균 P50 예측,0.0251
8,평균 예측 편향,-0.3128
9,0 예측 기준선 MAE,0.3379


In [56]:
# ============================================================
# 5-2. Pinball Loss 평가
# - P10: alpha = 0.1
# - P50: alpha = 0.5
# - P90: alpha = 0.9
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 평가 데이터 준비
# ------------------------------------------------------------

quantile_evaluation_df = (
    a1_base_demand_test_result_df[
        [
            "actual_demand",
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    ]
    .dropna()
    .copy()
)


# ------------------------------------------------------------
# 2. Pinball Loss 함수
# ------------------------------------------------------------

def calculate_pinball_loss(
    y_true,
    y_pred,
    quantile
):
    """
    분위수 예측용 Pinball Loss 계산

    quantile:
        P10 -> 0.1
        P50 -> 0.5
        P90 -> 0.9
    """

    error = y_true - y_pred

    loss = np.maximum(
        quantile * error,
        (quantile - 1) * error
    )

    return np.mean(loss)


# ------------------------------------------------------------
# 3. 분위수별 Pinball Loss 계산
# ------------------------------------------------------------

y_true = quantile_evaluation_df[
    "actual_demand"
].to_numpy()

p10_prediction = quantile_evaluation_df[
    "base_demand_p10"
].to_numpy()

p50_prediction = quantile_evaluation_df[
    "base_demand_p50"
].to_numpy()

p90_prediction = quantile_evaluation_df[
    "base_demand_p90"
].to_numpy()


p10_pinball_loss = calculate_pinball_loss(
    y_true=y_true,
    y_pred=p10_prediction,
    quantile=0.1
)

p50_pinball_loss = calculate_pinball_loss(
    y_true=y_true,
    y_pred=p50_prediction,
    quantile=0.5
)

p90_pinball_loss = calculate_pinball_loss(
    y_true=y_true,
    y_pred=p90_prediction,
    quantile=0.9
)


# ------------------------------------------------------------
# 4. 전체 평균 Pinball Loss
# ------------------------------------------------------------

mean_pinball_loss = np.mean(
    [
        p10_pinball_loss,
        p50_pinball_loss,
        p90_pinball_loss
    ]
)


# ------------------------------------------------------------
# 5. 결과 테이블 생성
# ------------------------------------------------------------

a1_pinball_metric_result_df = pd.DataFrame(
    {
        "model": [
            "LightGBM Quantile P10",
            "LightGBM Quantile P50",
            "LightGBM Quantile P90",
            "LightGBM Quantile Mean"
        ],
        "quantile": [
            0.1,
            0.5,
            0.9,
            np.nan
        ],
        "evaluation_data": [
            "Test",
            "Test",
            "Test",
            "Test"
        ],
        "evaluation_rows": [
            len(quantile_evaluation_df),
            len(quantile_evaluation_df),
            len(quantile_evaluation_df),
            len(quantile_evaluation_df)
        ],
        "pinball_loss": [
            p10_pinball_loss,
            p50_pinball_loss,
            p90_pinball_loss,
            mean_pinball_loss
        ]
    }
)


# ------------------------------------------------------------
# 6. 결과 출력
# ------------------------------------------------------------

print("=" * 65)
print("5-2. LightGBM 분위수별 Pinball Loss 평가")
print("=" * 65)

print(
    f"평가 행 수              : "
    f"{len(quantile_evaluation_df):,}"
)

print("-" * 65)

print(
    f"P10 Pinball Loss        : "
    f"{p10_pinball_loss:,.6f}"
)

print(
    f"P50 Pinball Loss        : "
    f"{p50_pinball_loss:,.6f}"
)

print(
    f"P90 Pinball Loss        : "
    f"{p90_pinball_loss:,.6f}"
)

print("-" * 65)

print(
    f"평균 Pinball Loss       : "
    f"{mean_pinball_loss:,.6f}"
)

print("=" * 65)

display(a1_pinball_metric_result_df)

5-2. LightGBM 분위수별 Pinball Loss 평가
평가 행 수              : 62,092
-----------------------------------------------------------------
P10 Pinball Loss        : 0.033638
P50 Pinball Loss        : 0.167166
P90 Pinball Loss        : 0.159446
-----------------------------------------------------------------
평균 Pinball Loss       : 0.120084


,model,quantile,evaluation_data,evaluation_rows,pinball_loss
0,LightGBM Quantile P10,0.1000,Test,62092,0.0336
1,LightGBM Quantile P50,0.5000,Test,62092,0.1672
2,LightGBM Quantile P90,0.9000,Test,62092,0.1594
3,LightGBM Quantile Mean,NaN,Test,62092,0.1201


In [57]:
# ============================================================
# 5-3. 예측구간 커버리지·폭 평가
# - 예측구간: P10 ~ P90
# - 명목 커버리지: 80%
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 평가 데이터 준비
# ------------------------------------------------------------

interval_evaluation_df = (
    a1_base_demand_test_result_df[
        [
            "actual_demand",
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    ]
    .dropna()
    .copy()
)


# ------------------------------------------------------------
# 2. 예측구간 포함 여부와 구간 폭 계산
# ------------------------------------------------------------

interval_evaluation_df["is_covered"] = (
    (
        interval_evaluation_df["actual_demand"]
        >= interval_evaluation_df["base_demand_p10"]
    )
    &
    (
        interval_evaluation_df["actual_demand"]
        <= interval_evaluation_df["base_demand_p90"]
    )
)

interval_evaluation_df["interval_width"] = (
    interval_evaluation_df["base_demand_p90"]
    - interval_evaluation_df["base_demand_p10"]
)


# ------------------------------------------------------------
# 3. 전체 데이터 커버리지·폭
# ------------------------------------------------------------

nominal_coverage = 80.0

overall_coverage = (
    interval_evaluation_df["is_covered"].mean()
    * 100
)

coverage_gap = (
    overall_coverage
    - nominal_coverage
)

average_interval_width = (
    interval_evaluation_df["interval_width"].mean()
)

median_interval_width = (
    interval_evaluation_df["interval_width"].median()
)

minimum_interval_width = (
    interval_evaluation_df["interval_width"].min()
)

maximum_interval_width = (
    interval_evaluation_df["interval_width"].max()
)


# ------------------------------------------------------------
# 4. 실제 수요가 발생한 행만 별도 평가
# ------------------------------------------------------------

positive_interval_df = interval_evaluation_df[
    interval_evaluation_df["actual_demand"] > 0
].copy()

if len(positive_interval_df) > 0:

    positive_coverage = (
        positive_interval_df["is_covered"].mean()
        * 100
    )

    positive_average_width = (
        positive_interval_df["interval_width"].mean()
    )

    positive_median_width = (
        positive_interval_df["interval_width"].median()
    )

else:

    positive_coverage = np.nan
    positive_average_width = np.nan
    positive_median_width = np.nan


# ------------------------------------------------------------
# 5. 실제 수요가 0인 행 별도 평가
# ------------------------------------------------------------

zero_interval_df = interval_evaluation_df[
    interval_evaluation_df["actual_demand"] == 0
].copy()

if len(zero_interval_df) > 0:

    zero_demand_coverage = (
        zero_interval_df["is_covered"].mean()
        * 100
    )

    zero_demand_average_width = (
        zero_interval_df["interval_width"].mean()
    )

else:

    zero_demand_coverage = np.nan
    zero_demand_average_width = np.nan


# ------------------------------------------------------------
# 6. 실제 수요 평균 대비 평균 구간 폭
# ------------------------------------------------------------

mean_actual_demand = (
    interval_evaluation_df["actual_demand"].mean()
)

if mean_actual_demand > 0:

    normalized_interval_width = (
        average_interval_width
        / mean_actual_demand
        * 100
    )

else:

    normalized_interval_width = np.nan


# ------------------------------------------------------------
# 7. 결과 테이블 생성
# ------------------------------------------------------------

a1_interval_metric_result_df = pd.DataFrame(
    {
        "evaluation_scope": [
            "전체 행",
            "수요 발생 행",
            "수요 0인 행"
        ],
        "evaluation_rows": [
            len(interval_evaluation_df),
            len(positive_interval_df),
            len(zero_interval_df)
        ],
        "nominal_coverage_percent": [
            nominal_coverage,
            nominal_coverage,
            nominal_coverage
        ],
        "actual_coverage_percent": [
            overall_coverage,
            positive_coverage,
            zero_demand_coverage
        ],
        "average_interval_width": [
            average_interval_width,
            positive_average_width,
            zero_demand_average_width
        ],
        "median_interval_width": [
            median_interval_width,
            positive_median_width,
            zero_interval_df["interval_width"].median()
            if len(zero_interval_df) > 0
            else np.nan
        ]
    }
)


# ------------------------------------------------------------
# 8. 결과 출력
# ------------------------------------------------------------

print("=" * 70)
print("5-3. LightGBM P10~P90 예측구간 평가")
print("=" * 70)

print(f"평가 행 수                  : {len(interval_evaluation_df):,}")
print(f"명목 커버리지               : {nominal_coverage:,.2f}%")
print(f"실제 전체 커버리지          : {overall_coverage:,.2f}%")
print(f"명목 대비 커버리지 차이     : {coverage_gap:+,.2f}%p")

print("-" * 70)

print(f"수요 발생 행 커버리지       : {positive_coverage:,.2f}%")
print(f"수요 0인 행 커버리지        : {zero_demand_coverage:,.2f}%")

print("-" * 70)

print(f"평균 예측구간 폭            : {average_interval_width:,.4f}")
print(f"중앙값 예측구간 폭          : {median_interval_width:,.4f}")
print(f"최소 예측구간 폭            : {minimum_interval_width:,.4f}")
print(f"최대 예측구간 폭            : {maximum_interval_width:,.4f}")
print(
    f"평균 실제 수요 대비 구간 폭 : "
    f"{normalized_interval_width:,.2f}%"
)

print("=" * 70)

display(a1_interval_metric_result_df)

5-3. LightGBM P10~P90 예측구간 평가
평가 행 수                  : 62,092
명목 커버리지               : 80.00%
실제 전체 커버리지          : 90.79%
명목 대비 커버리지 차이     : +10.79%p
----------------------------------------------------------------------
수요 발생 행 커버리지       : 55.41%
수요 0인 행 커버리지        : 100.00%
----------------------------------------------------------------------
평균 예측구간 폭            : 1.2234
중앙값 예측구간 폭          : 1.0654
최소 예측구간 폭            : 0.0000
최대 예측구간 폭            : 4.0899
평균 실제 수요 대비 구간 폭 : 362.04%


,evaluation_scope,evaluation_rows,nominal_coverage_percent,actual_coverage_percent,average_interval_width,median_interval_width
0,전체 행,62092,80.0000,90.7927,1.2234,1.0654
1,수요 발생 행,12820,80.0000,55.4134,1.6949,1.6006
2,수요 0인 행,49272,80.0000,99.9980,1.1007,1.0120


In [58]:
# ============================================================
# 5-4. 분위수 역전 확인
# - 정상 순서: P10 <= P50 <= P90
# - 추가 확인: 동일 분위수 및 예측구간 붕괴
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 평가 데이터 준비
# ------------------------------------------------------------

quantile_order_df = (
    a1_base_demand_test_result_df[
        [
            "actual_demand",
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    ]
    .dropna()
    .copy()
)

total_rows = len(quantile_order_df)

# 부동소수점 계산 오차 허용값
tolerance = 1e-10


# ------------------------------------------------------------
# 2. 분위수 역전 여부 확인
# ------------------------------------------------------------

quantile_order_df["p10_p50_crossing"] = (
    quantile_order_df["base_demand_p10"]
    >
    quantile_order_df["base_demand_p50"] + tolerance
)

quantile_order_df["p50_p90_crossing"] = (
    quantile_order_df["base_demand_p50"]
    >
    quantile_order_df["base_demand_p90"] + tolerance
)

quantile_order_df["p10_p90_crossing"] = (
    quantile_order_df["base_demand_p10"]
    >
    quantile_order_df["base_demand_p90"] + tolerance
)

quantile_order_df["any_quantile_crossing"] = (
    quantile_order_df[
        [
            "p10_p50_crossing",
            "p50_p90_crossing",
            "p10_p90_crossing"
        ]
    ]
    .any(axis=1)
)


# ------------------------------------------------------------
# 3. 분위수가 동일한 행 확인
# ------------------------------------------------------------

quantile_order_df["p10_p50_equal"] = np.isclose(
    quantile_order_df["base_demand_p10"],
    quantile_order_df["base_demand_p50"],
    atol=tolerance,
    rtol=0
)

quantile_order_df["p50_p90_equal"] = np.isclose(
    quantile_order_df["base_demand_p50"],
    quantile_order_df["base_demand_p90"],
    atol=tolerance,
    rtol=0
)

quantile_order_df["all_quantiles_equal"] = (
    quantile_order_df["p10_p50_equal"]
    &
    quantile_order_df["p50_p90_equal"]
)

quantile_order_df["interval_width"] = (
    quantile_order_df["base_demand_p90"]
    -
    quantile_order_df["base_demand_p10"]
)

quantile_order_df["collapsed_interval"] = np.isclose(
    quantile_order_df["interval_width"],
    0,
    atol=tolerance,
    rtol=0
)


# ------------------------------------------------------------
# 4. 횟수 및 비율 계산
# ------------------------------------------------------------

p10_p50_crossing_count = (
    quantile_order_df["p10_p50_crossing"].sum()
)

p50_p90_crossing_count = (
    quantile_order_df["p50_p90_crossing"].sum()
)

p10_p90_crossing_count = (
    quantile_order_df["p10_p90_crossing"].sum()
)

any_crossing_count = (
    quantile_order_df["any_quantile_crossing"].sum()
)

valid_order_count = (
    total_rows
    - any_crossing_count
)

all_equal_count = (
    quantile_order_df["all_quantiles_equal"].sum()
)

collapsed_interval_count = (
    quantile_order_df["collapsed_interval"].sum()
)


def calculate_rate(count, denominator):
    if denominator == 0:
        return np.nan

    return count / denominator * 100


p10_p50_crossing_rate = calculate_rate(
    p10_p50_crossing_count,
    total_rows
)

p50_p90_crossing_rate = calculate_rate(
    p50_p90_crossing_count,
    total_rows
)

p10_p90_crossing_rate = calculate_rate(
    p10_p90_crossing_count,
    total_rows
)

any_crossing_rate = calculate_rate(
    any_crossing_count,
    total_rows
)

valid_order_rate = calculate_rate(
    valid_order_count,
    total_rows
)

all_equal_rate = calculate_rate(
    all_equal_count,
    total_rows
)

collapsed_interval_rate = calculate_rate(
    collapsed_interval_count,
    total_rows
)


# ------------------------------------------------------------
# 5. 결과 테이블 생성
# ------------------------------------------------------------

a1_quantile_order_result_df = pd.DataFrame(
    {
        "check_item": [
            "P10 > P50",
            "P50 > P90",
            "P10 > P90",
            "하나 이상의 분위수 역전",
            "정상 분위수 순서",
            "P10·P50·P90 모두 동일",
            "P10~P90 구간 폭 0"
        ],
        "row_count": [
            p10_p50_crossing_count,
            p50_p90_crossing_count,
            p10_p90_crossing_count,
            any_crossing_count,
            valid_order_count,
            all_equal_count,
            collapsed_interval_count
        ],
        "rate_percent": [
            p10_p50_crossing_rate,
            p50_p90_crossing_rate,
            p10_p90_crossing_rate,
            any_crossing_rate,
            valid_order_rate,
            all_equal_rate,
            collapsed_interval_rate
        ]
    }
)


# ------------------------------------------------------------
# 6. 결과 출력
# ------------------------------------------------------------

print("=" * 70)
print("5-4. LightGBM 분위수 역전 확인")
print("=" * 70)

print(f"전체 평가 행 수             : {total_rows:,}")

print("-" * 70)

print(
    f"P10 > P50 역전              : "
    f"{p10_p50_crossing_count:,}건 "
    f"({p10_p50_crossing_rate:,.4f}%)"
)

print(
    f"P50 > P90 역전              : "
    f"{p50_p90_crossing_count:,}건 "
    f"({p50_p90_crossing_rate:,.4f}%)"
)

print(
    f"P10 > P90 역전              : "
    f"{p10_p90_crossing_count:,}건 "
    f"({p10_p90_crossing_rate:,.4f}%)"
)

print("-" * 70)

print(
    f"하나 이상의 분위수 역전     : "
    f"{any_crossing_count:,}건 "
    f"({any_crossing_rate:,.4f}%)"
)

print(
    f"정상 분위수 순서            : "
    f"{valid_order_count:,}건 "
    f"({valid_order_rate:,.4f}%)"
)

print("-" * 70)

print(
    f"P10·P50·P90 모두 동일       : "
    f"{all_equal_count:,}건 "
    f"({all_equal_rate:,.4f}%)"
)

print(
    f"P10~P90 구간 폭 0           : "
    f"{collapsed_interval_count:,}건 "
    f"({collapsed_interval_rate:,.4f}%)"
)

print("=" * 70)

display(a1_quantile_order_result_df)


# ------------------------------------------------------------
# 7. 역전 행 예시 출력
# ------------------------------------------------------------

crossing_example_df = (
    quantile_order_df[
        quantile_order_df["any_quantile_crossing"]
    ][
        [
            "actual_demand",
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    ]
    .head(20)
)

if any_crossing_count > 0:

    print("\n분위수 역전 행 예시:")
    display(crossing_example_df)

else:

    print("\n분위수 역전 행이 없습니다.")

5-4. LightGBM 분위수 역전 확인
전체 평가 행 수             : 62,092
----------------------------------------------------------------------
P10 > P50 역전              : 0건 (0.0000%)
P50 > P90 역전              : 0건 (0.0000%)
P10 > P90 역전              : 0건 (0.0000%)
----------------------------------------------------------------------
하나 이상의 분위수 역전     : 0건 (0.0000%)
정상 분위수 순서            : 62,092건 (100.0000%)
----------------------------------------------------------------------
P10·P50·P90 모두 동일       : 2,196건 (3.5367%)
P10~P90 구간 폭 0           : 2,196건 (3.5367%)


,check_item,row_count,rate_percent
0,P10 > P50,0,0.0000
1,P50 > P90,0,0.0000
2,P10 > P90,0,0.0000
3,하나 이상의 분위수 역전,0,0.0000
4,정상 분위수 순서,62092,100.0000
5,P10·P50·P90 모두 동일,2196,3.5367
6,P10~P90 구간 폭 0,2196,3.5367



분위수 역전 행이 없습니다.


In [59]:
# ============================================================
# 5-5. 점포·상품군·시간대별 평가
# - 점포: store_id
# - 상품군: category
# - 시간대: hour
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 평가 데이터 준비
# ------------------------------------------------------------

segment_evaluation_df = (
    a1_base_demand_test_result_df[
        [
            "store_id",
            "category",
            "hour",
            "actual_demand",
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    ]
    .dropna(
        subset=[
            "actual_demand",
            "base_demand_p10",
            "base_demand_p50",
            "base_demand_p90"
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# 2. 구간별 평가 함수
# ------------------------------------------------------------

def calculate_segment_metrics(
    evaluation_df,
    group_column
):

    metric_rows = []

    for group_value, group_df in evaluation_df.groupby(
        group_column,
        observed=True,
        dropna=False
    ):

        y_true = group_df[
            "actual_demand"
        ].to_numpy()

        p10_pred = group_df[
            "base_demand_p10"
        ].to_numpy()

        p50_pred = group_df[
            "base_demand_p50"
        ].to_numpy()

        p90_pred = group_df[
            "base_demand_p90"
        ].to_numpy()

        absolute_error = np.abs(
            y_true - p50_pred
        )

        squared_error = (
            y_true - p50_pred
        ) ** 2


        # ----------------------------------------------------
        # 기본 규모
        # ----------------------------------------------------

        evaluation_rows = len(group_df)

        actual_demand_sum = y_true.sum()
        prediction_sum = p50_pred.sum()

        zero_demand_rate = (
            (y_true == 0).mean()
            * 100
        )

        positive_mask = (
            y_true > 0
        )

        positive_demand_rows = (
            positive_mask.sum()
        )


        # ----------------------------------------------------
        # MAE·RMSE·WAPE
        # ----------------------------------------------------

        mae = absolute_error.mean()

        rmse = np.sqrt(
            squared_error.mean()
        )

        if np.abs(y_true).sum() > 0:

            wape = (
                absolute_error.sum()
                / np.abs(y_true).sum()
                * 100
            )

            prediction_to_actual_rate = (
                prediction_sum
                / actual_demand_sum
                * 100
            )

        else:

            wape = np.nan
            prediction_to_actual_rate = np.nan


        # ----------------------------------------------------
        # 전체 P10~P90 커버리지
        # ----------------------------------------------------

        covered_mask = (
            (y_true >= p10_pred)
            &
            (y_true <= p90_pred)
        )

        interval_coverage = (
            covered_mask.mean()
            * 100
        )

        interval_width = (
            p90_pred - p10_pred
        )

        average_interval_width = (
            interval_width.mean()
        )


        # ----------------------------------------------------
        # 실제 수요 발생 행 평가
        # ----------------------------------------------------

        if positive_demand_rows > 0:

            positive_mae = (
                absolute_error[
                    positive_mask
                ].mean()
            )

            positive_wape = (
                absolute_error[
                    positive_mask
                ].sum()
                / y_true[
                    positive_mask
                ].sum()
                * 100
            )

            positive_coverage = (
                covered_mask[
                    positive_mask
                ].mean()
                * 100
            )

        else:

            positive_mae = np.nan
            positive_wape = np.nan
            positive_coverage = np.nan


        # ----------------------------------------------------
        # 결과 한 행 생성
        # ----------------------------------------------------

        metric_rows.append(
            {
                group_column: group_value,
                "evaluation_rows": evaluation_rows,
                "positive_demand_rows": positive_demand_rows,
                "zero_demand_rate_percent": zero_demand_rate,
                "actual_demand_sum": actual_demand_sum,
                "p50_prediction_sum": prediction_sum,
                "prediction_to_actual_percent":
                    prediction_to_actual_rate,
                "MAE": mae,
                "RMSE": rmse,
                "WAPE_percent": wape,
                "positive_demand_MAE": positive_mae,
                "positive_demand_WAPE_percent":
                    positive_wape,
                "interval_coverage_percent":
                    interval_coverage,
                "positive_demand_coverage_percent":
                    positive_coverage,
                "average_interval_width":
                    average_interval_width
            }
        )

    result_df = pd.DataFrame(
        metric_rows
    )

    return result_df


# ------------------------------------------------------------
# 3. 점포별 평가
# ------------------------------------------------------------

a1_store_metric_result_df = (
    calculate_segment_metrics(
        evaluation_df=segment_evaluation_df,
        group_column="store_id"
    )
    .sort_values(
        "actual_demand_sum",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. 상품군별 평가
# ------------------------------------------------------------

a1_category_metric_result_df = (
    calculate_segment_metrics(
        evaluation_df=segment_evaluation_df,
        group_column="category"
    )
    .sort_values(
        "actual_demand_sum",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. 시간대별 평가
# ------------------------------------------------------------

a1_hour_metric_result_df = (
    calculate_segment_metrics(
        evaluation_df=segment_evaluation_df,
        group_column="hour"
    )
    .sort_values(
        "hour"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. 결과 출력
# ------------------------------------------------------------

print("=" * 75)
print("5-5-1. 점포별 평가")
print("=" * 75)

display(
    a1_store_metric_result_df.round(4)
)


print("\n")
print("=" * 75)
print("5-5-2. 상품군별 평가")
print("=" * 75)

display(
    a1_category_metric_result_df.round(4)
)


print("\n")
print("=" * 75)
print("5-5-3. 시간대별 평가")
print("=" * 75)

display(
    a1_hour_metric_result_df.round(4)
)

5-5-1. 점포별 평가


,store_id,evaluation_rows,positive_demand_rows,zero_demand_rate_percent,actual_demand_sum,p50_prediction_sum,prediction_to_actual_percent,MAE,RMSE,WAPE_percent,positive_demand_MAE,positive_demand_WAPE_percent,interval_coverage_percent,positive_demand_coverage_percent,average_interval_width
0,S01,21242,4400,79.2863,"7,189.0000",470.7824,6.5486,0.3344,0.8258,98.8129,1.5721,96.2206,90.7165,55.2045,1.2103
1,S02,19608,4207,78.5445,"6,965.0000",606.0343,8.7011,0.3514,0.8491,98.9157,1.5767,95.2358,90.5957,56.1683,1.2681
2,S03,21242,4213,80.1667,"6,828.0000",483.7886,7.0854,0.3185,0.8051,99.0954,1.5572,96.0822,91.0507,54.8778,1.1951




5-5-2. 상품군별 평가


,category,evaluation_rows,positive_demand_rows,zero_demand_rate_percent,actual_demand_sum,p50_prediction_sum,prediction_to_actual_percent,MAE,RMSE,WAPE_percent,positive_demand_MAE,positive_demand_WAPE_percent,interval_coverage_percent,positive_demand_coverage_percent,average_interval_width
0,produce,19608,5763,70.6089,"10,303.0000","1,103.5300",10.7108,0.5186,1.0749,98.7019,1.6809,94.0190,89.8052,65.3132,1.7360
1,meat,13072,2617,79.9801,"4,043.0000",106.4308,2.6325,0.3072,0.7679,99.3261,1.5207,98.4326,90.3152,51.6240,1.1453
2,dairy,9804,1830,81.3341,"2,879.0000",115.0840,3.9974,0.2912,0.7570,99.1532,1.5367,97.6784,91.0445,52.0765,1.1484
3,deli,11438,1799,84.2717,"2,674.0000",168.8534,6.3146,0.2314,0.6417,98.9971,1.4366,96.6478,90.6714,40.6893,0.9494
4,cheese,8170,811,90.0734,"1,083.0000",66.7070,6.1595,0.1313,0.4563,99.0363,1.2923,96.7719,93.7944,37.4846,0.5915




5-5-3. 시간대별 평가


,hour,evaluation_rows,positive_demand_rows,zero_demand_rate_percent,actual_demand_sum,p50_prediction_sum,prediction_to_actual_percent,MAE,RMSE,WAPE_percent,positive_demand_MAE,positive_demand_WAPE_percent,interval_coverage_percent,positive_demand_coverage_percent,average_interval_width
0,10,4902,958,80.4570,"1,572.0000",0.0000,0.0000,0.3207,0.8316,100.0000,1.6409,100.0000,89.7389,47.4948,1.1460
1,11,4902,893,81.7829,"1,424.0000",0.1268,0.0089,0.2905,0.7694,100.0050,1.5946,99.9981,90.5957,48.3763,1.1287
2,12,4902,891,81.8237,"1,399.0000",70.5560,5.0433,0.2845,0.7464,99.6920,1.5281,97.3244,92.5949,59.2593,1.2624
3,13,4902,981,79.9878,"1,639.0000",70.3500,4.2923,0.3334,0.8409,99.7205,1.6326,97.7141,90.2081,51.0703,1.2151
4,14,4902,916,81.3137,"1,587.0000",71.0120,4.4746,0.3210,0.8392,99.1617,1.6865,97.3435,90.9017,51.3100,1.1914
5,15,4902,947,80.6814,"1,510.0000",1.8730,0.1240,0.3081,0.7915,100.0290,1.5938,99.9525,90.4733,50.6864,1.1577
6,16,4902,916,81.3137,"1,476.0000",1.8722,0.1268,0.3012,0.7936,100.0328,1.6106,99.9530,91.1057,52.4017,1.1489
7,17,4902,927,81.0894,"1,517.0000",1.8722,0.1234,0.3095,0.8089,100.0031,1.6355,99.9399,90.5957,50.2697,1.1257
8,18,4902,1208,75.3570,"2,013.0000",275.4310,13.6826,0.4047,0.8932,98.5437,1.5403,92.4314,91.1873,64.2384,1.4542
9,19,4902,1195,75.6222,"2,030.0000",296.5553,14.6086,0.4104,0.9156,99.1072,1.5671,92.2502,90.7385,62.0084,1.4282


In [59]:
## 희소수요에 적합한 모델 구조로 개선
#데이터팀답변받음-> 2단계LightGBM진행